# Affordable Housing Development and Road Safety in New York City
### Data Engineering and Visualization — Winter Semester 2026 · German International University · Project Milestone 1

| | |
|---|---|
| **Student responsible** | Abdelrahman El-sayed (13004550) |
| **Other team members** | Not yet assigned (see Section 14) |
| **AI assistance** | ChatGPT (OpenAI) helped with planning, explanations and reviews; Claude (Anthropic) revised the code. Section 14 explains who did what. |
| **Deadline** | 9 November 2026 |
| **Study period** | 2022–2025 (2026 is not complete yet, so it is left out of yearly comparisons) |

---
## 1. Introduction and objectives

New York City builds and preserves thousands of affordable homes every year. At the same time, thousands of people are killed or seriously injured in traffic crashes. Housing and transportation are usually studied separately, so this project asks whether the two meet in the same places:

> **Do areas with more affordable housing production also have higher serious traffic-crash rates, and how does this change across locations, years, construction types and income groups?**

**Two terms we use a lot.** *Affordable housing production* means affordable units in buildings that HPD recorded as completed. There are two construction types: **new construction** (new homes) and **preservation** (repairing or refinancing existing affordable homes, which does not necessarily add homes or residents). When a number includes both types, we say so.

**Objectives**
1. Load, explore and clean one housing dataset and one transportation dataset from NYC Open Data.
2. Combine them by **ZIP-code area and year**, instead of matching single records.
3. Answer eight research questions with one chart each. The results describe areas, not individual people, and they show patterns, not causes.
4. Save the cleaned data for the Dash website, which uses the same calculation code as this notebook.

**Eight research questions**

| Section | Research question | Chart |
|---|---|---|
| **A — Spatial relationship** | **RQ1** Is affordable housing production associated with serious traffic-crash rates across NYC areas? | Scatter plot |
| | **RQ2** Which areas have both relatively high affordable housing production and relatively high serious-crash rates? | Map |
| **B — Time patterns** | **RQ3** How did serious crashes change between 2022 and 2025 across areas with different levels of affordable housing production? | Line chart |
| | **RQ4** How do crash times and weekdays differ between areas with different housing-production levels? | Heatmap |
| **C — Housing characteristics** | **RQ5** Do areas dominated by new construction and preservation show different serious-crash patterns? | Box plot |
| | **RQ6** Are affordable housing units in different income categories located in areas with different serious-crash rates? | Comparative bar chart |
| **D — Road-safety priorities** | **RQ7** Do areas with greater affordable housing production also have higher shares of crashes with reported historical similar serious-injury or fatality events? | Stacked bar chart |
| | **RQ8** Which areas combine unfinished affordable housing projects with relatively high serious-crash rates? | Ranked bar chart |

**How to read the notebook.** Each section says what we do, why we do it and what the result means. Validation checks print `[PASS]`, `[WARN]` or `[FAIL]`. If an important check fails, the notebook stops with a message instead of hiding the problem.

## 2. Datasets and selection

| Role | Dataset (catalog code, ID) | Publisher | One row is | Link |
|---|---|---|---|---|
| Housing & Development | Affordable Housing Production by Building (**H2**, `hg8x-zxpr`) | Dept. of Housing Preservation and Development (HPD) | One building in one HPD-financed affordable housing project started since 1 January 2014 | https://data.cityofnewyork.us/d/hg8x-zxpr |
| Transportation | Serious Injury Response, Tracking & Analysis Program — SIRTA (**T4**, `xeqp-qz8h`) | Dept. of Transportation (DOT) | One serious crash (someone was killed or seriously injured) reviewed by DOT | https://data.cityofnewyork.us/d/xeqp-qz8h |
| Geographic reference only | Modified ZIP Code Tabulation Areas — MODZCTA (`pri4-ifjk`) | Dept. of Health and Mental Hygiene (DOHMH) | One ZIP-code area: its boundary, its ZIP codes and a 2018 population estimate | https://data.cityofnewyork.us/d/pri4-ifjk |

**Dataset overview** (sizes from the October 2026 downloads; Section 3 prints the sizes of the current download)

| | Housing (H2) | SIRTA (T4) |
|---|---|---|
| Rows × columns | 9,551 × 41 | 13,754 × 23 |
| Location columns | borough, ZIP code (`postcode`), latitude/longitude, community board, council district, census tract, NTA, BBL/BIN | borough, ZIP code, latitude/longitude, police precinct, community board, council district, census tract, NTA |
| Date and time columns | project start, project completion and building completion dates | crash date and crash time |
| Main problems found (Sections 4–7) | confidential buildings without a location; blank income-band cells; buildings listed in more than one project; some completion dates before the project start | rows with no id or date; exact duplicates; one repeated collision id; ZIP mostly blank in 2022; placeholder ZIP 10000; many crash times of exactly 00:00 |
| Combined as | units completed per ZIP area and year | serious crashes per ZIP area and year |

**Why we chose these two datasets.** Both are in the course Dataset Catalog, both have more than nine columns, and both have ZIP codes and coordinates, so we can place their records in the same ZIP-code areas. The housing data shows where affordable housing is built or preserved; SIRTA shows where the most serious crashes happen. Together they fit one of the example questions in the brief: *do street-safety problems cluster around new developments?* The two datasets describe different things (buildings and crashes), so we do not match single records. Instead we count both per ZIP area and year and then join the counts (Section 8).

**What SIRTA covers.** SIRTA only contains serious crashes that DOT reviewed, not every collision. So every result here is about *serious* crashes.

**What H2 covers.** H2 only covers affordable housing financed by HPD, not all construction. Buildings published as `CONFIDENTIAL` (mostly small homeownership units) have no location.

**Why we use MODZCTA.** A ZIP code is a postal route, not an area. MODZCTA gives each ZIP-code area a boundary (so we can place a crash by its coordinates) and a population estimate (so we can compute rates per resident). We use it **only for geographic matching, maps and population**, not as a third dataset to analyse. **The TA has not yet confirmed whether this is allowed under the "exactly two datasets" rule.** We do not measure area sizes anywhere in the notebook.

## 3. Python environment and data loading
### 3.1 Packages
The notebook uses six libraries: `pandas`, `numpy`, `requests`, `plotly`, `geopandas` and `shapely`. Google Colab already has them, so the next cell only installs a library if it is missing.

In [1]:
# Install any library that is not already available (Colab usually has all of them).
import importlib.util, subprocess, sys

REQUIRED_PACKAGES = {"pandas": "pandas", "numpy": "numpy", "requests": "requests",
                     "plotly": "plotly", "geopandas": "geopandas", "shapely": "shapely"}
missing_packages = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_packages])
else:
    print("All required packages are already installed.")

All required packages are already installed.


### 3.2 Settings, chart style, logs and helper functions
These cells set the folders and study years, the chart style, the lists that record every check and cleaning step, and a few small helper functions used during cleaning.

In [2]:
# Libraries used in the notebook.
import hashlib, importlib, io, json, os, platform, re, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

# Folders for the raw downloads, the cleaned data and the reports.
PROJECT_ROOT = Path(os.environ.get("DEV_PROJECT_ROOT", "dev_project")).resolve()
DIRS = {name: PROJECT_ROOT / sub for name, sub in {
    "raw": "data/raw", "processed": "data/processed", "reports": "reports"}.items()}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

# To rerun on raw files saved earlier, set this to their date (for example "2026-10-15").
# The files must be in data/raw. Leave it as None to download fresh data.
RAW_SNAPSHOT_DATE = os.environ.get("DEV_RAW_SNAPSHOT_DATE") or None
# Only for testing the code without internet: a folder with housing.csv, sirta.csv and modzcta.geojson.
# It is empty by default, so the real data is never replaced by test files.
OFFLINE_SOURCE_DIR = os.environ.get("DEV_OFFLINE_SOURCE_DIR")

ANALYSIS_YEARS = [2022, 2023, 2024, 2025]
NYC_LAT_RANGE, NYC_LON_RANGE = (40.49, 40.92), (-74.27, -73.68)
BOROUGHS = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]

# Row counts from the October 2026 downloads. A newer download can be different; Section 3.4 reports it.
EXPECTED_SNAPSHOT = {"housing": {"rows": 9551, "cols": 41}, "sirta": {"rows": 13754, "cols": 23},
                     "modzcta": {"rows": 178}}

ENVIRONMENT = {"python": platform.python_version(), "platform": platform.platform(),
               **{pkg: importlib.import_module(pkg).__version__ for pkg in
                  ["pandas", "numpy", "requests", "plotly", "geopandas", "shapely"]}}
print("Project folder:", PROJECT_ROOT)
pd.Series(ENVIRONMENT, name="version").to_frame()

Project folder: /content/dev_project


,version
python,3.13.16
platform,Linux-6.6.122+-x86_64-with-glibc2.39
pandas,2.2.3
numpy,2.1.3
requests,2.32.4
plotly,5.24.1
geopandas,1.2.0
shapely,2.1.2


In [3]:
# Colours and style for the EDA and cleaning charts.
# The colours are easy to tell apart, also for colour-blind readers.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
NEUTRAL = "#b8b6ae"
INK, INK_SOFT, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e7e6e2"

pio.templates["dev"] = go.layout.Template(layout=dict(
    font=dict(family="Arial, Helvetica, sans-serif", size=13, color=INK),
    title=dict(x=0.0, xanchor="left", font=dict(size=16, color=INK)),
    colorway=SERIES, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=INK_SOFT, ticks="outside", title_font=dict(color=INK_SOFT)),
    yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=INK_SOFT, ticks="outside", title_font=dict(color=INK_SOFT)),
    legend=dict(bgcolor="rgba(0,0,0,0)", title_font=dict(color=INK_SOFT)),
    hoverlabel=dict(font_size=12), margin=dict(l=70, r=30, t=80, b=60)))
pio.templates.default = "plotly_white+dev"

def style(fig, title, subtitle=None, height=460):
    # Add a bold title, a smaller grey subtitle and the chart height.
    text = f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:{INK_SOFT}'>{subtitle}</span>" if subtitle else "")
    fig.update_layout(title=dict(text=text), height=height)
    return fig

In [4]:
# Every check, known data problem and cleaning step is saved in these lists,
# so we can print a summary at the end (Section 13) and save it as a report.
class ValidationError(AssertionError):
    # Used to stop the notebook when an important check fails.
    pass

VALIDATIONS, KNOWN_ANOMALIES, CLEANING_LOG = [], [], []

def check(check_id, condition, passed, failed, *, stage, severity="error"):
    # Save and print the result of a check. A failed "error" check stops the notebook;
    # a failed "warn" check only prints a warning.
    ok = bool(condition)
    status = "PASS" if ok else ("FAIL" if severity == "error" else "WARN")
    detail = passed if ok else failed
    VALIDATIONS.append({"check_id": check_id, "stage": stage, "status": status, "detail": detail})
    print(f"[{status}] {check_id}: {detail}")
    if status == "FAIL":
        raise ValidationError(f"Check '{check_id}' failed during '{stage}': {failed}")
    return ok

def record_anomaly(anomaly_id, dataset, n_records, description, handling):
    # Note a problem in the source data that we keep on purpose or resolve (it is not a bug in our code).
    KNOWN_ANOMALIES.append({"anomaly_id": anomaly_id, "dataset": dataset, "records": int(n_records),
                            "description": description, "handling": handling})

def log_step(dataset, step, action, affected, rows_before, rows_after, reason):
    # Note what a cleaning step changed and how many records it affected.
    CLEANING_LOG.append({"dataset": dataset, "step": step, "action": action, "records_affected": int(affected),
                         "rows_before": int(rows_before), "rows_after": int(rows_after), "reason": reason})

In [5]:
# Small helper functions used in the cleaning sections.
ZIP_PLACEHOLDERS = {"10000", "99999", "00000"}
NYC_ZIP_RANGE = (10001, 11697)
MONTH_ABBR = "Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec"

def as_bool(x):
    # Turn a True/False column that may contain missing values into plain True/False (missing counts as False).
    return pd.Series(x).fillna(False).to_numpy(dtype=bool)

def clean_text(s):
    # Remove extra spaces. An empty text becomes a missing value.
    out = s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    return out.mask(out.eq(""))

def normalize_id(s):
    # IDs are kept as text. Remove a trailing ".0" that appears when an ID was saved as a decimal number.
    return clean_text(s).str.replace(r"\.0+$", "", regex=True)

def parse_zip_strict(raw):
    # Return the 5-digit ZIP code and a status for each value.
    # Only a complete 5-digit NYC ZIP is valid, so values like "112210" or "ZIP 11221" are never cut down to a wrong ZIP.
    s = clean_text(raw)
    five = s.str.extract(r"^(\d{5})(?:\.0+)?$", expand=False)
    numeric = pd.to_numeric(five, errors="coerce")
    in_range = numeric.between(*NYC_ZIP_RANGE)
    status = np.select(
        [as_bool(s.isna()), as_bool(five.isna()), as_bool(five.isin(ZIP_PLACEHOLDERS)), ~as_bool(in_range)],
        ["missing", "malformed", "placeholder", "outside_nyc_range"], default="valid")
    zip5 = five.where(pd.Series(status, index=s.index).eq("valid")).astype("string")
    return zip5, pd.Series(status, index=s.index, dtype="string")

def to_number(s):
    # Convert text to numbers and count the values that could not be converted.
    text = clean_text(s)
    num = pd.to_numeric(text, errors="coerce")
    return num, int((text.notna() & num.isna()).sum())

def normalize_time(raw):
    # Return the time as HH:MM, the hour, and a status: valid, exact_midnight, invalid or missing.
    s = clean_text(raw)
    parts = s.str.extract(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?$")
    hh, mm = pd.to_numeric(parts[0], errors="coerce"), pd.to_numeric(parts[1], errors="coerce")
    valid = hh.between(0, 23) & mm.between(0, 59)
    hhmm = (hh.astype("Int64").astype("string").str.zfill(2) + ":" + mm.astype("Int64").astype("string").str.zfill(2)).where(valid)
    status = np.select([as_bool(s.isna()), ~as_bool(valid), as_bool(hhmm.eq("00:00"))],
                       ["missing", "invalid", "exact_midnight"], default="valid")
    return hhmm.astype("string"), hh.where(valid).astype("Int64"), pd.Series(status, index=s.index, dtype="string")

Two quick tests check the ZIP and time helpers on values where we know the right answer. If one of these helpers is changed by mistake, the notebook stops here.

In [6]:
# Test the ZIP parser and the time parser on values where we know the correct answer.
zip_cases = pd.Series(["11221", "11221.0", " 11211 ", "112210", "1122", "ZIP 11221", "10000", "07030", None])
zip_out, zip_status = parse_zip_strict(zip_cases)
expected_status = ["valid", "valid", "valid", "malformed", "malformed", "malformed", "placeholder", "outside_nyc_range", "missing"]
check("helper_zip_parser", list(zip_status) == expected_status and zip_out.iloc[0] == "11221" and zip_out.iloc[3] is pd.NA,
      "strict ZIP parser accepts only complete 5-digit NYC ZIPs", f"unexpected ZIP statuses: {list(zip_status)}", stage="setup")

time_cases = pd.Series(["0:00", "9:57", "17:30", "24:00", "12:60", "ab", None])
hhmm, hour, time_status = normalize_time(time_cases)
check("helper_time_parser", list(hhmm.fillna("NA")) == ["00:00", "09:57", "17:30", "NA", "NA", "NA", "NA"]
      and list(time_status) == ["exact_midnight", "valid", "valid", "invalid", "invalid", "invalid", "missing"],
      "time parser normalises H:MM and rejects impossible times", f"unexpected time parsing: {list(hhmm)} {list(time_status)}", stage="setup")

[PASS] helper_zip_parser: strict ZIP parser accepts only complete 5-digit NYC ZIPs
[PASS] helper_time_parser: time parser normalises H:MM and rejects impossible times


True

### 3.3 Loading the data from NYC Open Data
**What we do.** Download the two datasets and the MODZCTA reference from the NYC Open Data API, and save the downloaded files with a small manifest (URL, time, number of rows and a checksum).

**Why.** The datasets are updated over time (SIRTA every quarter). Saving the exact files means we can rerun the notebook later and get the same numbers.

**Getting every row.** The API only returns 1,000 rows if we do not set `$limit`. So the notebook first asks the API how many rows there are, requests all of them, and checks that the download has the same number. If the download fails, it is retried and then stopped with a clear message. Missing data is never replaced with made-up data.

In [7]:
# The three datasets, their NYC Open Data ids and the columns we expect in each one.
SOURCES = {
    "housing": {"id": "hg8x-zxpr", "title": "Affordable Housing Production by Building (H2)", "format": "csv",
                "page": "https://data.cityofnewyork.us/d/hg8x-zxpr"},
    "sirta":   {"id": "xeqp-qz8h", "title": "Serious Injury Response, Tracking & Analysis Program - SIRTA (T4)", "format": "csv",
                "page": "https://data.cityofnewyork.us/d/xeqp-qz8h"},
    "modzcta": {"id": "pri4-ifjk", "title": "Modified ZIP Code Tabulation Areas - MODZCTA (geographic reference)", "format": "geojson",
                "page": "https://data.cityofnewyork.us/d/pri4-ifjk"},
}
SODA_RESOURCE = "https://data.cityofnewyork.us/resource/{id}.{fmt}"
MODZCTA_EXPORT = "https://data.cityofnewyork.us/api/geospatial/pri4-ifjk?method=export&format=GeoJSON"

EXPECTED_COLUMNS = {
    "housing": ["project_id", "project_name", "project_start_date", "project_completion_date", "building_id",
                "house_number", "street_name", "borough", "postcode", "bbl", "bin", "community_board",
                "council_district", "census_tract", "neighborhood_tabulation_area", "latitude", "longitude",
                "latitude_internal", "longitude_internal", "building_completion_date", "reporting_construction_type",
                "extended_affordability_status", "prevailing_wage_status", "extremely_low_income_units",
                "very_low_income_units", "low_income_units", "moderate_income_units", "middle_income_units",
                "other_income_units", "studio_units", "_1_br_units", "_2_br_units", "_3_br_units", "_4_br_units",
                "_5_br_units", "_6_br_units", "unknown_br_units", "counted_rental_units",
                "counted_homeownership_units", "all_counted_units", "total_units"],
    "sirta": ["open_data_crash_id", "crash_date", "crash_time", "crash_police_precinct", "crash_borough",
              "total_non_severe_injuries", "total_severe_injuries", "total_fatalities", "latitude", "longitude",
              "on_street", "cross_street", "address", "highway_reference_marker", "zipcode",
              "similar_fatalities_severe", "similar_fatalities_all", "community_board", "council_district", "bin",
              "bbl", "census_tract_2020_", "neighborhood_tabulation_area_nta_2020_"],
    "modzcta": ["modzcta", "label", "zcta", "pop_est"],
}

def http_get(url, params=None, timeout=180, retries=3):
    # Download a URL and try up to three times. If it still fails, stop with a clear message.
    # It never returns partial or made-up data.
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.RequestException as err:
            last_error = err
            print(f"  attempt {attempt}/{retries} failed: {err}")
            time.sleep(3 * attempt)
    raise ConnectionError(f"Could not download {url} after {retries} attempts. Check that the Colab runtime has "
                          f"internet access and that data.cityofnewyork.us is reachable. Last error: {last_error}")

def api_row_count(dataset_id):
    response = http_get(SODA_RESOURCE.format(id=dataset_id, fmt="json"), params={"$select": "count(*) AS n"})
    return int(response.json()[0]["n"])

def download_source(name):
    # Download one dataset. Returns the raw file, the URL used and the row count reported by the API.
    meta = SOURCES[name]
    if meta["format"] == "csv":
        n_api = api_row_count(meta["id"])
        params = {"$limit": n_api + 1000, "$order": ":id"}    # ask for all rows, in a fixed order
        response = http_get(SODA_RESOURCE.format(id=meta["id"], fmt="csv"), params=params, timeout=600)
        return response.content, response.url, n_api
    try:
        response = http_get(SODA_RESOURCE.format(id=meta["id"], fmt="geojson"), params={"$limit": 5000})
    except ConnectionError:
        print("  falling back to the geospatial export endpoint")
        response = http_get(MODZCTA_EXPORT)
    return response.content, response.url, None

def raw_file(name, snapshot_date):
    ext = "geojson" if SOURCES[name]["format"] == "geojson" else "csv"
    return DIRS["raw"] / f"{name}_{SOURCES[name]['id']}_{snapshot_date}.{ext}"

In [8]:
# Get the three files in one of three ways: offline test files (only for testing the code),
# raw files saved on an earlier date, or (the normal case) a fresh download.
RAW_BYTES, MANIFEST = {}, {"sources": {}, "environment": ENVIRONMENT}

if OFFLINE_SOURCE_DIR:
    print("!" * 90 + "\nOFFLINE TEST MODE: reading local files from", OFFLINE_SOURCE_DIR,
          "\nResults are NOT real NYC Open Data results. Unset DEV_OFFLINE_SOURCE_DIR for the real run.\n" + "!" * 90)
    SOURCE_MODE, SNAPSHOT_DATE = "offline-test", datetime.now(timezone.utc).date().isoformat()
    local_names = {"housing": "housing.csv", "sirta": "sirta.csv", "modzcta": "modzcta.geojson"}
    for name, fname in local_names.items():
        RAW_BYTES[name] = (Path(OFFLINE_SOURCE_DIR) / fname).read_bytes()
        MANIFEST["sources"][name] = {"source_url": str(Path(OFFLINE_SOURCE_DIR) / fname), "api_row_count": None,
                                     "extracted_at_utc": datetime.now(timezone.utc).isoformat()}
elif RAW_SNAPSHOT_DATE:
    SOURCE_MODE, SNAPSHOT_DATE = "saved-snapshot", RAW_SNAPSHOT_DATE
    old_manifest_path = DIRS["raw"] / f"manifest_{SNAPSHOT_DATE}.json"
    old_manifest = json.loads(old_manifest_path.read_text()) if old_manifest_path.exists() else {"sources": {}}
    for name in SOURCES:
        path = raw_file(name, SNAPSHOT_DATE)
        if not path.exists():
            raise FileNotFoundError(f"RAW_SNAPSHOT_DATE={SNAPSHOT_DATE} but {path} is missing. "
                                    "Upload the saved raw files to data/raw or set RAW_SNAPSHOT_DATE = None.")
        RAW_BYTES[name] = path.read_bytes()
        MANIFEST["sources"][name] = old_manifest["sources"].get(name, {"source_url": str(path), "api_row_count": None,
                                                                       "extracted_at_utc": None})
else:
    SOURCE_MODE, SNAPSHOT_DATE = "fresh-download", datetime.now(timezone.utc).date().isoformat()
    for name in SOURCES:
        print(f"Downloading {name} ({SOURCES[name]['id']}) ...")
        content, url, n_api = download_source(name)
        RAW_BYTES[name] = content
        raw_file(name, SNAPSHOT_DATE).write_bytes(content)          # save the file exactly as downloaded
        MANIFEST["sources"][name] = {"source_url": url, "api_row_count": n_api,
                                     "extracted_at_utc": datetime.now(timezone.utc).isoformat()}

for name, content in RAW_BYTES.items():
    MANIFEST["sources"][name].update({"dataset_id": SOURCES[name]["id"], "title": SOURCES[name]["title"],
                                      "page": SOURCES[name]["page"], "bytes": len(content),
                                      "sha256": hashlib.sha256(content).hexdigest()})
MANIFEST.update({"snapshot_date": SNAPSHOT_DATE, "source_mode": SOURCE_MODE})
SNAPSHOT_TS = pd.Timestamp(SNAPSHOT_DATE)
print("Source mode:", SOURCE_MODE, "| snapshot date:", SNAPSHOT_DATE)

  attempt 1/3 failed: 503 Server Error: Service Unavailable for url: https://data.cityofnewyork.us/resource/xeqp-qz8h.json?%24select=count%28%2A%29+AS+n
  attempt 1/3 failed: 503 Server Error: Service Unavailable for url: https://data.cityofnewyork.us/resource/pri4-ifjk.geojson?%24limit=5000
Source mode: fresh-download | snapshot date: 2026-10-16


In [9]:
# Read every column as text first, so values like ZIP "11221" are not turned into 11221.0.
# Section 3.5 converts the columns to the right types and counts anything that cannot be converted.
h_raw = pd.read_csv(io.BytesIO(RAW_BYTES["housing"]), dtype=str, keep_default_na=True)
t_raw = pd.read_csv(io.BytesIO(RAW_BYTES["sirta"]), dtype=str, keep_default_na=True)
mz_raw = gpd.read_file(io.BytesIO(RAW_BYTES["modzcta"]))
mz_raw.columns = [c.lower() for c in mz_raw.columns]
mz_raw = mz_raw.set_geometry("geometry")

for name, frame in [("housing", h_raw), ("sirta", t_raw), ("modzcta", mz_raw)]:
    MANIFEST["sources"][name].update({"rows": int(len(frame)),
                                      "cols": int(frame.shape[1] - (1 if name == "modzcta" else 0))})
(DIRS["raw"] / f"manifest_{SNAPSHOT_DATE}.json").write_text(json.dumps(MANIFEST, indent=2, default=str))

pd.DataFrame(MANIFEST["sources"]).T[["dataset_id", "rows", "cols", "api_row_count", "bytes", "sha256", "extracted_at_utc"]]

,dataset_id,rows,cols,api_row_count,bytes,sha256,extracted_at_utc
housing,hg8x-zxpr,9551,41,9551,3026975,f77076f923e170ea6ba24662a9d5eed56fe81d4ab5b1e78905a8bdf11774fd74,2026-10-16T00:34:04.948525+00:00
sirta,xeqp-qz8h,13754,23,13754,2037656,ae5e20b2f2cc524a2ab5bb498ceab67d2b5cbee6ce8eaa60d9cbecd88fe418b1,2026-10-16T00:34:15.368587+00:00
modzcta,pri4-ifjk,178,4,None,3150358,bad811df00484f72985bbf49101dbb3096e565df977bf143f1c913e6e72b5a05,2026-10-16T00:34:38.222227+00:00


### 3.4 Checking the columns and the number of rows
If an expected column is missing, the notebook stops, because the later steps would give wrong results. If the number of rows differs from the October 2026 download, we only show a warning, because the datasets are updated regularly.

In [10]:
# Check the columns and the row counts of each download.
for name, frame in [("housing", h_raw), ("sirta", t_raw), ("modzcta", mz_raw)]:
    cols = [c for c in frame.columns if c != "geometry"]
    missing = sorted(set(EXPECTED_COLUMNS[name]) - set(cols))
    extra = sorted(set(cols) - set(EXPECTED_COLUMNS[name]))
    check(f"{name}_schema_complete", not missing, f"all {len(EXPECTED_COLUMNS[name])} expected columns present",
          f"missing columns {missing}: the source schema changed or the download is incomplete", stage="extraction")
    check(f"{name}_no_unexpected_columns", not extra, "no unexpected columns",
          f"unexpected columns {extra} (new fields in the source; review before use)", stage="extraction", severity="warn")
    api_n = MANIFEST["sources"][name].get("api_row_count")
    if api_n is not None:
        check(f"{name}_complete_download", len(frame) == api_n, f"downloaded {len(frame):,} rows = API count",
              f"downloaded {len(frame):,} rows but the API reports {api_n:,}: the download was truncated", stage="extraction")
    exp = EXPECTED_SNAPSHOT[name]
    same = len(frame) == exp["rows"] and ("cols" not in exp or len(cols) == exp["cols"])
    check(f"{name}_matches_recorded_snapshot", same, f"{len(frame):,} rows x {len(cols)} columns, as recorded on 2026-10-14",
          f"{len(frame):,} rows x {len(cols)} columns differs from the recorded 2026-10-14 snapshot "
          f"({exp['rows']:,} rows); this is a new snapshot, so recheck the counts quoted in the Markdown",
          stage="extraction", severity="warn")
check("sirta_more_than_default_page", len(t_raw) > 1000 and len(h_raw) > 1000,
      "both downloads exceed the API's default 1,000-row page", "a download has <= 1,000 rows: $limit was ignored",
      stage="extraction")

[PASS] housing_schema_complete: all 41 expected columns present
[PASS] housing_no_unexpected_columns: no unexpected columns
[PASS] housing_complete_download: downloaded 9,551 rows = API count
[PASS] housing_matches_recorded_snapshot: 9,551 rows x 41 columns, as recorded on 2026-10-14
[PASS] sirta_schema_complete: all 23 expected columns present
[PASS] sirta_no_unexpected_columns: no unexpected columns
[PASS] sirta_complete_download: downloaded 13,754 rows = API count
[PASS] sirta_matches_recorded_snapshot: 13,754 rows x 23 columns, as recorded on 2026-10-14
[PASS] modzcta_schema_complete: all 4 expected columns present
[PASS] modzcta_no_unexpected_columns: no unexpected columns
[PASS] modzcta_matches_recorded_snapshot: 178 rows x 4 columns, as recorded on 2026-10-14
[PASS] sirta_more_than_default_page: both downloads exceed the API's default 1,000-row page


True

### 3.5 Structure, column types and sample records
Here we look at a few sample rows, count the rows and columns, group the columns by role (ID, location, date, category, number), and convert dates and numbers to the right types. This is not cleaning yet: values are only converted, and any value that cannot be converted is counted so nothing disappears without notice.

In [11]:
# Show the first three rows of each dataset.
display(Markdown("**Sample housing records (first 3 rows, raw)**")); display(h_raw.head(3))
display(Markdown("**Sample SIRTA records (first 3 rows, raw)**")); display(t_raw.head(3))
display(Markdown("**Sample MODZCTA records (first 3 rows, without geometry)**")); display(pd.DataFrame(mz_raw.drop(columns="geometry")).head(3))

**Sample housing records (first 3 rows, raw)**

,project_id,project_name,project_start_date,project_completion_date,building_id,house_number,street_name,borough,postcode,bbl,bin,community_board,council_district,census_tract,neighborhood_tabulation_area,latitude,longitude,latitude_internal,longitude_internal,building_completion_date,reporting_construction_type,extended_affordability_status,prevailing_wage_status,extremely_low_income_units,very_low_income_units,low_income_units,moderate_income_units,middle_income_units,other_income_units,studio_units,_1_br_units,_2_br_units,_3_br_units,_4_br_units,_5_br_units,_6_br_units,unknown_br_units,counted_rental_units,counted_homeownership_units,all_counted_units,total_units
0,68872,177-06 WEXFORD TERRACE,2026-06-30T00:00:00.000,NaN,991637,177-06,WEXFORD TERRACE,Queens,11432,4098350023,4000000,QN-08,24,466,QN06,40.713096,-73.786005,40.712656,-73.786724,NaN,New Construction,No,Non Prevailing Wage,NaN,NaN,9,NaN,NaN,NaN,NaN,9,NaN,NaN,NaN,NaN,NaN,NaN,9,NaN,9,43
1,72436,ST. NICK'S. 819 GRAND STREET,2026-06-30T00:00:00.000,NaN,1021347,819,GRAND STREET,Brooklyn,11211,3029220001,3070436,BK-01,34,481,BK90,40.71205,-73.94036,40.712272,-73.940295,NaN,New Construction,No,Prevailing Wage,186,NaN,NaN,NaN,NaN,1,92,95,NaN,NaN,NaN,NaN,NaN,NaN,187,NaN,187,187
2,72449,BREMOND KING DAVIS.YR15.FY26,2026-06-30T00:00:00.000,NaN,21199,127,EDGECOMBE AVENUE,Manhattan,10030,1020480037,1060911,MN-10,9,22102,MN03,40.820728,-73.945285,40.820766,-73.945581,NaN,Preservation,No,Non Prevailing Wage,12,13,NaN,NaN,NaN,NaN,NaN,5,20,NaN,NaN,NaN,NaN,NaN,25,NaN,25,25


**Sample SIRTA records (first 3 rows, raw)**

,open_data_crash_id,crash_date,crash_time,crash_police_precinct,crash_borough,total_non_severe_injuries,total_severe_injuries,total_fatalities,latitude,longitude,on_street,cross_street,address,highway_reference_marker,zipcode,similar_fatalities_severe,similar_fatalities_all,community_board,council_district,bin,bbl,census_tract_2020_,neighborhood_tabulation_area_nta_2020_
0,4492364,2022-01-01T00:00:00.000,01:28,46,Bronx,1,1,0,40.8587070,-73.9037000,JEROME AVENUE,EAST 183 STREET,NaN,NaN,NaN,0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,4491159,2022-01-01T00:00:00.000,04:33,46,Bronx,0,1,0,NaN,NaN,JEROME AVENUE,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,4491487,2022-01-01T00:00:00.000,05:55,47,Bronx,0,1,0,40.8715500,-73.8642800,NaN,NaN,3207 WALLACE AVENUE,NaN,NaN,0,0,212,12,2056281,2045980025,372,BX1201


**Sample MODZCTA records (first 3 rows, without geometry)**

,modzcta,label,zcta,pop_est
0,10001,"10001, 10118","10001, 10119, 10199",23072
1,10002,10002,10002,74993
2,10003,10003,10003,54682


In [12]:
# Column groups used throughout the notebook, and the role of every column.
H_INCOME = ["extremely_low_income_units", "very_low_income_units", "low_income_units",
            "moderate_income_units", "middle_income_units", "other_income_units"]
H_BEDROOM = ["studio_units", "_1_br_units", "_2_br_units", "_3_br_units", "_4_br_units",
             "_5_br_units", "_6_br_units", "unknown_br_units"]
H_TENURE = ["counted_rental_units", "counted_homeownership_units"]
H_UNITS = H_INCOME + H_BEDROOM + H_TENURE + ["all_counted_units", "total_units"]
H_DATES = ["project_start_date", "project_completion_date", "building_completion_date"]
T_COUNTS = ["total_non_severe_injuries", "total_severe_injuries", "total_fatalities",
            "similar_fatalities_severe", "similar_fatalities_all"]

ROLES = {
    "housing": {"identifier": ["project_id", "building_id", "bbl", "bin"],
                "text": ["project_name", "house_number", "street_name"],
                "geographic": ["borough", "postcode", "community_board", "council_district", "census_tract",
                               "neighborhood_tabulation_area", "latitude", "longitude", "latitude_internal", "longitude_internal"],
                "temporal": H_DATES,
                "categorical": ["reporting_construction_type", "extended_affordability_status", "prevailing_wage_status"],
                "numeric": H_UNITS},
    "sirta": {"identifier": ["open_data_crash_id", "bin", "bbl"],
              "text": ["on_street", "cross_street", "address"],
              "geographic": ["crash_borough", "crash_police_precinct", "latitude", "longitude", "zipcode",
                             "highway_reference_marker", "community_board", "council_district",
                             "census_tract_2020_", "neighborhood_tabulation_area_nta_2020_"],
              "temporal": ["crash_date", "crash_time"],
              "numeric": T_COUNTS},
}

overview = pd.DataFrame([
    {"dataset": "Housing (H2)", "rows": len(h_raw), "columns": h_raw.shape[1],
     **{role: len(cols) for role, cols in ROLES["housing"].items()}},
    {"dataset": "SIRTA (T4)", "rows": len(t_raw), "columns": t_raw.shape[1],
     **{role: len(cols) for role, cols in ROLES["sirta"].items()}},
    {"dataset": "MODZCTA (reference)", "rows": len(mz_raw), "columns": mz_raw.shape[1] - 1},
]).fillna(0)
display(overview.astype({c: int for c in overview.columns if c != "dataset"}))

,dataset,rows,columns,identifier,text,geographic,temporal,categorical,numeric
0,Housing (H2),9551,41,4,3,10,3,3,18
1,SIRTA (T4),13754,23,3,3,10,2,0,5
2,MODZCTA (reference),178,4,0,0,0,0,0,0


In [13]:
# Convert the columns to the right types, then profile every column (type, missing values, distinct values).
def type_columns(raw, dates, numerics):
    # Convert the date and number columns and report how many values could not be converted.
    typed, report = raw.copy(), []
    for col in dates:
        parsed = pd.to_datetime(clean_text(raw[col]), format="ISO8601", errors="coerce")
        report.append({"column": col, "parsed_as": "date", "non_missing": int(raw[col].notna().sum()),
                       "unparseable": int((raw[col].notna() & parsed.isna()).sum())})
        typed[col] = parsed
    for col in numerics:
        num, bad = to_number(raw[col])
        report.append({"column": col, "parsed_as": "number", "non_missing": int(raw[col].notna().sum()), "unparseable": bad})
        typed[col] = num
    return typed, pd.DataFrame(report)

h_typed, h_parse_report = type_columns(h_raw, H_DATES, H_UNITS + ["latitude", "longitude"])
t_typed, t_parse_report = type_columns(t_raw, ["crash_date"], T_COUNTS + ["latitude", "longitude"])
parse_report = pd.concat([h_parse_report.assign(dataset="housing"), t_parse_report.assign(dataset="sirta")])
display(parse_report[parse_report["unparseable"] > 0] if (parse_report["unparseable"] > 0).any()
        else Markdown("All date and numeric columns parsed without any unparseable values."))
check("type_parsing_clean", (parse_report["unparseable"] == 0).all(), "no unparseable dates or numbers",
      f"{int(parse_report['unparseable'].sum())} unparseable values (listed above); they become missing and are counted",
      stage="typing", severity="warn")

def profile(df, roles):
    role_of = {c: r for r, cols in roles.items() for c in cols}
    return pd.DataFrame({"role": [role_of.get(c, "UNCLASSIFIED") for c in df.columns],
                         "dtype": df.dtypes.astype(str).values,
                         "non_missing": df.notna().sum().values,
                         "missing_pct": (df.isna().mean() * 100).round(1).values,
                         "distinct": df.nunique(dropna=True).values}, index=df.columns)

h_profile, t_profile = profile(h_typed, ROLES["housing"]), profile(t_typed, ROLES["sirta"])
display(Markdown("**Housing column profile**")); display(h_profile)
display(Markdown("**SIRTA column profile**")); display(t_profile)

All date and numeric columns parsed without any unparseable values.

[PASS] type_parsing_clean: no unparseable dates or numbers


**Housing column profile**

,role,dtype,non_missing,missing_pct,distinct
project_id,identifier,object,9551,0.0,5928
project_name,text,object,9551,0.0,3979
project_start_date,temporal,datetime64[ns],9551,0.0,2346
project_completion_date,temporal,datetime64[ns],7732,19.0,2066
building_id,identifier,object,7602,20.4,7478
house_number,text,object,9551,0.0,3154
street_name,text,object,9551,0.0,1567
borough,geographic,object,9551,0.0,5
postcode,geographic,object,7593,20.5,158
bbl,identifier,object,7530,21.2,6423


**SIRTA column profile**

,role,dtype,non_missing,missing_pct,distinct
open_data_crash_id,identifier,object,13310,3.2,13309
crash_date,temporal,datetime64[ns],13310,3.2,1641
crash_time,temporal,object,13310,3.2,1728
crash_police_precinct,geographic,object,13310,3.2,78
crash_borough,geographic,object,13255,3.6,5
total_non_severe_injuries,numeric,Int64,13310,3.2,16
total_severe_injuries,numeric,Int64,13310,3.2,8
total_fatalities,numeric,Int64,13310,3.2,6
latitude,geographic,Float64,10868,21.0,8570
longitude,geographic,Float64,10868,21.0,8219


## 4. Exploratory data analysis — Housing (H2)
We explore the data **before** cleaning it, so we can see which problems the cleaning has to fix.

### 4.1 Missing values

In [14]:
def missing_chart(prof, dataset_label):
    # Bar chart of the columns that have missing values (complete columns are left out).
    d = prof[prof["missing_pct"] > 0].sort_values("missing_pct").reset_index(names="column")
    fig = px.bar(d, x="missing_pct", y="column", orientation="h", color_discrete_sequence=[SERIES[0]],
                 hover_data={"non_missing": ":,", "missing_pct": ":.1f"},
                 labels={"missing_pct": "Missing (% of rows)", "column": ""})
    fig.update_traces(marker_line_width=0)
    return style(fig, f"{dataset_label}: share of missing values per column",
                 f"{len(d)} of {len(prof)} columns have missing values; complete columns are not shown",
                 height=max(320, 22 * len(d) + 140))

missing_chart(h_profile, "Housing").show()

**What the chart shows.** The income-band and bedroom columns are often blank. A blank could mean *zero units* or *not reported*, so we do not guess: Section 6.6 compares these columns with the totals to decide. The location columns (ZIP, coordinates, building ID) are blank together for the confidential buildings.

### 4.2 Descriptive statistics and categorical columns

In [15]:
# Basic statistics for the unit counts and the most common values of the category columns.
h_conf_raw = h_typed["project_name"].astype("string").str.strip().str.upper().eq("CONFIDENTIAL").fillna(False)
display(Markdown("**Numeric summary of unit counts (raw)**"))
display(h_typed[["all_counted_units", "total_units", "counted_rental_units", "counted_homeownership_units"] + H_INCOME]
        .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T.round(1))

display(Markdown("**Categorical distributions (raw)**"))
cat_tables = []
for col in ["borough", "reporting_construction_type", "extended_affordability_status", "prevailing_wage_status"]:
    vc = h_typed[col].value_counts(dropna=False).rename("buildings").reset_index().rename(columns={col: "value"})
    cat_tables.append(vc.assign(column=col)[["column", "value", "buildings"]])
display(pd.concat(cat_tables, ignore_index=True))

**Numeric summary of unit counts (raw)**

,count,mean,std,min,25%,50%,75%,95%,99%,max
all_counted_units,9522.0,33.7,65.8,1.0,2.0,8.0,30.0,161.0,338.6,917.0
total_units,9551.0,46.7,92.5,1.0,3.0,12.0,45.0,213.5,429.5,1320.0
counted_rental_units,6944.0,37.4,63.8,1.0,5.0,13.0,41.0,156.0,324.3,917.0
counted_homeownership_units,2730.0,22.3,68.3,1.0,1.0,1.0,2.0,175.0,384.0,489.0
extremely_low_income_units,2081.0,28.3,44.6,1.0,2.0,8.0,37.0,123.0,199.6,429.0
very_low_income_units,3400.0,25.5,49.3,1.0,3.0,7.0,21.0,121.0,263.0,453.0
low_income_units,5505.0,19.8,40.7,1.0,1.0,5.0,19.0,92.0,214.9,536.0
moderate_income_units,1694.0,11.6,30.3,1.0,1.0,3.0,8.0,49.3,149.1,454.0
middle_income_units,2691.0,16.9,34.5,1.0,3.0,6.0,15.0,70.0,172.4,448.0
other_income_units,1360.0,1.0,0.5,1.0,1.0,1.0,1.0,1.0,2.0,16.0


**Categorical distributions (raw)**

,column,value,buildings
0,borough,Brooklyn,3649
1,borough,Bronx,2630
2,borough,Manhattan,1485
3,borough,Queens,1342
4,borough,Staten Island,445
5,reporting_construction_type,Preservation,4785
6,reporting_construction_type,New Construction,4766
7,extended_affordability_status,No,8125
8,extended_affordability_status,Yes,1426
9,prevailing_wage_status,Non Prevailing Wage,9392


### 4.3 Dates: completions per year

In [16]:
# Units completed per year. Units in confidential buildings (no published location) get their own colour.
hh = h_typed[h_typed["building_completion_date"].notna()].assign(
    year=lambda d: d["building_completion_date"].dt.year.astype(int),
    location=np.where(h_conf_raw[h_typed["building_completion_date"].notna()], "Location confidential", "Location published"))
by_year = hh.groupby(["year", "location"], as_index=False).agg(units=("all_counted_units", "sum"), buildings=("project_id", "size"))
fig = px.bar(by_year, x="year", y="units", color="location", barmode="stack",
             color_discrete_map={"Location published": SERIES[0], "Location confidential": SERIES[1]},
             hover_data={"buildings": ":,", "units": ":,"},
             labels={"units": "Counted affordable units completed", "year": "Building completion year", "location": ""})
fig.add_vrect(x0=2021.5, x1=2025.5, fillcolor=SERIES[2], opacity=0.08, line_width=0,
              annotation_text="Study window", annotation_position="top left")
fig.update_xaxes(dtick=1)
style(fig, "Affordable units completed per year",
      f"{int(h_typed['building_completion_date'].isna().sum()):,} buildings have no completion date yet (in progress) and are not shown").show()

### 4.4 Outliers: building size

In [17]:
# Outliers: a few buildings are very large, so we apply the IQR rule on a log scale.
# Large buildings are only flagged, not removed, because large affordable projects are real.
units = h_typed["all_counted_units"].dropna()
log_u = np.log10(units[units > 0])
q1, q3 = log_u.quantile([0.25, 0.75])
UNITS_UPPER_FENCE = 10 ** (q3 + 1.5 * (q3 - q1))
fig = px.box(h_typed.dropna(subset=["all_counted_units"]), x="all_counted_units", y="reporting_construction_type",
             log_x=True, points="outliers", color="reporting_construction_type",
             color_discrete_sequence=SERIES[:2], hover_data=["project_id", "borough", "total_units"],
             labels={"all_counted_units": "Counted affordable units per building (log scale)", "reporting_construction_type": ""})
fig.add_vline(x=UNITS_UPPER_FENCE, line_dash="dash", line_color=INK_SOFT,
              annotation_text=f"log-IQR fence ≈ {UNITS_UPPER_FENCE:,.0f} units", annotation_position="top")
fig.update_layout(showlegend=False)
style(fig, "Building size distribution by construction type",
      "Points beyond the fence are unusually large buildings; they are flagged, not deleted").show()
display(Markdown("**Ten largest buildings (raw)**"))
display(h_typed.nlargest(10, "all_counted_units")[["project_id", "project_name", "borough", "reporting_construction_type",
                                                  "all_counted_units", "total_units", "building_completion_date"]])

**Ten largest buildings (raw)**

,project_id,project_name,borough,reporting_construction_type,all_counted_units,total_units,building_completion_date
6766,65361,711 SEA GIRT HDFC,Queens,Preservation,917,917,2017-11-21
3107,62191,NORTH COVE,Manhattan,New Construction,611,611,2025-10-23
3474,72581,3333 BROADWAY,Manhattan,Preservation,565,565,2022-06-30
4610,63772,BRONX POINT A,Bronx,New Construction,542,542,2023-11-02
7470,52774,JAMAICA CROSSING (HIGH-RISE),Queens,New Construction,539,539,2021-02-10
8893,55396,North Shore Plaza_ML,Staten Island,Preservation,536,536,2016-12-17
9266,52411,Twin Parks S.W.,Bronx,Preservation,536,536,2014-06-30
6021,61931,HUNTERS POINT SOUTH PARCEL C -NORTH TOWER,Queens,New Construction,535,800,2021-11-29
2424,67995,WILLETS POINT PHASE I BLDG 1 & 2,Queens,New Construction,534,534,2026-04-30
5220,66970,HUNTERS POINT SOUTH PARCELS F AND G,Queens,New Construction,517,689,2022-09-20


### 4.5 Consistency: affordable units vs total units

In [18]:
# A building cannot have more affordable units than total units. Points above the dotted line would break this rule.
rel = h_typed.dropna(subset=["all_counted_units", "total_units"])
fig = px.scatter(rel, x="total_units", y="all_counted_units", color="reporting_construction_type", log_x=True, log_y=True,
                 opacity=0.55, color_discrete_sequence=SERIES[:2], hover_data=["project_id", "borough"],
                 labels={"total_units": "Total units in building (log)", "all_counted_units": "Counted affordable units (log)",
                         "reporting_construction_type": "Construction type"})
mx = float(rel[["total_units", "all_counted_units"]].max().max())
fig.add_trace(go.Scatter(x=[1, mx], y=[1, mx], mode="lines", line=dict(color=INK_SOFT, dash="dot", width=2), name="counted = total"))
n_above = int((rel["all_counted_units"] > rel["total_units"]).sum())
style(fig, "Counted affordable units vs total units per building",
      f"{n_above} buildings lie above the dotted line (counted > total), which would be impossible").show()

### 4.6 Geographic fields, duplicates and examples of problematic records

In [19]:
# Duplicates: identical rows, and the same building listed twice in the same project.
h_key_rows = h_typed[h_typed["building_id"].notna()]
display(pd.DataFrame({"records": {
    "exact duplicate rows": int(h_raw.duplicated().sum()),
    "repeated (project_id, building_id) keys": int(h_key_rows.duplicated(["project_id", "building_id"]).sum()),
    "building ids appearing in more than one project": int((h_key_rows.groupby("building_id")["project_id"].nunique() > 1).sum()),
}}))

,records
exact duplicate rows,0
"repeated (project_id, building_id) keys",0
building ids appearing in more than one project,123


In [20]:
# How complete the location columns are, and a few examples of problem records.
display(Markdown("**Geographic completeness by confidentiality**"))
geo_complete = (h_typed.assign(confidential=h_conf_raw,
                               has_zip=h_typed["postcode"].notna(),
                               has_coordinates=h_typed["latitude"].notna() & h_typed["longitude"].notna(),
                               has_building_id=h_typed["building_id"].notna())
                .groupby("confidential")[["has_zip", "has_coordinates", "has_building_id"]].agg(["sum", "mean"]).round(3))
display(geo_complete)

display(Markdown("**Examples of problematic records**"))
order_issue = h_typed["building_completion_date"] < h_typed["project_start_date"]
examples_h = {
    "Building completed before project start": h_typed[order_issue].head(5),
    "Placeholder address text ('----')": h_typed[h_typed["house_number"].astype("string").str.strip().eq("----").fillna(False)].head(3),
    "House number turned into a date by a spreadsheet": h_typed[h_typed["house_number"].astype("string")
                                                      .str.fullmatch(rf"({MONTH_ABBR})-\d{{2}}", case=False).fillna(False)].head(3),
    "Placeholder BIN (borough code followed by zeros)": h_typed[h_typed["bin"].astype("string").str.fullmatch(r"[1-5]000000").fillna(False)].head(3),
}
show_cols = ["project_id", "project_name", "house_number", "street_name", "borough", "postcode", "bin",
             "project_start_date", "building_completion_date", "all_counted_units"]
for label, frame in examples_h.items():
    display(Markdown(f"*{label}* — {len(frame)} shown"))
    display(frame[show_cols])

**Geographic completeness by confidentiality**

has_zip        has_coordinates        has_building_id     
                 sum   mean             sum   mean             sum mean
confidential                                                           
False           7593  0.999            7593  0.999            7602  1.0
True               0  0.000               0  0.000               0  0.0

**Examples of problematic records**

*Building completed before project start* — 5 shown

,project_id,project_name,house_number,street_name,borough,postcode,bin,project_start_date,building_completion_date,all_counted_units
1340,76576,CONFIDENTIAL,----,----,Bronx,NaN,NaN,2025-02-27,2025-02-21,1
4714,70377,CONFIDENTIAL,----,----,Queens,NaN,NaN,2020-10-29,2020-10-09,1
5533,66670,JOE CENTRAL BROOKLYN LLC.YR15.FY19,518,CHESTER STREET,Brooklyn,11212,3082597,2019-06-27,2019-06-26,4
5534,66670,JOE CENTRAL BROOKLYN LLC.YR15.FY19,618,CHESTER STREET,Brooklyn,11212,3082812,2019-06-27,2019-06-26,4
5545,66670,JOE CENTRAL BROOKLYN LLC.YR15.FY19,609,GREENE AVENUE,Brooklyn,11221,3050267,2019-06-27,2019-06-26,6


...

,project_id,project_name,house_number,street_name,borough,postcode,bin,project_start_date,building_completion_date,all_counted_units
101,78674,CONFIDENTIAL,----,----,Brooklyn,NaN,NaN,2026-06-30,NaT,3
102,78759,CONFIDENTIAL,----,----,Bronx,NaN,NaN,2026-06-30,NaT,2
103,81944,CONFIDENTIAL,----,----,Bronx,NaN,NaN,2026-06-30,2026-06-30,1


*House number turned into a date by a spreadsheet* — 3 shown

,project_id,project_name,house_number,street_name,borough,postcode,bin,project_start_date,building_completion_date,all_counted_units
304,81991,2-65 WATJEAN APARTMENTS,Feb-65,WATJEAN COURT,Queens,11691,4000000,2026-05-07,NaT,12
779,76893,"MELECH MANN, HADASSAH MANN.PPP.FY26",Oct-32,BAY 24 STREET,Queens,11691,4300628,2025-09-30,2026-01-27,1
1001,75052,4-34 26TH AVENUE,Apr-34,26 AVENUE,Queens,11102,4000000,2025-06-18,NaT,42


*Placeholder BIN (borough code followed by zeros)* — 3 shown

,project_id,project_name,house_number,street_name,borough,postcode,bin,project_start_date,building_completion_date,all_counted_units
0,68872,177-06 WEXFORD TERRACE,177-06,WEXFORD TERRACE,Queens,11432,4000000,2026-06-30,NaT,9
7,72559,NRP-FOXY-SELFHELP. SOL ON PARK,3728,PARK AVENUE,Bronx,10456,2000000,2026-06-30,NaT,229
8,73047,FOXY. 1016 BEACH 19TH ST.,20-Oct,BEACH 19 STREET,Queens,11691,4000000,2026-06-30,NaT,138


## 5. Exploratory data analysis — Transportation (SIRTA)
The same checks for the crash data, plus the crash dates and times and how complete the location columns are each year.

### 5.1 Missing values and descriptive statistics

In [21]:
# Missing values and basic statistics for the injury, fatality and look-back counts.
missing_chart(t_profile, "SIRTA").show()
display(Markdown("**Numeric summary of injury, fatality and look-back counts (raw)**"))
display(t_typed[T_COUNTS].describe(percentiles=[0.5, 0.9, 0.99]).T.round(2))

**Numeric summary of injury, fatality and look-back counts (raw)**

,count,mean,std,min,50%,90%,99%,max
total_non_severe_injuries,13310.0,0.27,0.84,0.0,0.0,1.0,4.0,19.0
total_severe_injuries,13310.0,1.01,0.45,0.0,1.0,1.0,3.0,7.0
total_fatalities,13310.0,0.08,0.29,0.0,0.0,0.0,1.0,5.0
similar_fatalities_severe,11917.0,0.1,0.42,0.0,0.0,0.0,2.0,9.0
similar_fatalities_all,13310.0,1.54,5.86,0.0,0.0,4.0,18.0,296.0


**What the chart shows.** Community board, council district, census tract and NTA are mostly empty, so we cannot use them to join the datasets. The ZIP code is missing for many crashes and the coordinates for fewer — Section 5.3 shows that this changes a lot from year to year.

### 5.2 Dates, crash times and outliers

In [22]:
# Crashes per month. 2026 is only partly covered, so it is shaded.
dated = t_typed[t_typed["crash_date"].notna()]
monthly = dated.set_index("crash_date").resample("MS").size().rename("crashes").reset_index()
fig = px.line(monthly, x="crash_date", y="crashes", markers=True, color_discrete_sequence=[SERIES[0]],
              labels={"crash_date": "Month", "crashes": "Serious crashes"})
fig.update_traces(line_width=2, marker_size=6, hovertemplate="%{x|%b %Y}: %{y:,} crashes<extra></extra>")
fig.add_vrect(x0="2026-01-01", x1=str(monthly["crash_date"].max() + pd.offsets.MonthEnd(0)), fillcolor=NEUTRAL,
              opacity=0.25, line_width=0, annotation_text="2026: partial year", annotation_position="top left")
style(fig, "SIRTA serious crashes per month",
      f"{len(dated):,} dated records from {dated['crash_date'].min():%d %b %Y} to {dated['crash_date'].max():%d %b %Y}").show()

borough_year = (dated.assign(year=dated["crash_date"].dt.year, borough=dated["crash_borough"].fillna("(missing)"))
                .pivot_table(index="borough", columns="year", values="crash_date", aggfunc="size", fill_value=0))
display(Markdown("**Crashes by borough and year (raw)**")); display(borough_year)

**Crashes by borough and year (raw)**

year,2022,2023,2024,2025,2026
borough,,,,,
(missing),0,0,2,34,19
Bronx,600,600,622,543,252
Brooklyn,885,874,919,879,397
Manhattan,570,653,639,586,285
Queens,673,733,801,767,308
Staten Island,151,156,133,138,91


In [23]:
# Crash times: 00:00 appears much more often than other times. It is probably used when the real time
# is unknown, so later we leave these crashes out of the time-of-day analysis.
hhmm_eda, hour_eda, status_eda = normalize_time(t_typed["crash_time"])
time_df = pd.DataFrame({"hour": hour_eda, "kind": np.where(status_eda.eq("exact_midnight"), "Recorded exactly 00:00", "Other times")})
time_df = time_df.dropna(subset=["hour"]).groupby(["hour", "kind"], as_index=False).size()
on_the_hour = hhmm_eda[hhmm_eda.str.endswith(":00").fillna(False) & hhmm_eda.ne("00:00").fillna(False)].value_counts()
midnight_ratio = int(hhmm_eda.eq("00:00").sum()) / on_the_hour.mean() if len(on_the_hour) else np.nan
fig = px.bar(time_df, x="hour", y="size", color="kind", barmode="stack",
             color_discrete_map={"Other times": SERIES[0], "Recorded exactly 00:00": SERIES[1]},
             labels={"hour": "Hour of day", "size": "Crashes", "kind": ""})
fig.update_xaxes(dtick=2)
style(fig, "Crashes by hour of day",
      f"Exactly 00:00 occurs {midnight_ratio:.1f}× as often as an average other on-the-hour time; it is treated as uncertain").show()

In [24]:
# Outliers: how many people were injured or killed in each crash. Crashes with many victims are possible, so we keep them.
people = t_typed[["total_non_severe_injuries", "total_severe_injuries", "total_fatalities"]].sum(axis=1, min_count=1)
people_counts = people.dropna().astype(int).clip(upper=10).value_counts().sort_index().rename_axis("people").reset_index(name="crashes")
people_counts["label"] = people_counts["people"].astype(str).replace({"10": "10+"})
fig = px.bar(people_counts, x="label", y="crashes", color_discrete_sequence=[SERIES[0]],
             labels={"label": "People injured or killed in the crash", "crashes": "Crashes"})
style(fig, "People injured or killed per serious crash",
      f"Maximum {int(people.max())}; multi-victim crashes are plausible and kept").show()

### 5.3 Which location columns each crash has, by year
This matters for the integration: if crashes cannot be placed in an area, they cannot be counted in that area.

In [25]:
# For each year: how many crashes have a ZIP code, coordinates, both or neither.
loc = dated.assign(year=dated["crash_date"].dt.year,
                   has_zip=dated["zipcode"].notna(),
                   has_coords=dated["latitude"].notna() & dated["longitude"].notna())
loc["availability"] = np.select([loc["has_zip"] & loc["has_coords"], loc["has_coords"], loc["has_zip"]],
                                ["ZIP and coordinates", "Coordinates only", "ZIP only"], default="Neither")
avail = loc.groupby(["year", "availability"], as_index=False).size()
avail["share"] = avail["size"] / avail.groupby("year")["size"].transform("sum") * 100
fig = px.bar(avail, x="year", y="share", color="availability", barmode="stack",
             category_orders={"availability": ["ZIP and coordinates", "Coordinates only", "ZIP only", "Neither"]},
             color_discrete_map={"ZIP and coordinates": SERIES[0], "Coordinates only": SERIES[2],
                                 "ZIP only": SERIES[3], "Neither": NEUTRAL},
             hover_data={"size": ":,", "share": ":.1f"},
             labels={"share": "Share of the year's crashes (%)", "year": "Crash year", "availability": "Location fields"})
style(fig, "Which location fields each crash has, by year",
      "The ZIP field alone is far less complete in 2022 than later; coordinates fill most of the gap").show()
display(loc.groupby("year")[["has_zip", "has_coords"]].mean().mul(100).round(1).rename(columns=lambda c: c + "_pct"))

,has_zip_pct,has_coords_pct
year,,
2022,16.3,89.0
2023,68.1,81.5
2024,70.0,66.2
2025,80.3,88.1
2026,74.0,87.6


### 5.4 Duplicates and problematic records

In [26]:
# Examples of problem records: rows with no id and no date, identical rows, repeated collision ids and the placeholder ZIP.
display(Markdown("**Examples of problematic records**"))
t_orphan_raw = t_typed["open_data_crash_id"].isna() & t_typed["crash_date"].isna()
id_counts = t_typed["open_data_crash_id"].value_counts()
examples_t = {
    "Rows with no crash id and no date (orphans)": t_typed[t_orphan_raw].head(3),
    "Exact duplicate rows (all columns identical)": t_typed[t_raw.duplicated(keep=False)].head(6),
    "Collision ids that appear more than once": t_typed[t_typed["open_data_crash_id"].isin(id_counts[id_counts > 1].index)],
    "Placeholder ZIP 10000": t_typed[t_typed["zipcode"].astype("string").str.strip().eq("10000").fillna(False)].head(3),
}
for label, frame in examples_t.items():
    display(Markdown(f"*{label}* — {len(frame)} shown"))
    display(frame.drop(columns=["address", "bin", "bbl"], errors="ignore"))

**Examples of problematic records**

*Rows with no crash id and no date (orphans)* — 3 shown

,open_data_crash_id,crash_date,crash_time,crash_police_precinct,crash_borough,total_non_severe_injuries,total_severe_injuries,total_fatalities,latitude,longitude,on_street,cross_street,highway_reference_marker,zipcode,similar_fatalities_severe,similar_fatalities_all,community_board,council_district,census_tract_2020_,neighborhood_tabulation_area_nta_2020_
4248,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,305,42,1164,BK0502
4249,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,309,35,333,BK0901
4250,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,310,47,38,BK1001


*Exact duplicate rows (all columns identical)* — 5 shown

,open_data_crash_id,crash_date,crash_time,crash_police_precinct,crash_borough,total_non_severe_injuries,total_severe_injuries,total_fatalities,latitude,longitude,on_street,cross_street,highway_reference_marker,zipcode,similar_fatalities_severe,similar_fatalities_all,community_board,council_district,census_tract_2020_,neighborhood_tabulation_area_nta_2020_
4356,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,302,35,35,BK0203
4432,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,103,1,201,MN0301
4433,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,103,1,201,MN0301
4506,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,302,35,35,BK0203
4526,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,302,35,35,BK0203


*Collision ids that appear more than once* — 2 shown

,open_data_crash_id,crash_date,crash_time,crash_police_precinct,crash_borough,total_non_severe_injuries,total_severe_injuries,total_fatalities,latitude,longitude,on_street,cross_street,highway_reference_marker,zipcode,similar_fatalities_severe,similar_fatalities_all,community_board,council_district,census_tract_2020_,neighborhood_tabulation_area_nta_2020_
2147,4597666,2023-01-13,21:01,44,Bronx,0,1,0,<NA>,<NA>,WEST 170 STREET,Plaza Dr,NaN,NaN,1,5,NaN,NaN,NaN,NaN
10709,4597666,2026-01-13,21:01,44,Bronx,0,1,1,<NA>,<NA>,WEST 170 STREET,Plaza Dr,NaN,NaN,0,0,NaN,NaN,NaN,NaN


*Placeholder ZIP 10000* — 3 shown

,open_data_crash_id,crash_date,crash_time,crash_police_precinct,crash_borough,total_non_severe_injuries,total_severe_injuries,total_fatalities,latitude,longitude,on_street,cross_street,highway_reference_marker,zipcode,similar_fatalities_severe,similar_fatalities_all,community_board,council_district,census_tract_2020_,neighborhood_tabulation_area_nta_2020_
3119,4576497,2022-10-26,07:54,22,Manhattan,0,1,0,40.771538,-73.97742,WEST DRIVE,TRANSVERSE ROAD NUMBER ONE,NaN,10000,0,0,NaN,NaN,NaN,NaN
3792,4629630,2023-05-16,13:31,22,Manhattan,0,1,0,40.76871,-73.969986,NaN,NaN,NaN,10000,0,0,NaN,NaN,NaN,NaN
4100,4638550,2023-06-18,14:39,22,Manhattan,0,1,0,40.781506,-73.962685,NaN,NaN,NaN,10000,0,0,NaN,NaN,NaN,NaN


### 5.5 Summary of the problems found (both datasets)
This table is calculated from the raw data. It lists each problem and how we plan to handle it in the cleaning sections (6 and 7).

In [27]:
# Count each problem in the raw data and note how the cleaning will handle it.
h_bid = h_typed[h_typed["building_id"].notna()]
initial_findings = pd.DataFrame([
    ("Housing", "Confidential buildings without location", int(h_conf_raw.sum()), "Keep; borough totals only; excluded from ZIP-area metrics"),
    ("Housing", "Exact duplicate rows", int(h_raw.duplicated().sum()), "Remove if any"),
    ("Housing", "Duplicate (project_id, building_id) keys", int(h_bid.duplicated(["project_id", "building_id"]).sum()), "Must be zero"),
    ("Housing", "Building IDs in more than one project", int((h_bid.groupby("building_id")["project_id"].nunique() > 1).sum()), "Keep; flag"),
    ("Housing", "Building completed before project start", int(order_issue.sum()), "Keep; flag"),
    ("Housing", "No building completion date (in progress)", int(h_typed["building_completion_date"].isna().sum()), "Status, not an error; used for RQ8 (unfinished projects)"),
    ("Housing", "Counted units > total units", int((h_typed["all_counted_units"] > h_typed["total_units"]).sum()), "Exclude unit values if any"),
    ("Housing", "all_counted_units missing", int(h_typed["all_counted_units"].isna().sum()), "Derive only if two component groups agree"),
    ("SIRTA", "Orphan rows (no id, no date)", int(t_orphan_raw.sum()), "Exclude with reason"),
    ("SIRTA", "Exact duplicate rows", int(t_raw.duplicated().sum()), "Exclude extra copies"),
    ("SIRTA", "Collision ids appearing more than once", int((id_counts > 1).sum()), "Investigate (Section 7.6)"),
    ("SIRTA", "Placeholder ZIP 10000", int(t_typed["zipcode"].astype("string").str.strip().eq("10000").sum()), "Set to missing"),
    ("SIRTA", "Crash time exactly 00:00", int(hhmm_eda.eq("00:00").sum()), "Keep; exclude from hour-of-day analysis"),
    ("SIRTA", "Dated crashes with neither ZIP nor coordinates", int((~loc["has_zip"] & ~loc["has_coords"]).sum()), "Borough totals only"),
], columns=["dataset", "issue", "records", "planned handling"])
initial_findings

,dataset,issue,records,planned handling
0,Housing,Confidential buildings without location,1949,Keep; borough totals only; excluded from ZIP-area metrics
1,Housing,Exact duplicate rows,0,Remove if any
2,Housing,"Duplicate (project_id, building_id) keys",0,Must be zero
3,Housing,Building IDs in more than one project,123,Keep; flag
4,Housing,Building completed before project start,20,Keep; flag
5,Housing,No building completion date (in progress),1666,"Status, not an error; used for RQ8 (unfinished projects)"
6,Housing,Counted units > total units,0,Exclude unit values if any
7,Housing,all_counted_units missing,29,Derive only if two component groups agree
8,SIRTA,"Orphan rows (no id, no date)",444,Exclude with reason
9,SIRTA,Exact duplicate rows,3,Exclude extra copies


## 6. Pre-integration cleaning — Housing (H2)
**Our rule for the housing data.** We do not delete any housing row. We only correct a value when we know the right value for sure (for example, a placeholder such as `----` becomes missing). Everything else gets a **flag** column, so each analysis can decide whether to include those rows. Rows that cannot enter the ZIP-area analysis are counted in the record ledger (Section 9.3).

### 6.1 Text and category standardisation
We make spelling and capital letters consistent, so that boroughs and construction types are grouped correctly. The construction type must be exactly *New Construction* or *Preservation*.

In [28]:
# Work on a copy of the housing data and give every row a stable id (H00000, H00001, ...).
h = h_typed.copy()
h.insert(0, "record_id", [f"H{i:05d}" for i in range(len(h))])
H_ROWS_START = len(h)
h_before = h.copy()          # keep a copy to compare before and after cleaning (Section 6.12)

# 6.1 Remove extra spaces and use the same spelling for boroughs and construction types.
for col in ["project_name", "house_number", "street_name", "borough", "community_board", "neighborhood_tabulation_area",
            "reporting_construction_type", "extended_affordability_status", "prevailing_wage_status"]:
    h[col] = clean_text(h[col])
h["borough"] = h["borough"].str.title()
CONSTRUCTION_MAP = {"new construction": "New Construction", "preservation": "Preservation"}
h["construction_type"] = h["reporting_construction_type"].str.lower().map(CONSTRUCTION_MAP).astype("string")
h["extended_affordability"] = h["extended_affordability_status"].str.lower().map({"yes": True, "no": False}).astype("boolean")
h["prevailing_wage"] = h["prevailing_wage_status"].str.lower().map(
    {"prevailing wage": True, "non prevailing wage": False}).astype("boolean")

bad_boro = h["borough"].notna() & ~h["borough"].isin(BOROUGHS)
bad_type = h["reporting_construction_type"].notna() & h["construction_type"].isna()
check("housing_borough_values", bad_boro.sum() == 0, "every borough value is one of the five boroughs",
      f"{int(bad_boro.sum())} unexpected borough values: {h.loc[bad_boro, 'borough'].unique()[:5]}", stage="housing cleaning")
check("housing_construction_type_values", bad_type.sum() == 0, "construction type is always New Construction or Preservation",
      f"{int(bad_type.sum())} unmapped construction types: {h.loc[bad_type, 'reporting_construction_type'].unique()[:5]}",
      stage="housing cleaning")
log_step("housing", "6.1", "standardise text, borough, construction type, yes/no flags", 0, H_ROWS_START, len(h),
         "consistent categories for grouping and filtering")
h["construction_type"].value_counts(dropna=False).to_frame("buildings")

[PASS] housing_borough_values: every borough value is one of the five boroughs
[PASS] housing_construction_type_values: construction type is always New Construction or Preservation


,buildings
construction_type,
Preservation,4785
New Construction,4766


### 6.2 Placeholders and identifiers
Placeholder values such as `----` or the building number (BIN) `3000000` are not real data, so they become missing. IDs are stored as clean text.

In [29]:
# 6.2 Replace placeholder values with missing values and clean the ID columns.
TEXT_PLACEHOLDERS = {"----", "N/A", "NA", "NONE"}
n_text_ph = 0
for col in ["house_number", "street_name"]:
    is_ph = h[col].str.upper().isin(TEXT_PLACEHOLDERS).fillna(False)
    n_text_ph += int(is_ph.sum())
    h.loc[is_ph, col] = pd.NA
for col in ["project_id", "building_id", "bbl", "bin", "council_district"]:
    h[col] = normalize_id(h[col])
h["bin_placeholder"] = h["bin"].str.fullmatch(r"[1-5]000000").fillna(False).astype(bool)
h.loc[h["bin_placeholder"], "bin"] = pd.NA
# A house number like "Oct-44" was probably a Queens address such as "10-44" that a spreadsheet turned into a date.
# We cannot be sure what the original was, so we only flag it and do not change it.
h["house_number_date_like"] = h["house_number"].str.fullmatch(rf"({MONTH_ABBR})-\d{{2}}", case=False).fillna(False).astype(bool)
non_numeric_ids = {c: int((h[c].notna() & ~h[c].str.fullmatch(r"\d+").fillna(False)).sum()) for c in ["project_id", "building_id"]}
check("housing_ids_numeric", sum(non_numeric_ids.values()) == 0, "project and building ids are numeric text",
      f"non-numeric ids found: {non_numeric_ids}", stage="housing cleaning", severity="warn")
log_step("housing", "6.2", "address placeholders '----' -> missing", n_text_ph, len(h), len(h), "placeholder is not an address")
log_step("housing", "6.2", "placeholder BINs (e.g. 3000000) -> missing, flagged", int(h["bin_placeholder"].sum()), len(h), len(h),
         "a borough code followed by zeros is not a real building number")
log_step("housing", "6.2", "date-like house numbers flagged", int(h["house_number_date_like"].sum()), len(h), len(h),
         "original address cannot be recovered with certainty")
print(f"Address placeholders cleared: {n_text_ph:,} | placeholder BINs: {int(h['bin_placeholder'].sum()):,} | "
      f"date-like house numbers: {int(h['house_number_date_like'].sum())}")

[PASS] housing_ids_numeric: project and building ids are numeric text
Address placeholders cleared: 3,898 | placeholder BINs: 482 | date-like house numbers: 7


### 6.3 ZIP codes and coordinates
Only complete 5-digit NYC ZIP codes and coordinates inside New York City can be used to place a building in a ZIP area. Everything else becomes missing.

In [30]:
# 6.3 Keep only valid NYC ZIP codes and coordinates that lie inside the city.
h["zip5"], h["zip_status"] = parse_zip_strict(h["postcode"])
coords_ok = h["latitude"].between(*NYC_LAT_RANGE) & h["longitude"].between(*NYC_LON_RANGE)
h["coords_invalid"] = h["latitude"].notna() & h["longitude"].notna() & ~coords_ok
h.loc[~coords_ok, ["latitude", "longitude"]] = np.nan
h["has_coords"] = coords_ok.fillna(False).astype(bool)
log_step("housing", "6.3", "ZIP parsed strictly; invalid ZIPs -> missing", int((h["zip_status"] != "valid").sum() - h["postcode"].isna().sum()),
         len(h), len(h), "only complete 5-digit NYC ZIPs are usable")
log_step("housing", "6.3", "coordinates outside NYC -> missing", int(h["coords_invalid"].sum()), len(h), len(h),
         "a point outside the city cannot be placed in a ZIP area")
h["zip_status"].value_counts().to_frame("buildings")

,buildings
zip_status,
valid,7593
missing,1958


### 6.4 Dates: range and order
All dates should fall inside the period the dataset covers, and a building should not be completed before its project starts.

In [31]:
# 6.4 Check that the dates are in range and in the right order.
SCOPE_START = pd.Timestamp("2014-01-01")      # the dataset covers projects started on or after 1 January 2014
h["date_out_of_scope"] = (
    (h["project_start_date"].notna() & ~h["project_start_date"].between(SCOPE_START, SNAPSHOT_TS)) |
    (h["building_completion_date"] > SNAPSHOT_TS) | (h["building_completion_date"] < SCOPE_START)).fillna(False)
h["date_order_issue"] = (h["building_completion_date"] < h["project_start_date"]).fillna(False)
h["date_order_gap_days"] = (h["project_start_date"] - h["building_completion_date"]).dt.days.where(h["date_order_issue"]).astype("Int64")
check("housing_dates_in_scope", h["date_out_of_scope"].sum() == 0, "all dates fall between 2014-01-01 and the snapshot date",
      f"{int(h['date_out_of_scope'].sum())} records have dates outside the dataset scope", stage="housing cleaning", severity="warn")
n_order = int(h["date_order_issue"].sum())
record_anomaly("HOUSING-DATE-ORDER", "housing", n_order,
               "building completion date earlier than project start date",
               "retained with the completion year as reported; listed as a limitation (Section 12)")
log_step("housing", "6.4", "completion-before-start records flagged", n_order, len(h), len(h), "cannot tell which date is wrong")
display(Markdown(f"**{n_order} buildings completed before their project start.** Gap in days:"))
display(h.loc[h["date_order_issue"], "date_order_gap_days"].describe().to_frame("gap_days").T.round(0))
display(h.loc[h["date_order_issue"], ["record_id", "project_id", "construction_type", "project_start_date",
                                      "building_completion_date", "date_order_gap_days", "all_counted_units"]]
        .sort_values("date_order_gap_days", ascending=False).head(10))

[PASS] housing_dates_in_scope: all dates fall between 2014-01-01 and the snapshot date


**20 buildings completed before their project start.** Gap in days:

,count,mean,std,min,25%,50%,75%,max
gap_days,20.0,79.0,133.0,1.0,1.0,4.0,84.0,357.0


,record_id,project_id,construction_type,project_start_date,building_completion_date,date_order_gap_days,all_counted_units
8132,H08132,49635,Preservation,2015-12-31,2015-01-08,357,1
8129,H08129,49635,Preservation,2015-12-31,2015-02-07,327,1
8127,H08127,49635,Preservation,2015-12-31,2015-02-26,308,1
8126,H08126,49635,Preservation,2015-12-31,2015-04-02,273,1
8125,H08125,49635,Preservation,2015-12-31,2015-05-04,241,1
8135,H08135,49635,Preservation,2015-12-31,2015-11-30,31,1
4714,H04714,70377,New Construction,2020-10-29,2020-10-09,20,1
1340,H01340,76576,New Construction,2025-02-27,2025-02-21,6,1
6423,H06423,61837,Preservation,2018-06-25,2018-06-19,6,20
6422,H06422,61837,Preservation,2018-06-25,2018-06-19,6,24


**Decision.** For preservation work, a building may have been finished before the city's financing (the project start) was recorded, but the data cannot confirm this, and we cannot tell which date is wrong. So we keep these records with their reported completion year and flag them. Section 12 lists how many there are.

### 6.5 Impossible numbers
A building cannot have a negative or fractional number of units. Such values become missing; we do not guess a replacement.

In [32]:
# 6.5 Negative or fractional unit counts are impossible, so they become missing.
negatives = {c: int((h[c] < 0).sum()) for c in H_UNITS}
non_integer = {c: int((h[c].notna() & (h[c] % 1 != 0)).sum()) for c in H_UNITS}
for c in H_UNITS:
    h.loc[(h[c] < 0) | (h[c].notna() & (h[c] % 1 != 0)), c] = np.nan     # impossible value -> missing
check("housing_no_negative_units", sum(negatives.values()) == 0, "no negative unit counts",
      f"negative unit counts set to missing: { {k: v for k, v in negatives.items() if v} }", stage="housing cleaning", severity="warn")
check("housing_integer_units", sum(non_integer.values()) == 0, "all unit counts are whole numbers",
      f"non-integer unit counts set to missing: { {k: v for k, v in non_integer.items() if v} }", stage="housing cleaning", severity="warn")
log_step("housing", "6.5", "negative or fractional unit counts -> missing", sum(negatives.values()) + sum(non_integer.values()),
         len(h), len(h), "a building cannot have negative or fractional units")

[PASS] housing_no_negative_units: no negative unit counts
[PASS] housing_integer_units: all unit counts are whole numbers


### 6.6 Are blank unit columns zeros or unknown values?
A blank income-band cell could mean *zero units in this band* or *not reported*. Instead of guessing, we compare each group of columns with the building's total:
* if the reported numbers add up exactly to the total, the blanks in that row must be zeros, so we fill them with 0;
* if all columns in the group are blank, the breakdown is **unknown** and stays missing;
* if they do not add up, the row is marked as a **mismatch** and left unchanged.

The table below also shows which total (`all_counted_units` or `total_units`) each group should be compared with. We pick the one that matches most often in the data.

In [33]:
# The three groups of unit columns that should add up to a building total.
COMPONENT_GROUPS = {"income": H_INCOME, "bedroom": H_BEDROOM, "tenure": H_TENURE}

def reconcile_components(df, components, total_col):
    # For each row, compare the sum of the columns with the total:
    # total_missing, components_missing (all blank), match, or mismatch.
    comp = df[components]
    n_present = comp.notna().sum(axis=1)
    comp_sum = comp.sum(axis=1, min_count=1)
    total = df[total_col]
    status = np.select([as_bool(total.isna()), as_bool(n_present.eq(0)), as_bool(comp_sum.eq(total))],
                       ["total_missing", "components_missing", "match"], default="mismatch")
    return pd.Series(status, index=df.index), comp_sum

recon = []
for group, comps in COMPONENT_GROUPS.items():
    for total in ["all_counted_units", "total_units"]:
        st, _ = reconcile_components(h, comps, total)
        recon.append({"group": group, "compared_with": total, **st.value_counts().to_dict()})
recon = pd.DataFrame(recon).fillna(0).set_index(["group", "compared_with"]).astype(int)
if "match" not in recon.columns:
    recon["match"] = 0
display(recon)
REFERENCE_TOTAL = {g: recon.loc[g]["match"].idxmax() for g in COMPONENT_GROUPS}
print("Reference total chosen for each component group:", REFERENCE_TOTAL)

match  total_missing  mismatch  components_missing
group   compared_with                                                        
income  all_counted_units   9522             29         0                   0
        total_units         6770              0      2752                  29
bedroom all_counted_units   9522             29         0                   0
        total_units         6770              0      2752                  29
tenure  all_counted_units   9522             29         0                   0
        total_units         6770              0      2752                  29

Reference total chosen for each component group: {'income': 'all_counted_units', 'bedroom': 'all_counted_units', 'tenure': 'all_counted_units'}


### 6.7–6.8 Filling missing values only when we know the answer
A missing total is filled only when two separate breakdowns (income bands and rental/homeownership) give exactly the same number. Blank income-band cells become 0 only when the reported bands add up exactly to the total. All other blanks stay missing.

In [34]:
# 6.7 Fill a missing total only when the income-band sum and the rental + homeownership sum agree.
income_sum = h[H_INCOME].sum(axis=1, min_count=1)
tenure_sum = h[H_TENURE].sum(axis=1, min_count=1)
derivable = h["all_counted_units"].isna() & income_sum.notna() & tenure_sum.notna() & income_sum.eq(tenure_sum) & income_sum.gt(0)
h["units_counted"] = h["all_counted_units"].where(~derivable, income_sum)
h["units_counted_derived"] = derivable.fillna(False).astype(bool)
log_step("housing", "6.7", "missing all_counted_units derived from agreeing income and tenure sums",
         int(h["units_counted_derived"].sum()), len(h), len(h), "two independent breakdowns give the same total")

# 6.8 Where the breakdown adds up exactly to the total, the blank cells must be 0, so we fill them.
blank_before = int(h[H_INCOME + H_BEDROOM + H_TENURE].isna().sum().sum())
for group, comps in COMPONENT_GROUPS.items():
    total_col = "units_counted" if REFERENCE_TOTAL[group] == "all_counted_units" else REFERENCE_TOTAL[group]
    status, _ = reconcile_components(h, comps, total_col)
    h[f"{group}_breakdown_status"] = status.astype("string")
    match = status.eq("match")
    h.loc[match, comps] = h.loc[match, comps].fillna(0)
blank_after = int(h[H_INCOME + H_BEDROOM + H_TENURE].isna().sum().sum())
log_step("housing", "6.8", "structural zeros filled in reconciled breakdowns", blank_before - blank_after, len(h), len(h),
         "components add up exactly to the reference total, so blanks are zeros")
display(pd.DataFrame({g: h[f"{g}_breakdown_status"].value_counts() for g in COMPONENT_GROUPS}).fillna(0).astype(int))
print(f"Blank component cells: {blank_before:,} before -> {blank_after:,} after; "
      f"{int(h['units_counted'].isna().sum())} buildings still have unknown counted units")

,income,bedroom,tenure
match,9522,9522,9522
total_missing,29,29,29


Blank component cells: 106,280 before -> 464 after; 29 buildings still have unknown counted units


### 6.9 Outliers and impossible unit totals
If a building reports more affordable units than total units, we cannot tell which number is wrong, so its units are left out of the unit sums. Very large buildings are real (large affordable projects exist): they are flagged by the log-scale IQR rule from Section 4.4 and kept.

In [35]:
# 6.9 Flag very large buildings (kept) and remove impossible unit values.
h["counted_exceeds_total"] = (h["units_counted"] > h["total_units"]).fillna(False)
h["total_units_nonpositive"] = (h["total_units"] <= 0).fillna(False)
h["units_size_outlier"] = (h["units_counted"] > UNITS_UPPER_FENCE).fillna(False)
impossible = h["counted_exceeds_total"] | h["total_units_nonpositive"]
# Affordable units > total units is impossible and we cannot tell which number is wrong,
# so the unit value is set to missing and left out of the sums.
h.loc[impossible, "units_counted"] = np.nan
h["units_unknown"] = h["units_counted"].isna()
log_step("housing", "6.9", "impossible unit combinations (counted > total, total <= 0) -> units missing",
         int(impossible.sum()), len(h), len(h), "logically impossible; cannot tell which value is wrong")
log_step("housing", "6.9", "very large buildings flagged (log-IQR), kept", int(h["units_size_outlier"].sum()), len(h), len(h),
         "large affordable developments are plausible")
for c in H_UNITS + ["units_counted"]:
    h[c] = h[c].round().astype("Int64")
print(f"Impossible unit combinations: {int(impossible.sum())} | large-building flags (kept): {int(h['units_size_outlier'].sum())}")

Impossible unit combinations: 0 | large-building flags (kept): 0


### 6.10 Duplicates and buildings in several projects
One row should be one building in one project. We check that there are no identical rows and no repeated (project, building) pairs, and we look at buildings that appear in more than one project.

In [36]:
# 6.10 Check for duplicate rows and keys, and find buildings that belong to more than one project.
h["is_confidential"] = h["project_name"].str.upper().eq("CONFIDENTIAL").fillna(False).astype(bool)
nonconf = ~h["is_confidential"]
with_bid = h[h["building_id"].notna()]
check("housing_project_id_present", h["project_id"].notna().all(), "every row has a project id",
      f"{int(h['project_id'].isna().sum())} rows lack a project id", stage="housing cleaning")
check("housing_key_unique", not with_bid.duplicated(["project_id", "building_id"]).any(),
      "(project_id, building_id) is unique: one row per building per project",
      f"{int(with_bid.duplicated(['project_id', 'building_id']).sum())} duplicate keys", stage="housing cleaning")
check("housing_exact_duplicates", not h_raw.duplicated().any(), "no exact duplicate rows",
      f"{int(h_raw.duplicated().sum())} exact duplicate rows in the raw data", stage="housing cleaning")
check("housing_nonconfidential_have_building_id", h.loc[nonconf, "building_id"].notna().all(),
      "every non-confidential row has a building id",
      f"{int(h.loc[nonconf, 'building_id'].isna().sum())} non-confidential rows lack a building id", stage="housing cleaning", severity="warn")

projects_per_building = with_bid.groupby("building_id")["project_id"].nunique()
h["building_project_count"] = h["building_id"].map(projects_per_building).astype("Int64")
h["building_in_multiple_projects"] = h["building_project_count"].gt(1).fillna(False).astype(bool)
multi = h[h["building_in_multiple_projects"]]
multi_summary = multi.groupby("building_id").agg(projects=("project_id", "nunique"),
                                                 construction_types=("construction_type", lambda s: " + ".join(sorted(s.dropna().unique()))),
                                                 distinct_completion_dates=("building_completion_date", "nunique"))
record_anomaly("HOUSING-MULTI-PROJECT", "housing", len(multi),
               f"{multi['building_id'].nunique()} buildings appear in more than one project",
               "retained (HPD counts units per project) and flagged")
display(Markdown(f"**{multi['building_id'].nunique()} buildings appear in {len(multi)} project rows.** "
                 "Combination of construction types across their projects:"))
display(multi_summary["construction_types"].value_counts().to_frame("buildings"))

[PASS] housing_project_id_present: every row has a project id
[PASS] housing_key_unique: (project_id, building_id) is unique: one row per building per project
[PASS] housing_exact_duplicates: no exact duplicate rows
[PASS] housing_nonconfidential_have_building_id: every non-confidential row has a building id


**123 buildings appear in 247 project rows.** Combination of construction types across their projects:

,buildings
construction_types,
Preservation,108
New Construction,13
New Construction + Preservation,2


**Decision.** Each row is *one building in one project*, and that combination is unique, so there are no duplicate rows to remove. A building that appears in two projects (for example, preserved in one program and refinanced later in another) is not a duplicate: HPD counts the units of each project. We keep these rows and flag them. Section 12 notes that a physical building may therefore be counted more than once.

### 6.11 Confidential buildings and unfinished buildings
Confidential buildings have no published ZIP code, coordinates or building ID, so they cannot be placed in a ZIP area; they are still counted in the borough totals. A blank building completion date means the building is still in progress; these unfinished buildings are used in RQ8.

In [37]:
# 6.11 Confidential buildings have no location, so they can only be counted at borough level.
# A blank completion date means the building is not finished yet ("in_progress").
conf = h[h["is_confidential"]]
check("housing_confidential_have_no_location",
      conf["zip5"].isna().all() and (~conf["has_coords"]).all() and conf["building_id"].isna().all(),
      "confidential rows carry no ZIP, coordinates or building id (as published)",
      "some confidential rows carry location data; review the confidentiality rule", stage="housing cleaning", severity="warn")
h["completion_status"] = np.where(h["building_completion_date"].notna(), "completed", "in_progress")
h["completion_year"] = h["building_completion_date"].dt.year.astype("Int64")
h["start_year"] = h["project_start_date"].dt.year.astype("Int64")
h["status_inconsistent"] = (h["completion_status"].eq("in_progress") & h["project_completion_date"].notna()).astype(bool)
record_anomaly("HOUSING-CONFIDENTIAL", "housing", len(conf),
               f"confidential buildings without published location ({int(conf['units_counted'].sum()):,} counted units in total)",
               "kept in citywide and borough totals; cannot be placed in a ZIP area")
log_step("housing", "6.11", "completion status derived (missing completion date = in progress)",
         int((h["completion_status"] == "in_progress").sum()), len(h), len(h), "a blank completion date marks unfinished work")
pd.crosstab(h["completion_status"], h["is_confidential"], margins=True).rename(columns={False: "published", True: "confidential"})

[PASS] housing_confidential_have_no_location: confidential rows carry no ZIP, coordinates or building id (as published)


is_confidential,published,confidential,All
completion_status,,,
completed,6006,1879,7885
in_progress,1596,70,1666
All,7602,1949,9551


### 6.12 Before and after cleaning
What the cleaning changed, and how many records carry each flag. All housing rows are kept.

In [38]:
# 6.12 Compare the data before and after cleaning, and count the flags.
def housing_metrics(df, units_col):
    return {
        "rows": len(df),
        "counted units (all rows)": int(pd.to_numeric(df[units_col], errors="coerce").sum()),
        "rows with unknown counted units": int(pd.to_numeric(df[units_col], errors="coerce").isna().sum()),
        "blank income-band cells": int(df[H_INCOME].isna().sum().sum()),
        "blank bedroom cells": int(df[H_BEDROOM].isna().sum().sum()),
        "usable 5-digit ZIPs": int(df["zip5"].notna().sum()) if "zip5" in df else int(df["postcode"].notna().sum()),
        "valid coordinates": int(df["has_coords"].sum()) if "has_coords" in df else int(df["latitude"].notna().sum()),
        "placeholder address cells": int(df["house_number"].astype("string").str.strip().eq("----").sum()
                                         + df["street_name"].astype("string").str.strip().eq("----").sum()),
    }
before = housing_metrics(h_before, "all_counted_units")
after = housing_metrics(h, "units_counted")
h_compare = pd.DataFrame({"before cleaning": before, "after cleaning": after})
h_compare["change"] = h_compare["after cleaning"] - h_compare["before cleaning"]
display(h_compare)

flag_cols = ["is_confidential", "date_order_issue", "building_in_multiple_projects", "units_size_outlier",
             "counted_exceeds_total", "units_counted_derived", "bin_placeholder", "house_number_date_like", "coords_invalid", "status_inconsistent"]
flags = h[flag_cols].sum().sort_values(ascending=True).rename("records").rename_axis("flag").reset_index()
fig = px.bar(flags, x="records", y="flag", orientation="h", color_discrete_sequence=[SERIES[0]], text="records",
             labels={"records": "Records flagged (kept)", "flag": ""})
fig.update_traces(textposition="outside", cliponaxis=False)
style(fig, "Housing records flagged during cleaning", f"All {len(h):,} rows are retained; flags let each analysis include or exclude them", height=420).show()

,before cleaning,after cleaning,change
rows,9551,9551,0
counted units (all rows),320946,320946,0
rows with unknown counted units,29,29,0
blank income-band cells,40575,174,-40401
blank bedroom cells,56277,232,-56045
usable 5-digit ZIPs,7593,7593,0
valid coordinates,7593,7593,0
placeholder address cells,3898,0,-3898


## 7. Pre-integration cleaning — Transportation (SIRTA)
**Our rule for the crash data.** We only remove records that cannot be used at all (no crash id, no date and no injury counts) or that are exact copies of another row. Every removed record is kept in a separate table with the reason. Uncertain values are flagged, not guessed: a missing injury count stays missing, and a missing look-back value is *unknown*, not zero.

### 7.1 Empty rows and exact duplicates (justified deletion)
Rows with no crash id, no date and no injury counts ("orphan" rows) cannot be linked to any crash or year. Exact copies of a row would count the same crash twice. Both are removed and saved in a separate table with the reason.

In [39]:
# Work on a copy of the crash data and give every row a stable id (T00000, T00001, ...).
t = t_typed.copy()
t.insert(0, "record_id", [f"T{i:05d}" for i in range(len(t))])
T_ROWS_START = len(t)

# 7.1 Remove orphan rows (no id, date or injury counts) and exact copies of a row.
injury_cols = ["total_non_severe_injuries", "total_severe_injuries", "total_fatalities"]
is_orphan = t["open_data_crash_id"].isna() & t["crash_date"].isna() & t[injury_cols].isna().all(axis=1)
is_exact_dup = t_raw.duplicated(keep="first")            # True for the 2nd, 3rd ... copy of an identical row
exclusion_reason = np.select([as_bool(is_orphan), as_bool(is_exact_dup)],
                             ["orphan: no crash id, date or injury counts", "exact duplicate of an earlier row"], default="")
t["exclusion_reason"] = exclusion_reason
dup_overlap = int((is_orphan & is_exact_dup).sum())
print(f"Orphans: {int(is_orphan.sum())} | exact duplicate copies: {int(is_exact_dup.sum())} "
      f"(of which {dup_overlap} are also orphans)")
display(Markdown("Columns that are filled in orphan rows:"))
display(t.loc[is_orphan, t_raw.columns].notna().sum().loc[lambda s: s > 0].to_frame("non-missing"))
display(Markdown("Exact duplicate groups (all copies):"))
display(t.loc[t_raw.duplicated(keep=False), ["record_id"] + list(t_raw.columns[:8])])

t_excluded = t[t["exclusion_reason"] != ""].copy()
t = t[t["exclusion_reason"] == ""].copy()
log_step("sirta", "7.1", "orphan rows excluded", int(is_orphan.sum()), T_ROWS_START, T_ROWS_START - int(is_orphan.sum()),
         "cannot be tied to any crash or year")
log_step("sirta", "7.1", "exact copies inside the orphan group (already counted once, as orphans)", dup_overlap,
         T_ROWS_START - int(is_orphan.sum()), T_ROWS_START - int(is_orphan.sum()), "identical copies of other orphan rows")
log_step("sirta", "7.1", "exact duplicate copies excluded (outside the orphan group)", int((is_exact_dup & ~is_orphan).sum()),
         T_ROWS_START - int(is_orphan.sum()), len(t), "identical copy of another row")

Orphans: 444 | exact duplicate copies: 3 (of which 3 are also orphans)


Columns that are filled in orphan rows:

,non-missing
community_board,444
council_district,444
bin,433
bbl,433
census_tract_2020_,444
neighborhood_tabulation_area_nta_2020_,444


Exact duplicate groups (all copies):

,record_id,open_data_crash_id,crash_date,crash_time,crash_police_precinct,crash_borough,total_non_severe_injuries,total_severe_injuries,total_fatalities
4356,T04356,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>
4432,T04432,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>
4433,T04433,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>
4506,T04506,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>
4526,T04526,NaN,NaT,NaN,NaN,NaN,<NA>,<NA>,<NA>


### 7.2–7.4 Numbers, dates and crash times
Injury and fatality counts must be whole numbers of zero or more; missing counts stay missing (not zero). We take the year and weekday from the crash date, and we flag crash times of exactly 00:00 as uncertain.

In [40]:
# 7.2 Convert the ids and counts to whole numbers. Impossible values become missing; missing values stay missing.
crash_id_num, bad_ids = to_number(t["open_data_crash_id"])
check("sirta_crash_id_numeric", bad_ids == 0 and (crash_id_num.dropna() % 1 == 0).all(), "collision ids are whole numbers",
      f"{bad_ids} non-numeric collision ids", stage="sirta cleaning", severity="warn")
t["crash_id"] = crash_id_num.round().astype("Int64")
count_issues = {}
for c in T_COUNTS:
    bad = (t[c] < 0) | (t[c].notna() & (t[c] % 1 != 0))
    count_issues[c] = int(bad.sum())
    t.loc[bad, c] = np.nan                       # impossible value -> missing (we never guess a number)
    t[c] = t[c].round().astype("Int64")
check("sirta_counts_valid", sum(count_issues.values()) == 0, "injury, fatality and look-back counts are non-negative whole numbers",
      f"impossible counts set to missing: { {k: v for k, v in count_issues.items() if v} }", stage="sirta cleaning", severity="warn")
t["injury_counts_missing"] = t[["total_severe_injuries", "total_fatalities"]].isna().any(axis=1)
missing_counts = t[T_COUNTS].isna().sum().to_frame("missing (kept as missing, not zero)")
display(missing_counts)

# 7.3 Dates: SIRTA starts in 2022, so dates before 2022 or after the download date are flagged.
SIRTA_START = pd.Timestamp("2022-01-01")
t["date_out_of_range"] = (t["crash_date"].notna() & ~t["crash_date"].between(SIRTA_START, SNAPSHOT_TS)).fillna(False)
check("sirta_dates_present", t["crash_date"].notna().all(), "every retained crash has a date",
      f"{int(t['crash_date'].isna().sum())} retained crashes have no date", stage="sirta cleaning", severity="warn")
check("sirta_dates_in_range", ~t["date_out_of_range"].any(), "all crash dates fall between 2022-01-01 and the snapshot date",
      f"{int(t['date_out_of_range'].sum())} crash dates outside the expected range", stage="sirta cleaning", severity="warn")
t["crash_year"] = t["crash_date"].dt.year.astype("Int64")
t["crash_month"] = t["crash_date"].dt.to_period("M").dt.to_timestamp()
t["weekday"] = t["crash_date"].dt.day_name()

# 7.4 Crash time as HH:MM and hour; 00:00 is flagged because it is probably a default for "unknown".
t["crash_time_hhmm"], t["crash_hour"], t["crash_time_status"] = normalize_time(t["crash_time"])
t["time_exact_midnight"] = t["crash_time_status"].eq("exact_midnight").fillna(False).astype(bool)
log_step("sirta", "7.4", "crash time normalised to HH:MM; exact 00:00 flagged as uncertain", int(t["time_exact_midnight"].sum()),
         len(t), len(t), f"00:00 is {midnight_ratio:.1f}x more frequent than other on-the-hour times")
t["crash_time_status"].value_counts().to_frame("crashes")

[PASS] sirta_crash_id_numeric: collision ids are whole numbers
[PASS] sirta_counts_valid: injury, fatality and look-back counts are non-negative whole numbers


,"missing (kept as missing, not zero)"
total_non_severe_injuries,0
total_severe_injuries,0
total_fatalities,0
similar_fatalities_severe,1393
similar_fatalities_all,0


[PASS] sirta_dates_present: every retained crash has a date
[PASS] sirta_dates_in_range: all crash dates fall between 2022-01-01 and the snapshot date


,crashes
crash_time_status,
valid,13083
exact_midnight,227


### 7.5 Do the collision ids and crash dates agree?
NYPD gives collision ids in order, so crashes with similar ids should have similar dates. If a crash's date is far from the dates of its neighbouring ids, the date may be wrong. This is only a **warning, not proof**. Every flagged record is listed with its neighbours and put into one of four groups:

| Group | Meaning | What we do |
|---|---|---|
| Repeated collision id | The same id appears twice | Resolved in 7.6 |
| Non-standard id series | The id is below the normal NYPD sequence, or a small group of flagged ids and dates rise together. These ids follow a different numbering, so this check **cannot judge their dates** | Kept; flagged `id_nonstandard`; not treated as a date error |
| Possible date-entry error | The date is outside the usual date range (5th–95th percentile) of the neighbouring ids | Kept; flagged `date_suspect`; listed in Section 12 |
| Weak evidence | Unusual, but still inside the neighbours' date range | Kept; flagged `date_suspect` |

In [41]:
# Short explanations for the four groups of flagged records.
AUDIT_CATEGORIES = {
    "repeated collision id": "the same id appears twice; resolved in 7.6",
    "non-standard id series": "id outside the regular NYPD sequence (or part of a small group whose ids and dates rise together); "
                              "the id-based check cannot judge the date, which is plausible",
    "possible date-entry error": "date lies outside the 5-95% date range of neighbouring ids",
    "weak evidence": "unusual, but inside the neighbouring date range",
}

def id_date_audit(df, window=101, threshold_days=180, n_neighbours=3, series_min=3, series_rho=0.9, series_span_days=180):
    # Sort the crashes by collision id and compare each crash date with the typical (median) date
    # of the 100 or so crashes with neighbouring ids. A gap of more than 180 days is flagged.
    d = df.loc[df["crash_id"].notna() & df["crash_date"].notna(), ["record_id", "crash_id", "crash_date"]]
    d = d.sort_values(["crash_id", "crash_date"]).reset_index(drop=True)
    days = (d["crash_date"] - SIRTA_START).dt.days.astype(float)
    roll = days.rolling(window, center=True, min_periods=21)
    d["neighbour_median"] = SIRTA_START + pd.to_timedelta(roll.median(), unit="D")
    d["neighbour_p05"] = SIRTA_START + pd.to_timedelta(roll.quantile(0.05), unit="D")
    d["neighbour_p95"] = SIRTA_START + pd.to_timedelta(roll.quantile(0.95), unit="D")
    d["gap_days"] = (d["crash_date"] - d["neighbour_median"]).dt.days
    flagged = d[d["gap_days"].abs() > threshold_days].copy()
    # Ids lower than the ids used in January 2022 are not part of the normal sequence.
    id_counts = d["crash_id"].value_counts()
    regular_start = d.loc[d["crash_date"] < SIRTA_START + pd.Timedelta(days=31), "crash_id"].quantile(0.01)
    flagged["repeated_id"] = flagged["crash_id"].isin(id_counts[id_counts > 1].index)
    flagged["id_below_regular_sequence"] = flagged["crash_id"] < regular_start
    candidates = flagged[~flagged["repeated_id"]]
    flagged["in_id_series"] = False
    series_rho_obs, series_span_obs = np.nan, np.nan
    base = candidates[candidates["id_below_regular_sequence"]]
    if len(base) >= 2:
        # Ids below the normal sequence form a separate series. Another flagged crash is added to that series
        # only if its date is close to the series dates (within 30 days) and ids and dates still rise together.
        lo_d, hi_d = base["crash_date"].min() - pd.Timedelta(days=30), base["crash_date"].max() + pd.Timedelta(days=30)
        members = list(base.index)
        for idx_c, row in candidates.drop(base.index).iterrows():
            trial = candidates.loc[members + [idx_c]]
            if lo_d <= row["crash_date"] <= hi_d and trial["crash_id"].rank().corr(trial["crash_date"].rank()) >= series_rho:
                members.append(idx_c)
        series = candidates.loc[members]
    elif len(candidates) >= series_min:
        series = candidates
    else:
        series = candidates.iloc[0:0]
    if len(series) >= 2:
        series_rho_obs = series["crash_id"].rank().corr(series["crash_date"].rank())
        series_span_obs = (series["crash_date"].max() - series["crash_date"].min()).days
        if series_rho_obs >= series_rho and series_span_obs <= series_span_days:
            flagged.loc[series.index, "in_id_series"] = True
    # Put every flagged crash into one of the four groups described above.
    outside = (flagged["crash_date"] < flagged["neighbour_p05"]) | (flagged["crash_date"] > flagged["neighbour_p95"])
    flagged["category"] = np.select(
        [as_bool(flagged["repeated_id"]), as_bool(flagged["id_below_regular_sequence"] | flagged["in_id_series"]), as_bool(outside)],
        ["repeated collision id", "non-standard id series", "possible date-entry error"], default="weak evidence")
    flagged["assessment"] = flagged["category"].map(AUDIT_CATEGORIES)
    # List the neighbouring ids and dates, so each flag can be checked by eye.
    flagged["neighbours"] = [
        "; ".join(f"{int(r.crash_id)}={r.crash_date:%Y-%m-%d}" for r in
                  d.iloc[max(0, i - n_neighbours):i + n_neighbours + 1].itertuples() if r.record_id != rec)
        for i, rec in zip(flagged.index, flagged["record_id"])]
    flagged.attrs.update({"regular_start_id": float(regular_start), "series_rho": series_rho_obs, "series_span_days": series_span_obs})
    return flagged.reset_index(drop=True)

id_audit = id_date_audit(t)
# Add the results as flag columns. No date is changed.
t["id_date_flag"] = t["record_id"].isin(id_audit["record_id"])        # flagged in any group
t["date_suspect"] = t["record_id"].isin(id_audit.loc[id_audit["category"].isin(["possible date-entry error", "weak evidence"]), "record_id"])
t["id_nonstandard"] = t["record_id"].isin(id_audit.loc[id_audit["category"].eq("non-standard id series"), "record_id"])
print(f"Regular id sequence starts near {id_audit.attrs['regular_start_id']:,.0f}; flagged records by category:")
display(id_audit["category"].value_counts().to_frame("records"))
if not np.isnan(id_audit.attrs["series_rho"]):
    print(f"Candidate separate id series: id/date rank correlation {id_audit.attrs['series_rho']:.2f}, "
          f"date span {id_audit.attrs['series_span_days']:.0f} days")
display(id_audit[["record_id", "crash_id", "crash_date", "neighbour_median", "gap_days", "category", "assessment", "neighbours"]])
display(Markdown("Flagged crash years (2026 records are outside the 2022-2025 study window):"))
display(t.loc[t["id_date_flag"], "crash_year"].value_counts().sort_index().to_frame("records"))

Regular id sequence starts near 4,491,188; flagged records by category:


,records
category,
non-standard id series,5
repeated collision id,1


Candidate separate id series: id/date rank correlation 1.00, date span 42 days


,record_id,crash_id,crash_date,neighbour_median,gap_days,category,assessment,neighbours
0,T13332,2126022,2026-04-02,2022-01-04,1549,non-standard id series,id outside the regular NYPD sequence (or part of a small group whose ids and dates rise together); the id-based chec...,3026022=2026-04-18; 3526022=2026-04-26; 4426022=2026-05-13
1,T13150,3026022,2026-04-18,2022-01-04,1565,non-standard id series,id outside the regular NYPD sequence (or part of a small group whose ids and dates rise together); the id-based chec...,2126022=2026-04-02; 3526022=2026-04-26; 4426022=2026-05-13; 4491088=2022-01-01
2,T13117,3526022,2026-04-26,2022-01-04,1573,non-standard id series,id outside the regular NYPD sequence (or part of a small group whose ids and dates rise together); the id-based chec...,2126022=2026-04-02; 3026022=2026-04-18; 4426022=2026-05-13; 4491088=2022-01-01; 4491159=2022-01-01
3,T13023,4426022,2026-05-13,2022-01-04,1590,non-standard id series,id outside the regular NYPD sequence (or part of a small group whose ids and dates rise together); the id-based chec...,2126022=2026-04-02; 3026022=2026-04-18; 3526022=2026-04-26; 4491088=2022-01-01; 4491159=2022-01-01; 4491200=2022-01-01
4,T13116,4526022,2026-05-14,2022-05-07,1468,non-standard id series,id outside the regular NYPD sequence (or part of a small group whose ids and dates rise together); the id-based chec...,4525905=2022-04-23; 4525922=2022-05-08; 4525993=2022-05-09; 4526051=2022-05-07; 4526062=2022-05-09; 4526074=2022-05-07
5,T10709,4597666,2026-01-13,2023-01-12,1097,repeated collision id,the same id appears twice; resolved in 7.6,4597591=2023-01-14; 4597618=2023-01-12; 4597666=2023-01-13; 4597816=2023-01-13; 4597854=2023-01-13; 4597905=2023-01-14


Flagged crash years (2026 records are outside the 2022-2025 study window):

,records
crash_year,
2026,6


### 7.6 Repeated collision ids
The same collision id should not appear twice. We compare the two rows side by side and only remove one when the evidence clearly points to it.

In [42]:
# 7.6 Compare the rows that share a collision id. If exactly one of the two has a date that fits
# the id sequence (Section 7.5), keep that one. Otherwise keep both and mark the conflict as unresolved.
dup_ids = t.loc[t["crash_id"].duplicated(keep=False) & t["crash_id"].notna(), "crash_id"].unique()
compare_cols = ["crash_date", "crash_time_hhmm", "crash_police_precinct", "crash_borough", "on_street", "cross_street",
                "latitude", "longitude", "zipcode"] + T_COUNTS
id_resolution = []
for cid in dup_ids:
    grp = t[t["crash_id"] == cid]
    differing = [c for c in compare_cols if grp[c].astype("string").nunique(dropna=False) > 1]
    consistent = ~grp["id_date_flag"]
    if consistent.sum() == 1 and len(grp) == 2:
        keep_id = grp.loc[consistent, "record_id"].iloc[0]
        drop_id = grp.loc[~consistent, "record_id"].iloc[0]
        decision = (f"kept {keep_id} (date consistent with the id sequence); excluded {drop_id} "
                    "(its date contradicts the id sequence, see 7.5)")
        t.loc[t["record_id"] == drop_id, "exclusion_reason"] = "repeated collision id; date contradicts id sequence"
        t.loc[t["record_id"] == keep_id, "id_conflict"] = "resolved"
    else:
        decision = "unresolved: both records retained and flagged"
        t.loc[t["crash_id"] == cid, "id_conflict"] = "unresolved"
    # If the copies report different fatality numbers, we keep the recorded value and flag the conflict.
    fatality_conflict = "total_fatalities" in differing
    if fatality_conflict:
        t.loc[t["crash_id"] == cid, "fatality_conflict"] = True
    id_resolution.append({"crash_id": int(cid), "records": ", ".join(grp["record_id"]),
                          "dates": ", ".join(grp["crash_date"].dt.strftime("%Y-%m-%d")),
                          "fatalities": ", ".join(grp["total_fatalities"].astype("string").fillna("NA")),
                          "differing_fields": ", ".join(differing), "decision": decision,
                          "fatality_value": "conflict: recorded value kept, not changed" if fatality_conflict else "consistent"})
    display(Markdown(f"**Collision {int(cid)}** — side-by-side comparison"))
    display(grp.set_index("record_id")[compare_cols].T)
t["id_conflict"] = t["id_conflict"].astype("string") if "id_conflict" in t else pd.Series(pd.NA, index=t.index, dtype="string")
t["fatality_conflict"] = (t["fatality_conflict"].astype("boolean").fillna(False).astype(bool) if "fatality_conflict" in t
                          else pd.Series(False, index=t.index))
id_resolution = pd.DataFrame(id_resolution)
display(id_resolution)

newly_excluded = t[t["exclusion_reason"] != ""]
t_excluded = pd.concat([t_excluded, newly_excluded])
rows_before_10_6 = len(t)
t = t[t["exclusion_reason"] == ""].copy()
log_step("sirta", "7.6", "repeated collision id resolved using the id-sequence evidence", len(newly_excluded),
         rows_before_10_6, len(t), "same crash recorded twice; one date contradicts the id sequence")
for row in id_resolution.itertuples():
    record_anomaly(f"SIRTA-ID-{row.crash_id}", "sirta", len(row.records.split(", ")),
                   f"collision id {row.crash_id} recorded on {row.dates} with fatalities {row.fatalities}",
                   f"{row.decision}; fatality value: {row.fatality_value}")
remaining_suspect = int(t["date_suspect"].sum())
n_out_of_range = int(t["date_out_of_range"].sum())
if n_out_of_range:
    record_anomaly("SIRTA-DATE-OUT-OF-RANGE", "sirta", n_out_of_range, "crash dates before 2022-01-01 or after the snapshot date",
                   "retained and flagged; they fall outside the 2022-2025 window and do not enter the panel")
n_nonstandard = int(t["id_nonstandard"].sum())
record_anomaly("SIRTA-NONSTANDARD-IDS", "sirta", n_nonstandard,
               "collision ids outside the regular NYPD sequence (crash years: "
               + (", ".join(f"{int(y)}: {n}" for y, n in t.loc[t["id_nonstandard"], "crash_year"].value_counts().sort_index().items()) or "none") + ")",
               "retained; dates are plausible and the id-based check cannot assess them; not treated as date errors")
record_anomaly("SIRTA-SUSPECT-DATES", "sirta", remaining_suspect,
               "crash dates that contradict neighbouring collision ids (possible date-entry errors or weak evidence)",
               "retained with original dates and flagged; listed as a limitation (Section 12)")
print(f"After resolution: {n_nonstandard} records with non-standard ids, {remaining_suspect} possible date problems")

**Collision 4597666** — side-by-side comparison

record_id,T02147,T10709
crash_date,2023-01-13 00:00:00,2026-01-13 00:00:00
crash_time_hhmm,21:01,21:01
crash_police_precinct,44,44
crash_borough,Bronx,Bronx
on_street,WEST 170 STREET,WEST 170 STREET
cross_street,Plaza Dr,Plaza Dr
latitude,<NA>,<NA>
longitude,<NA>,<NA>
zipcode,NaN,NaN
total_non_severe_injuries,0,0


,crash_id,records,dates,fatalities,differing_fields,decision,fatality_value
0,4597666,"T02147, T10709","2023-01-13, 2026-01-13","0, 1","crash_date, total_fatalities, similar_fatalities_severe, similar_fatalities_all","kept T02147 (date consistent with the id sequence); excluded T10709 (its date contradicts the id sequence, see 7.5)","conflict: recorded value kept, not changed"


After resolution: 5 records with non-standard ids, 0 possible date problems


**Why we can explain this decision.** The two rows have the same time, street and precinct, so they describe one crash. The collision id sits among early-2023 ids, which supports the 2023 date, so the 2026 copy is removed (it is outside 2022–2025 anyway). The **fatality difference cannot be resolved** from the data — the 2026 copy may include a later death — so the value of the kept record is not changed. The record is flagged with `fatality_conflict = True` and the limitation is reported. If the evidence did not point to one copy, both would be kept and flagged as unresolved.

### 7.7 ZIP codes, coordinates and borough
The placeholder ZIP 10000 and malformed ZIP codes become missing, coordinates outside NYC become missing, and borough names are written the same way as in the housing data.

In [43]:
# 7.7 Same ZIP and coordinate rules as for housing, and consistent borough names.
t["zip5"], t["zip_status"] = parse_zip_strict(t["zipcode"])
coords_ok = t["latitude"].between(*NYC_LAT_RANGE) & t["longitude"].between(*NYC_LON_RANGE)
t["coords_invalid"] = t["latitude"].notna() & t["longitude"].notna() & ~coords_ok
t.loc[~coords_ok, ["latitude", "longitude"]] = np.nan
t["has_coords"] = coords_ok.fillna(False).astype(bool)
t["crash_borough"] = clean_text(t["crash_borough"]).str.title()
check("sirta_borough_values", t["crash_borough"].dropna().isin(BOROUGHS).all(), "borough values are the five boroughs",
      f"unexpected borough values: {set(t['crash_borough'].dropna()) - set(BOROUGHS)}", stage="sirta cleaning")
for col in ["on_street", "cross_street", "highway_reference_marker"]:
    t[col] = clean_text(t[col])
log_step("sirta", "7.7", "placeholder/invalid ZIPs -> missing", int(t["zip_status"].isin(["placeholder", "malformed", "outside_nyc_range"]).sum()),
         len(t), len(t), "10000 is not a NYC ZIP; only complete 5-digit ZIPs are usable")
log_step("sirta", "7.7", "coordinates outside NYC -> missing", int(t["coords_invalid"].sum()), len(t), len(t),
         "a point outside the city cannot be placed in a ZIP area")
t["zip_status"].value_counts().to_frame("crashes")

[PASS] sirta_borough_values: borough values are the five boroughs


,crashes
zip_status,
valid,8028
missing,5237
placeholder,44


### 7.8 Highway crashes
Expressways, highway-style parkways and major bridges pass through ZIP areas they do not really serve, so we keep their crashes apart from **local-street** crashes. A crash counts as a highway crash if it has a highway reference marker or its street name matches a known expressway, highway-style parkway or bridge. Ordinary streets that are called "parkway" (Ocean, Eastern, Pelham, Bay Parkway) or "turnpike" (Union, Jericho) are **not** matched. Shore Parkway is unclear (part of it is the Belt Parkway, part is a service road), so it stays local unless it has a highway marker.

In [44]:
# Street names of expressways, highway-style parkways and major bridges.
LIMITED_ACCESS = [r"EXPRESSWAY", r"\bEXPWY\b", r"\bEXPY\b", r"\bEXWY\b", r"\bBQE\b", r"\bLIE\b", r"\bFDR\b",
                  r"F\.?D\.? ?R\.? DR", r"BELT (?:PARKWAY|PKWY)", r"GRAND CENTRAL (?:PARKWAY|PKWY)", r"CROSS ISLAND (?:PARKWAY|PKWY)",
                  r"JACKIE ROBINSON (?:PARKWAY|PKWY)", r"HUTCHINSON RIVER (?:PARKWAY|PKWY)", r"BRONX RIVER (?:PARKWAY|PKWY)",
                  r"HENRY HUDSON (?:PARKWAY|PKWY)", r"MOSHOLU (?:PARKWAY|PKWY)", r"HARLEM RIVER (?:DRIVE|DR)\b", r"MAJOR DEEGAN",
                  r"VAN WYCK", r"VANWYCK", r"CROSS BRONX", r"CLEARVIEW", r"NASSAU EXP", r"THRUWAY",
                  r"(?:VERRAZZANO|VERRAZANO|THROGS NECK|WHITESTONE|GEORGE WASHINGTON|KOSCIUSZKO|GOETHALS|OUTERBRIDGE|BAYONNE|TRIBOROUGH|RFK) BRIDGE"]
HIGHWAY_RE = "|".join(LIMITED_ACCESS)
# A crash is a highway crash if it has a highway marker or its street name matches the list above.
street = t["on_street"].str.upper()
by_marker = t["highway_reference_marker"].notna()
by_name = street.str.contains(HIGHWAY_RE, regex=True).fillna(False)
t["highway_basis"] = np.select([as_bool(by_marker & by_name), as_bool(by_marker), as_bool(by_name)],
                               ["marker and street name", "marker only", "street name only"], default="local street")
t["is_highway"] = t["highway_basis"].ne("local street")
t["crash_scope"] = np.where(t["is_highway"], "highway", "local")
display(t["highway_basis"].value_counts().to_frame("crashes"))
display(Markdown("**Streets matched by name only (review list):**"))
display(t.loc[t["highway_basis"].eq("street name only"), "on_street"].value_counts().head(12).to_frame("crashes"))
display(Markdown("**Streets containing 'PARKWAY/PKWY' that remain local (surface roads):**"))
display(t.loc[~t["is_highway"] & street.str.contains(r"PARKWAY|PKWY", regex=True).fillna(False), "on_street"]
        .value_counts().head(10).to_frame("crashes"))
log_step("sirta", "7.8", "highway crashes flagged", int(t["is_highway"].sum()), len(t), len(t),
         "limited-access roads do not serve the ZIP areas they cross")

,crashes
highway_basis,
local street,11899
marker and street name,1027
street name only,358
marker only,25


**Streets matched by name only (review list):**

,crashes
on_street,
TRIBOROUGH BRIDGE,30
CROSS ISLAND PARKWAY,21
LONG ISLAND EXPRESSWAY,19
FDR DRIVE,18
VERRAZANO BRIDGE UPPER,17
GRAND CENTRAL PKWY,15
GRAND CENTRAL PARKWAY,14
WHITESTONE EXPRESSWAY,13
MAJOR DEEGAN EXPRESSWAY,12


**Streets containing 'PARKWAY/PKWY' that remain local (surface roads):**

,crashes
on_street,
EASTERN PARKWAY,40
OCEAN PARKWAY,25
EASTERN PKWY,20
OCEAN PKWY,16
SHORE PARKWAY,15
ROCKAWAY PARKWAY,14
ROCKAWAY PKWY,14
FORT HAMILTON PARKWAY,9
BAY RIDGE PARKWAY,8


### 7.9 The 3-year look-back fields — DOT's definition
The DOT column descriptions on NYC Open Data say:
* **`similar_fatalities_severe`** — the number of fatalities and severe injuries from crashes *of the same type at the same location* in the past three years;
* **`similar_fatalities_all`** — the same, but counting all injuries.

So these fields count **people hurt in earlier similar crashes**, not crashes. The column descriptions do not say exactly how DOT decides "same type" and "same location"; the dataset page links a data dictionary (`SIRTA_Data_Dictionary.xlsx`) with more detail. We label a crash **repeat** when `similar_fatalities_severe` is 1 or more, **first** when it is 0, and **unknown** when it is blank. A blank is never treated as zero.

In [45]:
# Label each crash: repeat (an earlier similar serious injury was recorded), first (none) or unknown (blank).
t["repeat_location_status"] = np.select(
    [as_bool(t["similar_fatalities_severe"].isna()), as_bool(t["similar_fatalities_severe"].ge(1))],
    ["unknown", "repeat"], default="first")
both_known = t[["total_severe_injuries", "total_fatalities"]].notna().all(axis=1)
no_serious = both_known & t["total_severe_injuries"].eq(0) & t["total_fatalities"].eq(0)
check("sirta_serious_definition", no_serious.sum() == 0,
      "every crash with known counts has at least one severe injury or fatality, as the program defines",
      f"{int(no_serious.sum())} crashes have no severe injury or fatality", stage="sirta cleaning", severity="warn")
display(t["repeat_location_status"].value_counts().to_frame("crashes"))

[PASS] sirta_serious_definition: every crash with known counts has at least one severe injury or fatality, as the program defines


,crashes
repeat_location_status,
first,11099
unknown,1393
repeat,817


### 7.10 Before and after cleaning
Raw rows = removed rows + kept rows, and how many kept records carry each flag.

In [46]:
# 7.10 Check that every raw row is either kept or listed as removed, and count the flags.
t_ledger_clean = pd.DataFrame([
    ("Raw rows", T_ROWS_START),
    ("Excluded: orphan rows", int((t_excluded["exclusion_reason"] == "orphan: no crash id, date or injury counts").sum())),
    ("Excluded: exact duplicate copies", int((t_excluded["exclusion_reason"] == "exact duplicate of an earlier row").sum())),
    ("Excluded: repeated id with contradicted date", int((t_excluded["exclusion_reason"] == "repeated collision id; date contradicts id sequence").sum())),
    ("Retained after cleaning", len(t)),
], columns=["stage", "records"])
display(t_ledger_clean)
check("sirta_rows_reconcile", t_ledger_clean["records"].iloc[1:4].sum() + len(t) == T_ROWS_START,
      "raw rows = excluded + retained", "excluded + retained does not equal the raw row count", stage="sirta cleaning")
check("sirta_excluded_table_complete", len(t_excluded) == T_ROWS_START - len(t),
      "every excluded record is kept in the exclusion table", "excluded records are missing from the exclusion table", stage="sirta cleaning")
print(f"Note: {dup_overlap} of the {int(is_orphan.sum())} orphan rows are exact copies of other orphan rows; "
      "they are counted once, as orphans, so the exclusions add up without double counting.")

t_flags = pd.Series({
    "possible date problem (id check)": int(t["date_suspect"].sum()), "non-standard collision id": int(t["id_nonstandard"].sum()),
    "exact 00:00 time": int(t["time_exact_midnight"].sum()),
    "highway crash": int(t["is_highway"].sum()), "repeat location unknown": int(t["repeat_location_status"].eq("unknown").sum()),
    "injury counts missing": int(t["injury_counts_missing"].sum()), "fatality conflict": int(t["fatality_conflict"].sum()),
    "placeholder ZIP": int(t["zip_status"].eq("placeholder").sum()), "no ZIP and no coordinates": int((t["zip5"].isna() & ~t["has_coords"]).sum()),
    "borough missing": int(t["crash_borough"].isna().sum()),
}).sort_values().rename("records").rename_axis("flag").reset_index()
fig = px.bar(t_flags, x="records", y="flag", orientation="h", text="records", color_discrete_sequence=[SERIES[0]],
             labels={"records": "Records flagged (kept)", "flag": ""})
fig.update_traces(textposition="outside", cliponaxis=False)
style(fig, "SIRTA records flagged during cleaning",
      f"{len(t):,} records retained; {len(t_excluded):,} excluded with a recorded reason", height=420).show()

,stage,records
0,Raw rows,13754
1,Excluded: orphan rows,444
2,Excluded: exact duplicate copies,0
3,Excluded: repeated id with contradicted date,1
4,Retained after cleaning,13309


[PASS] sirta_rows_reconcile: raw rows = excluded + retained
[PASS] sirta_excluded_table_complete: every excluded record is kept in the exclusion table
Note: 3 of the 444 orphan rows are exact copies of other orphan rows; they are counted once, as orphans, so the exclusions add up without double counting.


### 7.11 Checks on the cleaned data
These checks confirm that the cleaned tables follow the rules the integration depends on (valid ZIPs, no negative counts, consistent status columns, unique ids).

In [47]:
# Checks on the cleaned housing and crash tables before we combine them.
stage = "cleaned-data validation"
check("housing_rows_preserved", len(h) == H_ROWS_START, f"all {H_ROWS_START:,} housing rows retained", "housing rows were lost", stage=stage)
check("housing_zip_format", h["zip5"].dropna().str.fullmatch(r"\d{5}").all(), "every housing ZIP is exactly 5 digits", "malformed ZIP survived", stage=stage)
check("sirta_zip_format", t["zip5"].dropna().str.fullmatch(r"\d{5}").all(), "every SIRTA ZIP is exactly 5 digits", "malformed ZIP survived", stage=stage)
check("housing_units_nonnegative", (h[H_UNITS + ["units_counted"]].fillna(0) >= 0).all().all(), "no negative unit counts", "negative units remain", stage=stage)
check("sirta_counts_nonnegative", (t[T_COUNTS].fillna(0) >= 0).all().all(), "no negative injury or fatality counts", "negative counts remain", stage=stage)
check("housing_status_consistent", (h["completion_status"].eq("completed") == h["building_completion_date"].notna()).all(),
      "completion status matches the presence of a completion date", "status and completion date disagree", stage=stage)
check("housing_confidential_flag", h["is_confidential"].sum() == int(h_conf_raw.sum()), "confidential flag matches the raw data",
      "confidential count changed during cleaning", stage=stage)
check("sirta_ids_unique_or_flagged",
      (~t["crash_id"].duplicated(keep=False) | t["id_conflict"].eq("unresolved").fillna(False)).all(),
      "collision ids are unique except explicitly unresolved conflicts", "duplicate collision ids remain unflagged", stage=stage)
check("sirta_dates_parsed", t["crash_date"].notna().all(), "every retained crash has a parsed date", "retained crash without a date", stage=stage)
check("sirta_time_status_known", t["crash_time_status"].notna().all(), "every crash has a time status", "time status missing", stage=stage)

[PASS] housing_rows_preserved: all 9,551 housing rows retained
[PASS] housing_zip_format: every housing ZIP is exactly 5 digits
[PASS] sirta_zip_format: every SIRTA ZIP is exactly 5 digits
[PASS] housing_units_nonnegative: no negative unit counts
[PASS] sirta_counts_nonnegative: no negative injury or fatality counts
[PASS] housing_status_consistent: completion status matches the presence of a completion date
[PASS] housing_confidential_flag: confidential flag matches the raw data
[PASS] sirta_ids_unique_or_flagged: collision ids are unique except explicitly unresolved conflicts
[PASS] sirta_dates_parsed: every retained crash has a parsed date
[PASS] sirta_time_status_known: every crash has a time status


True

## 8. Data integration
**Why we combine counts instead of single records.** The two datasets have no shared record id. A building and a crash in the same ZIP code are separate events, so joining the raw rows on ZIP would only multiply rows (every building × every crash in that ZIP). Instead, we count both datasets per area and year and then join the counts one-to-one.

**Chosen level: ZIP-code area (MODZCTA) × calendar year, 2022–2025.**

| Other option we considered | Why we did not use it |
|---|---|
| SIRTA's own ZIP column | It is blank for most 2022 crashes and much more complete later, so every area would show a false jump from 2022 to 2023 |
| Community district / council district | Only about a quarter of SIRTA records have these columns |
| Census tract | The two datasets use different census versions, and there would be very few crashes per tract and year |
| Borough | Complete, but only five areas — too coarse for eight questions (we still report borough totals) |
| A distance circle around each building | Needs extra choices (radius, overlapping circles) without a clear benefit for this project |

**Why ZIP area × year makes sense.** ZIP-code areas are small enough to separate neighbourhoods but large enough to contain several crashes per year, and both datasets can be placed in them (by coordinates or ZIP code). Housing completions and crashes are both recorded by date, so a calendar year fits both. The join keys are `modzcta` (the ZIP-code area) and `year`.

**How a record is placed in an area (the same rule for every year).**
1. If the record has valid NYC coordinates, we find the MODZCTA area that contains the point (a *point-in-polygon* spatial join).
2. A point exactly on a border touches two areas. Then we take the area of the record's own ZIP code if it is one of the two, otherwise the lower area code, so the result is always the same. The number of such cases is printed.
3. If there are no coordinates but there is a valid ZIP code, we look the ZIP up in the MODZCTA ZIP lists.
4. Otherwise the record is **unassigned**: it is still counted in the borough and citywide totals, but not in the ZIP-area analysis.

### 8.1 The MODZCTA areas
We remove the placeholder area 99999, repair any broken area shapes, and give each area its borough (from the first three digits of its ZIP code) and its 2018 population.

In [48]:
# 8.1 Prepare the MODZCTA areas: remove the placeholder area, fix broken shapes, add borough and population.
def borough_from_zip_prefix(code):
    # The first three digits of a NYC ZIP code give the borough: 100-102 Manhattan, 103 Staten Island,
    # 104 Bronx, 112 Brooklyn, and 110/111/113/114/116 Queens.
    p3 = int(str(code)[:3])
    return ("Manhattan" if 100 <= p3 <= 102 else "Staten Island" if p3 == 103 else "Bronx" if p3 == 104
            else "Brooklyn" if p3 == 112 else "Queens" if p3 in (110, 111, 113, 114, 116) else None)

mz = mz_raw.copy()
mz["modzcta"] = normalize_id(mz["modzcta"])
mz["pop_est"] = pd.to_numeric(mz["pop_est"], errors="coerce")
placeholder_areas = mz["modzcta"].eq("99999") | ~mz["modzcta"].str.fullmatch(r"\d{5}").fillna(False)
print(f"Reference records: {len(mz)} | placeholder/invalid area codes excluded: {list(mz.loc[placeholder_areas, 'modzcta'])}")
mz = mz[~placeholder_areas].copy()
invalid_geom = ~mz.geometry.is_valid
if invalid_geom.any():
    mz.loc[invalid_geom, "geometry"] = mz.loc[invalid_geom, "geometry"].buffer(0)   # buffer(0) is a common fix for a broken polygon
mz = mz.to_crs("EPSG:4326") if mz.crs is not None else mz.set_crs("EPSG:4326")
mz["area_borough"] = mz["modzcta"].map(borough_from_zip_prefix)
stage = "geographic reference"
check("reference_unique_areas", mz["modzcta"].is_unique, f"{len(mz)} unique analytical areas", "duplicate MODZCTA codes", stage=stage)
check("reference_geometry_valid", mz.geometry.is_valid.all(), f"all geometries valid ({int(invalid_geom.sum())} repaired)",
      "invalid geometries remain after repair", stage=stage)
check("reference_population_positive", mz["pop_est"].gt(0).all(), "every area has a positive 2018 population estimate",
      f"{int((~mz['pop_est'].gt(0)).sum())} areas lack a usable population; their rates will be undefined",
      stage=stage, severity="warn")
check("reference_borough_assigned", mz["area_borough"].notna().all(), "every area has a borough from its ZIP prefix",
      "some areas have an unrecognised ZIP prefix", stage=stage)

Reference records: 178 | placeholder/invalid area codes excluded: ['99999']
[PASS] reference_unique_areas: 177 unique analytical areas
[PASS] reference_geometry_valid: all geometries valid (0 repaired)
[PASS] reference_population_positive: every area has a positive 2018 population estimate
[PASS] reference_borough_assigned: every area has a borough from its ZIP prefix


True

### 8.2 Postal ZIP codes and Census ZIP areas
A **ZIP code** is a postal delivery route. A **ZCTA** is the Census Bureau's area version of it, and MODZCTA joins a few small ZCTAs together. Some postal ZIP codes have no ZCTA of their own — for example ZIPs for a single building (10158) or ZIPs created after 2010 (11249 in Williamsburg). So the reference file has **two** ZIP columns, and we use both:
* `zcta` — the Census ZCTAs that make up each area;
* `label` — the postal ZIP codes the area stands for (this is where 11249 appears, inside area 11211).

If a ZIP code would point to more than one area, it is **ambiguous** and we do not use it.

In [49]:
# Build a lookup from every ZIP code listed in MODZCTA (the zcta and label columns) to its area.
# A ZIP code that points to more than one area is ambiguous, so it is left out of the lookup.
def split_codes(value):
    return [c.strip() for c in str(value).split(",") if re.fullmatch(r"\d{5}", c.strip())]

lookup_rows = []
for row in mz.itertuples():
    lookup_rows.append((row.modzcta, row.modzcta, "modzcta"))
    for field in ["zcta", "label"]:
        if field in mz.columns and pd.notna(getattr(row, field)):
            lookup_rows += [(code, row.modzcta, field) for code in split_codes(getattr(row, field))]
zip_lookup_long = pd.DataFrame(lookup_rows, columns=["zip", "modzcta", "field"]).drop_duplicates()
targets = zip_lookup_long.groupby("zip")["modzcta"].nunique()
AMBIGUOUS_ZIPS = sorted(targets[targets > 1].index)
REF_ZIP_LOOKUP = (zip_lookup_long[~zip_lookup_long["zip"].isin(AMBIGUOUS_ZIPS)]
                  .drop_duplicates("zip").set_index("zip")["modzcta"])

field_sets = {f: set(zip_lookup_long.loc[zip_lookup_long["field"] == f, "zip"]) for f in ["zcta", "label"]}
field_comparison = pd.DataFrame({
    "codes": [len(field_sets["zcta"] & field_sets["label"]), len(field_sets["label"] - field_sets["zcta"]),
              len(field_sets["zcta"] - field_sets["label"])],
    "examples": [", ".join(sorted(field_sets["zcta"] & field_sets["label"])[:5]),
                 ", ".join(sorted(field_sets["label"] - field_sets["zcta"])[:8]),
                 ", ".join(sorted(field_sets["zcta"] - field_sets["label"])[:8])]},
    index=["in both zcta and label", "only in label (postal ZIPs)", "only in zcta (Census-only codes)"])
display(field_comparison)
check("zip_lookup_unambiguous", len(AMBIGUOUS_ZIPS) == 0, f"{len(REF_ZIP_LOOKUP)} ZIP codes map to exactly one area",
      f"{len(AMBIGUOUS_ZIPS)} ZIPs map to several areas and are excluded from ZIP-based assignment: {AMBIGUOUS_ZIPS[:10]}",
      stage="geographic reference", severity="warn")

,codes,examples
in both zcta and label,180,"10001, 10002, 10003, 10004, 10005"
only in label (postal ZIPs),3,"10118, 11243, 11249"
only in zcta (Census-only codes),34,"10103, 10110, 10111, 10112, 10115, 10119, 10152, 10153"


[PASS] zip_lookup_unambiguous: 217 ZIP codes map to exactly one area


True

### 8.3 Placing records by their coordinates

In [50]:
# 8.3 Place each record that has valid coordinates in the area that contains the point (spatial join).
def coordinate_area(df):
    # Returns the area found for each record and how many areas the point touches.
    # A point exactly on a border touches two areas: then we take the area of the record's own ZIP code
    # if it is one of them, otherwise the first area code, so the result is always the same.
    out = pd.DataFrame({"mz_from_coords": pd.Series(pd.NA, index=df.index, dtype="string"),
                        "coord_candidates": pd.Series(0, index=df.index, dtype="Int64")})
    has = df["has_coords"]
    if not has.any():
        return out
    pts = gpd.GeoDataFrame({"_idx": df.index[has]},
                           geometry=gpd.points_from_xy(df.loc[has, "longitude"], df.loc[has, "latitude"]), crs="EPSG:4326")
    hits = gpd.sjoin(pts, mz[["modzcta", "geometry"]], how="inner", predicate="intersects")
    candidates = hits.groupby("_idx")["modzcta"].agg(lambda s: sorted(set(s)))
    reported = df["zip5"].map(REF_ZIP_LOOKUP)
    def choose(idx, options):
        rep = reported.get(idx)
        return rep if (len(options) > 1 and isinstance(rep, str) and rep in options) else options[0]
    out.loc[candidates.index, "mz_from_coords"] = [choose(i, o) for i, o in candidates.items()]
    out.loc[candidates.index, "coord_candidates"] = candidates.str.len().astype("Int64")
    return out

h = h.drop(columns=["mz_from_coords", "coord_candidates"], errors="ignore").join(coordinate_area(h))
t = t.drop(columns=["mz_from_coords", "coord_candidates"], errors="ignore").join(coordinate_area(t))
print("Boundary ties (points touching more than one area):",
      {"housing": int(h["coord_candidates"].gt(1).sum()), "sirta": int(t["coord_candidates"].gt(1).sum())})

Boundary ties (points touching more than one area): {'housing': 0, 'sirta': 0}


### 8.4 ZIP codes that MODZCTA does not list
If a record has no coordinates and its ZIP code is not in either MODZCTA list, we cannot place it reliably, so it stays **unassigned**. The table shows these ZIP codes and how many records they affect. Placeholders such as 10000 were already removed in Sections 6–7.

In [51]:
# Count the records whose ZIP code is not in the MODZCTA lists. Without coordinates, these records cannot be placed.
observed = pd.concat([h[["zip5", "has_coords"]].assign(dataset="housing"), t[["zip5", "has_coords"]].assign(dataset="sirta")], ignore_index=True)
unlisted = observed[observed["zip5"].notna() & ~observed["zip5"].isin(REF_ZIP_LOOKUP.index)]
unlisted_report = (unlisted.groupby(["zip5", "dataset"]).agg(records=("has_coords", "size"), with_coordinates=("has_coords", "sum"))
                   .reset_index().rename(columns={"zip5": "zip"}))
unlisted_report["records_unassigned_because_of_zip"] = unlisted_report["records"] - unlisted_report["with_coordinates"]
display(unlisted_report if len(unlisted_report) else Markdown("Every observed ZIP is listed in the reference."))

,zip,dataset,records,with_coordinates,records_unassigned_because_of_zip
0,10158,housing,1,1,0
1,10281,sirta,2,2,0
2,11251,sirta,2,2,0


### 8.5 One area per record
We use the coordinates first and the ZIP code only when there are no valid coordinates. The method used is saved with every record.

**Crashes with no reported borough.** A few crashes have an empty borough column. For the borough totals in Section 9.2, such a crash gets the borough of its ZIP area if it was placed in one, and this is marked as *derived from ZIP area*. A crash that has neither a borough nor an area is counted as **Unknown**. We never guess a location, and the original borough column is not changed.

In [52]:
def assign_area(df):
    # One area per record: the coordinates first, the ZIP code only if there are no coordinates.
    # We also save how the record was placed and, if it was not placed, why.
    zip_ref = df["zip5"].map(REF_ZIP_LOOKUP).astype("string")
    df["mz_from_zip"] = zip_ref
    df["zip_lookup_method"] = np.select([as_bool(df["zip5"].isna()), as_bool(zip_ref.notna())],
                                        ["no valid ZIP", "reference"], default="unmapped")
    df["modzcta"] = df["mz_from_coords"].fillna(df["mz_from_zip"]).astype("string")
    df["assignment_method"] = np.select(
        [as_bool(df["mz_from_coords"].notna() & df["coord_candidates"].gt(1)), as_bool(df["mz_from_coords"].notna()),
         as_bool(df["mz_from_zip"].notna())],
        ["coordinates (boundary tie)", "coordinates", "reported ZIP"], default="unassigned")
    df["coords_outside_reference"] = df["has_coords"] & df["mz_from_coords"].isna()
    df["unassigned_reason"] = np.select(
        [as_bool(df["assignment_method"].ne("unassigned")), as_bool(df["coords_outside_reference"] & df["zip5"].isna()),
         as_bool(~df["has_coords"] & df["zip5"].isna()), as_bool(df["zip_lookup_method"].eq("unmapped"))],
        ["", "coordinates outside all areas and no ZIP", "no coordinates and no valid ZIP", "ZIP not in the reference (or ambiguous)"],
        default="other")
    df["zip_coord_agree"] = (df["mz_from_coords"] == zip_ref).where(df["mz_from_coords"].notna() & zip_ref.notna())
    df["area_borough"] = df["modzcta"].map(mz.set_index("modzcta")["area_borough"]).astype("string")
    return df

h, t = assign_area(h), assign_area(t)
display(pd.DataFrame({"housing": h["assignment_method"].value_counts(), "sirta": t["assignment_method"].value_counts()}).fillna(0).astype(int))

# Borough used in the borough totals: the reported borough if there is one; otherwise the borough of the
# crash's ZIP area (marked "derived from ZIP area"); otherwise "Unknown". The reported borough is not changed.
t["summary_borough"] = t["crash_borough"].fillna(t["area_borough"]).fillna("Unknown").astype("string")
t["borough_source"] = np.select([as_bool(t["crash_borough"].notna()), as_bool(t["area_borough"].notna())],
                                ["reported", "derived from ZIP area"], default="unknown")

,housing,sirta
assignment_method,,
coordinates,7592,10743
reported ZIP,1,1301
unassigned,1958,1265


### 8.6 How reliable is the placement?
How many records could be placed, and how — overall, by year and by borough? We also look at records whose ZIP code and coordinates point to different areas. If the two areas are **neighbours**, the point is probably close to a border (or the postal ZIP differs a little from the Census area), which supports using the coordinates first.

In [53]:
# Two areas are "neighbours" if their shapes touch.
neighbour_pairs = gpd.sjoin(mz[["modzcta", "geometry"]], mz[["modzcta", "geometry"]], predicate="intersects")
NEIGHBOURS = set(zip(neighbour_pairs["modzcta_left"], neighbour_pairs["modzcta_right"]))

def reliability_table(df, by):
    # Share of records with coordinates / a valid ZIP, how they were placed, and how often ZIP and coordinates agree.
    d = df.assign(valid_coordinates=df["has_coords"], valid_zip=df["zip5"].notna(),
                  via_coordinates=df["assignment_method"].str.startswith("coordinates"),
                  via_zip=df["assignment_method"].eq("reported ZIP"), unassigned=df["assignment_method"].eq("unassigned"),
                  agree=df["zip_coord_agree"].astype("float"))
    g = d.groupby(by, dropna=False)
    out = g[["valid_coordinates", "valid_zip", "via_coordinates", "via_zip", "unassigned"]].mean().mul(100).round(1)
    out.columns = [c + "_pct" for c in out.columns]
    out.insert(0, "records", g.size())
    out["zip_coord_comparisons"] = g["agree"].count()
    out["zip_coord_agreement_pct"] = g["agree"].mean().mul(100).round(1)
    return out

def disagreement_profile(df):
    # When ZIP and coordinates point to different areas, how often are those areas neighbours?
    d = df[df["zip_coord_agree"].eq(False).fillna(False)]
    reported = d["zip5"].map(REF_ZIP_LOOKUP)
    neighbour = [(a, b) in NEIGHBOURS for a, b in zip(d["mz_from_coords"], reported)]
    return {"disagreements": len(d), "neighbouring_area_pct": round(100 * np.mean(neighbour), 1) if len(d) else np.nan}

display(Markdown("**Housing, by completion status**"))
display(reliability_table(h.assign(group=np.where(h["is_confidential"], "confidential", h["completion_status"])), "group"))
display(Markdown("**SIRTA, by crash year**"))
sirta_reliability_year = reliability_table(t, "crash_year")
display(sirta_reliability_year)
display(Markdown("**SIRTA, by reported borough**"))
display(reliability_table(t, "crash_borough"))
dis = pd.DataFrame({"housing": disagreement_profile(h), "sirta": disagreement_profile(t)}).T
dis["overall_agreement_pct"] = [round(100 * h["zip_coord_agree"].astype("float").mean(), 1),
                                round(100 * t["zip_coord_agree"].astype("float").mean(), 1)]
display(Markdown("**ZIP vs coordinate disagreements**")); display(dis)

boro_mismatch = t["crash_borough"].notna() & t["area_borough"].notna() & t["crash_borough"].ne(t["area_borough"])
print(f"SIRTA crashes whose reported borough differs from the borough of their assigned area: {int(boro_mismatch.sum())} "
      f"({100 * boro_mismatch.mean():.1f}%) — expected near borough edges (e.g. areas spanning Queens/Brooklyn)")

**Housing, by completion status**

,records,valid_coordinates_pct,valid_zip_pct,via_coordinates_pct,via_zip_pct,unassigned_pct,zip_coord_comparisons,zip_coord_agreement_pct
group,,,,,,,,
completed,6006,100.0,100.0,100.0,0.0,0.0,6003,94.6
confidential,1949,0.0,0.0,0.0,0.0,100.0,0,NaN
in_progress,1596,99.5,99.5,99.5,0.0,0.5,1588,96.9


**SIRTA, by crash year**

,records,valid_coordinates_pct,valid_zip_pct,via_coordinates_pct,via_zip_pct,unassigned_pct,zip_coord_comparisons,zip_coord_agreement_pct
crash_year,,,,,,,,
2022,2879,89.0,16.3,88.1,0.8,11.2,446,91.5
2023,3016,81.5,68.0,80.8,8.6,10.6,1790,91.3
2024,3116,66.2,69.9,65.5,22.1,12.4,1489,92.1
2025,2947,88.1,79.3,86.9,7.9,5.2,2102,92.0
2026,1351,87.7,73.5,86.6,7.2,6.2,896,91.2


**SIRTA, by reported borough**

,records,valid_coordinates_pct,valid_zip_pct,via_coordinates_pct,via_zip_pct,unassigned_pct,zip_coord_comparisons,zip_coord_agreement_pct
crash_borough,,,,,,,,
Bronx,2616,81.9,58.2,80.0,9.8,10.1,1265,91.6
Brooklyn,3954,83.8,65.8,83.3,10.7,6.0,2175,92.5
Manhattan,2733,79.8,62.3,78.2,9.9,11.9,1430,91.4
Queens,3282,80.8,58.1,80.7,9.0,10.4,1612,90.4
Staten Island,669,79.2,44.2,78.8,8.2,13.0,241,95.0
<NA>,55,83.6,0.0,81.8,0.0,18.2,0,NaN


**ZIP vs coordinate disagreements**

,disagreements,neighbouring_area_pct,overall_agreement_pct
housing,374.0,100.0,95.1
sirta,559.0,99.8,91.7


SIRTA crashes whose reported borough differs from the borough of their assigned area: 12 (0.1%) — expected near borough edges (e.g. areas spanning Queens/Brooklyn)


In [54]:
# Chart: how each year's crashes were placed in an area.
rel_plot = (t[t["crash_year"].isin(ANALYSIS_YEARS)].groupby(["crash_year", "assignment_method"], as_index=False).size())
rel_plot["share"] = rel_plot["size"] / rel_plot.groupby("crash_year")["size"].transform("sum") * 100
fig = px.bar(rel_plot, x="crash_year", y="share", color="assignment_method", barmode="stack",
             category_orders={"assignment_method": ["coordinates", "coordinates (boundary tie)", "reported ZIP", "unassigned"]},
             color_discrete_map={"coordinates": SERIES[0], "coordinates (boundary tie)": SERIES[6],
                                 "reported ZIP": SERIES[2], "unassigned": NEUTRAL},
             hover_data={"size": ":,", "share": ":.1f"},
             labels={"crash_year": "Crash year", "share": "Share of the year's crashes (%)", "assignment_method": "Placed in an area via"})
fig.update_xaxes(dtick=1)
style(fig, "How each year's serious crashes were placed in a ZIP area",
      "Coordinate-first assignment keeps coverage high in every year, but not identical across years").show()

### 8.7 Does better location data change the year-to-year comparison?
If more crashes can be placed in an area in one year than in another, the area counts go up or down for that reason alone. So we compare the change in **all crashes** (citywide, no location needed) with the change in **crashes placed in an area**. The share placed per borough and year is added to the integrated table in Section 9 as extra information.

In [55]:
# All crashes in 2022-2025, and how many of them could be placed in an area each year.
t_window_all = t[t["crash_year"].isin(ANALYSIS_YEARS)]
coverage_year = t_window_all.groupby("crash_year").agg(dated_crashes=("record_id", "size"),
                                                       assigned_crashes=("modzcta", lambda s: int(s.notna().sum())))
coverage_year["assignment_rate_pct"] = (100 * coverage_year["assigned_crashes"] / coverage_year["dated_crashes"]).round(2)
first, last = ANALYSIS_YEARS[0], ANALYSIS_YEARS[-1]
change_dated = coverage_year.loc[last, "dated_crashes"] / coverage_year.loc[first, "dated_crashes"] - 1
change_assigned = coverage_year.loc[last, "assigned_crashes"] / coverage_year.loc[first, "assigned_crashes"] - 1
COVERAGE_BIAS_PP = 100 * (change_assigned - change_dated)
display(coverage_year)
print(f"Change {first}->{last}: all dated crashes {100 * change_dated:+.1f}%, assigned crashes {100 * change_assigned:+.1f}%. "
      f"Coverage differences shift the apparent change by {COVERAGE_BIAS_PP:+.1f} percentage points.")

# Share of each borough's crashes per year that could be placed in an area (by the reported borough).
# Added to the panel in Section 9 as coverage information.
COVERAGE_RATES = (t_window_all.dropna(subset=["crash_borough"])
                  .groupby(["crash_borough", "crash_year"])["modzcta"].apply(lambda s: s.notna().mean())
                  .rename("crash_assignment_rate").reset_index()
                  .rename(columns={"crash_borough": "area_borough", "crash_year": "year"}))
COVERAGE_RATES["year"] = COVERAGE_RATES["year"].astype(int)
display(COVERAGE_RATES.pivot(index="area_borough", columns="year", values="crash_assignment_rate").mul(100).round(1))

,dated_crashes,assigned_crashes,assignment_rate_pct
crash_year,,,
2022,2879,2557,88.82
2023,3016,2696,89.39
2024,3116,2729,87.58
2025,2947,2795,94.84


Change 2022->2025: all dated crashes +2.4%, assigned crashes +9.3%. Coverage differences shift the apparent change by +6.9 percentage points.


year,2022,2023,2024,2025
area_borough,,,,
Bronx,88.8,88.3,86.8,95.0
Brooklyn,91.1,94.1,92.1,97.6
Manhattan,86.1,87.3,86.5,90.4
Queens,89.7,86.8,84.1,96.5
Staten Island,81.5,88.5,86.5,89.9


### 8.8 Counting per area and year, then joining
**What we do.** We count each cleaned dataset per `(modzcta, year)`, make a table with every area × year combination, and join both counts onto it. The joins use `validate="one_to_one"`, so pandas raises an error if a key appears twice instead of silently duplicating rows.

**Which records are counted.** Housing: buildings completed in 2022–2025 that could be placed in an area. Crashes: crashes from 2022–2025 that could be placed in an area. All other records are counted in the record ledger (Section 9.3) with the reason.

**Join keys:** `modzcta` and `year`. Section 9.4 runs these functions again to check that the result is reproducible.

In [56]:
# Short names for the income-band columns and the lists of count columns in the integrated table.
INCOME_BANDS = {"extremely_low_income_units": "units_eli", "very_low_income_units": "units_vli",
                "low_income_units": "units_li", "moderate_income_units": "units_mod",
                "middle_income_units": "units_mid", "other_income_units": "units_other_income"}
HOUSING_COUNT_COLS = ["buildings_completed", "units_completed", "buildings_units_unknown", "units_new_construction",
                      "units_preservation", *INCOME_BANDS.values(), "units_income_known"]
CRASH_COUNT_COLS = ["crashes_all", "crashes_local", "crashes_highway", "severe_injuries_all", "severe_injuries_local",
                    "fatalities_all", "fatalities_local", "crashes_injury_unknown", "crashes_local_repeat",
                    "crashes_local_first", "crashes_local_repeat_unknown", "crashes_via_coordinates", "crashes_date_suspect"]

def aggregate_housing(h_records, years=ANALYSIS_YEARS):
    # Completed buildings in 2022-2025 that were placed in an area, summed per area and year.
    # Income bands only count buildings whose band numbers add up to the total (Section 6.6).
    d = h_records[h_records["completion_status"].eq("completed") & h_records["completion_year"].isin(years)
                  & h_records["modzcta"].notna()].copy()
    d["year"] = d["completion_year"].astype(int)
    u = d["units_counted"]
    d["_nc"] = u.where(d["construction_type"].eq("New Construction"), 0)
    d["_pres"] = u.where(d["construction_type"].eq("Preservation"), 0)
    known = d["income_breakdown_status"].eq("match") & d["units_counted"].notna()
    for src, alias in INCOME_BANDS.items():
        d[alias] = d[src].where(known, 0)
    d["_inc_known"] = u.where(known, 0)
    d["_u_unknown"] = u.isna().astype(int)
    agg = d.groupby(["modzcta", "year"]).agg(
        buildings_completed=("record_id", "size"), units_completed=("units_counted", "sum"),
        buildings_units_unknown=("_u_unknown", "sum"), units_new_construction=("_nc", "sum"),
        units_preservation=("_pres", "sum"), **{a: (a, "sum") for a in INCOME_BANDS.values()},
        units_income_known=("_inc_known", "sum")).reset_index()
    return agg

def aggregate_crashes(t_records, years=ANALYSIS_YEARS):
    # Crashes in 2022-2025 that were placed in an area, summed per area and year.
    # Local-street and highway crashes are also counted separately.
    d = t_records[t_records["crash_year"].isin(years) & t_records["modzcta"].notna()].copy()
    d["year"] = d["crash_year"].astype(int)
    local = ~d["is_highway"]
    d["_local"], d["_hwy"] = local.astype(int), d["is_highway"].astype(int)
    d["_sev_local"] = d["total_severe_injuries"].where(local, 0)
    d["_fat_local"] = d["total_fatalities"].where(local, 0)
    d["_inj_unknown"] = d["injury_counts_missing"].astype(int)
    for status in ["repeat", "first", "unknown"]:
        d[f"_rep_{status}"] = (local & d["repeat_location_status"].eq(status)).astype(int)
    d["_via_coords"] = d["assignment_method"].str.startswith("coordinates").astype(int)
    d["_suspect"] = d["date_suspect"].astype(int)
    agg = d.groupby(["modzcta", "year"]).agg(
        crashes_all=("record_id", "size"), crashes_local=("_local", "sum"), crashes_highway=("_hwy", "sum"),
        severe_injuries_all=("total_severe_injuries", "sum"), severe_injuries_local=("_sev_local", "sum"),
        fatalities_all=("total_fatalities", "sum"), fatalities_local=("_fat_local", "sum"),
        crashes_injury_unknown=("_inj_unknown", "sum"), crashes_local_repeat=("_rep_repeat", "sum"),
        crashes_local_first=("_rep_first", "sum"), crashes_local_repeat_unknown=("_rep_unknown", "sum"),
        crashes_via_coordinates=("_via_coords", "sum"), crashes_date_suspect=("_suspect", "sum")).reset_index()
    return agg

def join_panel(h_agg, t_agg, years=ANALYSIS_YEARS):
    # Start from every area x year combination and add the housing and crash counts.
    # Stop if a key appears twice, because the join would then multiply rows.
    for label, agg in [("housing", h_agg), ("crash", t_agg)]:
        dups = int(agg.duplicated(["modzcta", "year"]).sum())
        if dups:
            raise ValidationError(f"{label} aggregate has {dups} duplicate (modzcta, year) keys; the join would multiply rows")
    grid = pd.MultiIndex.from_product([sorted(mz["modzcta"]), years], names=["modzcta", "year"]).to_frame(index=False)
    joined = (grid.merge(mz[["modzcta", "area_borough", "pop_est"]], on="modzcta", how="left", validate="many_to_one")
                  .merge(h_agg, on=["modzcta", "year"], how="left", validate="one_to_one", indicator="housing_merge")
                  .merge(t_agg, on=["modzcta", "year"], how="left", validate="one_to_one", indicator="crash_merge"))
    return joined

In [57]:
# Count, join, and show the empty cells that the joins created (they are handled in Section 9).
h_agg = aggregate_housing(h)
t_agg = aggregate_crashes(t)
panel_joined = join_panel(h_agg, t_agg)
print(f"Housing aggregate: {len(h_agg)} area-years | crash aggregate: {len(t_agg)} area-years | "
      f"joined grid: {len(panel_joined)} rows x {panel_joined.shape[1]} columns")
display(Markdown("**Missing values created by the joins (before post-integration cleaning)**"))
join_na = panel_joined[HOUSING_COUNT_COLS[:2] + CRASH_COUNT_COLS[:2] + ["pop_est"]].isna().sum().to_frame("missing after join")
display(join_na)
display(panel_joined[["housing_merge", "crash_merge"]].apply(lambda s: s.value_counts()).fillna(0).astype(int))

Housing aggregate: 383 area-years | crash aggregate: 685 area-years | joined grid: 708 rows x 31 columns


**Missing values created by the joins (before post-integration cleaning)**

,missing after join
buildings_completed,325
units_completed,325
crashes_all,23
crashes_local,23
pop_est,0


,housing_merge,crash_merge
both,383,685
left_only,325,23
right_only,0,0


## 9. Post-integration cleaning (required)
**New missing values from the joins.** If an area had no housing row or no crash row in a year, the join leaves those cells empty. The empty cells do not all mean the same thing:

| Columns | What an empty cell means | What we do |
|---|---|---|
| Housing counts | No HPD-financed building with a published location was completed there that year (H2 lists all of them) | Fill with **0** — a real zero. Confidential buildings are counted separately at borough level |
| Crash counts | No serious crash was *placed in* that area that year, but some crashes could not be placed anywhere | Fill with **0** and add the borough's placement rate, so a zero is not read as "certainly no crash" |
| Population | No population estimate | Rates stay **missing**, never 0 or infinite |

**Redundant or inconsistent columns.** The two merge-indicator columns are replaced by `has_housing_completion` and `has_assigned_crash`, and `pop_est` becomes `population_2018` (missing if it is not positive). Section 9.2 checks each area's borough against the boroughs reported by its records, and makes sure that crashes with no reported borough are not lost from the borough totals.

**Data-type mismatches.** After the joins, the count columns are decimals (because of the empty cells) and the keys have mixed types. We convert counts back to whole numbers, rates to decimals and keys to text.

### 9.1 Applying the post-integration cleaning

In [58]:
def post_integration_clean(joined):
    p = joined.copy()
    # Replace the merge indicators with simple True/False columns.
    p["has_housing_completion"] = p["housing_merge"].eq("both")
    p["has_assigned_crash"] = p["crash_merge"].eq("both")
    # Empty counts become 0 (see the table above); the borough placement rate is added as context.
    p[HOUSING_COUNT_COLS] = p[HOUSING_COUNT_COLS].fillna(0)
    p[CRASH_COUNT_COLS] = p[CRASH_COUNT_COLS].fillna(0)
    p = p.merge(COVERAGE_RATES, on=["area_borough", "year"], how="left", validate="many_to_one")
    # Rates per resident; they stay missing (not 0) when the population is missing.
    p["population_2018"] = p["pop_est"].where(p["pop_est"] > 0)
    pop = p["population_2018"]
    p["units_per_1k_residents"] = 1000 * p["units_completed"] / pop
    p["local_crash_rate_per_100k"] = 100000 * p["crashes_local"] / pop
    p["fatality_rate_per_100k"] = 100000 * p["fatalities_local"] / pop
    # Remove the redundant columns and fix the data types: whole numbers, decimals and text keys.
    p = p.drop(columns=["housing_merge", "crash_merge", "pop_est"])
    for c in HOUSING_COUNT_COLS + CRASH_COUNT_COLS:
        p[c] = p[c].round().astype("Int64")
    for c in ["population_2018", "crash_assignment_rate", "units_per_1k_residents", "local_crash_rate_per_100k", "fatality_rate_per_100k"]:
        p[c] = p[c].astype("Float64")
    p["modzcta"], p["area_borough"] = p["modzcta"].astype("string"), p["area_borough"].astype("string")
    p["year"] = p["year"].astype(int)
    lead = ["modzcta", "year", "area_borough", "population_2018", "has_housing_completion", "has_assigned_crash"]
    return p[lead + [c for c in p.columns if c not in lead]].sort_values(["modzcta", "year"]).reset_index(drop=True)

def build_panel(h_records, t_records, years=ANALYSIS_YEARS):
    # The whole integration in one step: count, join, clean. Used in Section 9.4 to rebuild the table and compare.
    return post_integration_clean(join_panel(aggregate_housing(h_records, years), aggregate_crashes(t_records, years), years))

# Apply the cleaning, then show what is still missing and how the data types changed.
panel = post_integration_clean(panel_joined)
display(Markdown("**Missing values after post-integration cleaning**"))
display(panel.isna().sum().loc[lambda s: s > 0].to_frame("missing (undefined, by design)")
        if panel.isna().any().any() else Markdown("No missing values remain in the panel."))
print(f"Panel: {panel.shape[0]} rows x {panel.shape[1]} columns")
display(Markdown("**Data types before and after post-integration cleaning (selected columns)**"))
type_cols = ["modzcta", "year", "units_completed", "crashes_all", "crashes_local", "population_2018"]
display(pd.DataFrame({"after join": panel_joined[[c if c != "population_2018" else "pop_est" for c in type_cols]].dtypes.astype(str).values,
                      "after cleaning": panel[type_cols].dtypes.astype(str).values}, index=type_cols))
display(panel.head())

**Missing values after post-integration cleaning**

No missing values remain in the panel.

Panel: 708 rows x 35 columns


**Data types before and after post-integration cleaning (selected columns)**

,after join,after cleaning
modzcta,object,string
year,int64,int64
units_completed,Int64,Int64
crashes_all,float64,Int64
crashes_local,float64,Int64
population_2018,int64,Float64


,modzcta,year,area_borough,population_2018,has_housing_completion,has_assigned_crash,buildings_completed,units_completed,buildings_units_unknown,units_new_construction,units_preservation,units_eli,units_vli,units_li,units_mod,units_mid,units_other_income,units_income_known,crashes_all,crashes_local,crashes_highway,severe_injuries_all,severe_injuries_local,fatalities_all,fatalities_local,crashes_injury_unknown,crashes_local_repeat,crashes_local_first,crashes_local_repeat_unknown,crashes_via_coordinates,crashes_date_suspect,crash_assignment_rate,units_per_1k_residents,local_crash_rate_per_100k,fatality_rate_per_100k
0,10001,2022,Manhattan,23072.0,False,True,0,0,0,0,0,0,0,0,0,0,0,0,16,16,0,17,17,1,1,0,0,16,0,16,0,0.861404,0.0,69.348128,4.334258
1,10001,2023,Manhattan,23072.0,True,True,2,378,0,378,0,0,94,238,46,0,0,378,25,25,0,23,23,2,2,0,2,18,5,23,0,0.872894,16.383495,108.356449,8.668516
2,10001,2024,Manhattan,23072.0,True,True,2,170,0,170,0,0,54,89,27,0,0,170,26,26,0,29,29,0,0,0,1,21,4,22,0,0.865415,7.368239,112.690707,0.0
3,10001,2025,Manhattan,23072.0,True,True,1,70,0,70,0,0,31,25,14,0,0,70,20,20,0,20,20,1,1,0,3,17,0,18,0,0.904437,3.033981,86.68516,4.334258
4,10002,2022,Manhattan,74993.0,True,True,2,135,0,135,0,18,56,56,0,4,1,135,22,18,4,27,21,2,1,0,3,15,0,21,0,0.861404,1.800168,24.00224,1.333458


### 9.2 Unfinished projects, borough check and borough totals
The first cell below comes from the corrected notebook that was run in Colab (its fix for missing values in the borough comparison is kept exactly as it was). It counts unfinished buildings per area, checks that each area's borough agrees with the boroughs reported by its records, and builds borough-by-year totals. These totals keep what the area table cannot show: confidential housing units and crashes that could not be placed in an area.

**Borough correction.** An earlier version of these totals left out the crashes whose borough column was empty (36 crashes from 2022–2025, 28 of them placed in an area). They are now counted using the borough of their ZIP area, or as *Unknown*, and the check below the cell confirms that the borough totals add up to the full data. The last cell splits the unfinished units by construction type for RQ8.

In [59]:
# Information per area (not per year): unfinished buildings and units completed before 2022

pipeline = (
    h[
        h["completion_status"].eq("in_progress")
        & h["modzcta"].notna()
    ]
    .groupby("modzcta")
    .agg(
        pipeline_buildings=("record_id", "size"),
        pipeline_units=("units_counted", "sum")
    )
)

prior_stock = (
    h[
        h["completion_status"].eq("completed")
        & h["completion_year"].between(
            2014, ANALYSIS_YEARS[0] - 1
        )
        & h["modzcta"].notna()
    ]
    .groupby("modzcta")
    .agg(
        units_completed_2014_2021=("units_counted", "sum")
    )
)

# Determine the majority borough for each MODZCTA
votes = pd.concat(
    [
        h[["modzcta", "borough"]],
        t[["modzcta", "crash_borough"]].rename(
            columns={"crash_borough": "borough"}
        )
    ]
).dropna()

majority = (
    votes.groupby("modzcta")["borough"]
    .agg(lambda s: s.value_counts().idxmax())
)

# Build area summary
area_summary = (
    mz[["modzcta", "area_borough", "pop_est"]]
    .rename(columns={"pop_est": "population_2018"})
    .merge(
        pipeline,
        on="modzcta",
        how="left",
        validate="one_to_one"
    )
    .merge(
        prior_stock,
        on="modzcta",
        how="left",
        validate="one_to_one"
    )
)

# Fill missing counts
for c in [
    "pipeline_buildings",
    "pipeline_units",
    "units_completed_2014_2021"
]:
    area_summary[c] = (
        area_summary[c]
        .fillna(0)
        .round()
        .astype("Int64")
    )

# Map each area to its majority borough
area_summary["record_majority_borough"] = (
    area_summary["modzcta"]
    .map(majority)
    .astype("string")
)

# Fix: missing values (pd.NA) are treated as False
# before the borough comparison, so the check cannot crash.

borough_agrees = (
    area_summary["record_majority_borough"]
    .eq(area_summary["area_borough"])
    .fillna(False)
    .to_numpy(dtype=bool)
)

# Classify borough agreement
area_summary["borough_check"] = np.where(
    area_summary["record_majority_borough"].isna(),
    "no records",
    np.where(
        borough_agrees,
        "agrees",
        "differs"
    )
)

# Remove geometry column if present
area_summary = pd.DataFrame(
    area_summary.drop(
        columns="geometry",
        errors="ignore"
    )
)

# Display borough validation results
display(
    area_summary["borough_check"]
    .value_counts()
    .to_frame("areas")
)

# Display areas with borough disagreements
display(
    area_summary[
        area_summary["borough_check"].eq("differs")
    ]
)


# Borough-year totals
# Keep confidential units and unassigned crashes
# that cannot be represented in the area panel.

hb = h[
    h["completion_status"].eq("completed")
    & h["completion_year"].isin(ANALYSIS_YEARS)
]

borough_year = (
    hb.groupby(["borough", "completion_year"])
    .agg(
        units_completed_all=(
            "units_counted",
            "sum"
        ),
        units_completed_confidential=(
            "units_counted",
            lambda s: s[
                hb.loc[s.index, "is_confidential"]
            ].sum()
        ),
        units_completed_in_areas=(
            "units_counted",
            lambda s: s[
                hb.loc[s.index, "modzcta"].notna()
            ].sum()
        )
    )
    .rename_axis(["borough", "year"])
    .reset_index()
)

# Crashes with no reported borough are no longer dropped here: summary_borough uses the
# borough of the crash's ZIP area when it was placed in one, and "Unknown" otherwise (Section 8.5).
tb = (
    t_window_all
    .groupby(["summary_borough", "crash_year"])
    .agg(
        crashes_all=("record_id", "size"),
        crashes_assigned=(
            "modzcta",
            lambda s: int(s.notna().sum())
        ),
        crashes_borough_derived=(
            "borough_source",
            lambda s: int(s.eq("derived from ZIP area").sum())
        )
    )
    .rename_axis(["borough", "year"])
    .reset_index()
)

# Merge housing and crash borough-year summaries
borough_year = borough_year.merge(
    tb,
    on=["borough", "year"],
    how="outer",
    validate="one_to_one"
).fillna(0)

# Ensure year is an integer
borough_year["year"] = borough_year["year"].astype(int)

# Display final borough-year summary
display(borough_year)

,areas
borough_check,
agrees,176
no records,1


,modzcta,area_borough,population_2018,pipeline_buildings,pipeline_units,units_completed_2014_2021,record_majority_borough,borough_check


,borough,year,units_completed_all,units_completed_confidential,units_completed_in_areas,crashes_all,crashes_assigned,crashes_borough_derived
0,Bronx,2022,12006,54,11952,600,533,0
1,Bronx,2023,17422,207,17215,600,530,0
2,Bronx,2024,9373,39,9334,622,540,0
3,Bronx,2025,7043,75,6968,543,516,0
4,Brooklyn,2022,3469,82,3387,885,806,0
5,Brooklyn,2023,7046,137,6909,874,822,0
6,Brooklyn,2024,4979,69,4910,919,846,0
7,Brooklyn,2025,6023,72,5951,879,858,0
8,Manhattan,2022,2696,9,2687,570,491,0
9,Manhattan,2023,3702,8,3694,653,570,0


In [60]:
# Check that the borough table adds up: every 2022-2025 crash and every completed unit must be
# counted exactly once, including crashes with no reported borough.
display(Markdown("**Where each 2022–2025 crash's borough comes from**"))
display(pd.crosstab(t_window_all["borough_source"], t_window_all["modzcta"].notna().map({True: "placed in an area", False: "not placed"}),
                    margins=True, margins_name="total"))
check("borough_summary_unique_keys", not borough_year.duplicated(["borough", "year"]).any(),
      "one row per borough and year in the borough table", "the borough table has repeated (borough, year) rows", stage="post-integration cleaning")
check("borough_summary_reconciles",
      int(borough_year["crashes_all"].sum()) == len(t_window_all)
      and int(borough_year["crashes_assigned"].sum()) == int(panel["crashes_all"].sum())
      and int(borough_year["units_completed_all"].sum()) == int(hb["units_counted"].sum()),
      f"borough totals match the full data: {len(t_window_all):,} crashes ({int(panel['crashes_all'].sum()):,} placed in an area) "
      f"and {int(hb['units_counted'].sum()):,} completed units",
      "the borough totals do not add up to the crash or housing totals", stage="post-integration cleaning")

**Where each 2022–2025 crash's borough comes from**

modzcta,not placed,placed in an area,total
borough_source,,,
derived from ZIP area,0,28,28
reported,1173,10749,11922
unknown,8,0,8
total,1181,10777,11958


[PASS] borough_summary_unique_keys: one row per borough and year in the borough table
[PASS] borough_summary_reconciles: borough totals match the full data: 11,958 crashes (10,777 placed in an area) and 101,249 completed units


True

In [61]:
# Unfinished affordable units per area, split by construction type (used in RQ8).
# "Unfinished" means the building has no completion date yet, but its project has already started.
pipe_records = h[h["completion_status"].eq("in_progress") & h["modzcta"].notna() & (h["project_start_date"] <= SNAPSHOT_TS)]
area_pipeline = (pipe_records.groupby("modzcta")
                 .agg(pipeline_buildings=("record_id", "size"), pipeline_units=("units_counted", "sum"),
                      pipeline_units_nc=("units_counted", lambda s: s[pipe_records.loc[s.index, "construction_type"].eq("New Construction")].sum()),
                      pipeline_units_pres=("units_counted", lambda s: s[pipe_records.loc[s.index, "construction_type"].eq("Preservation")].sum()),
                      earliest_pipeline_start=("start_year", "min"))
                 .reset_index())
for c in ["pipeline_buildings", "pipeline_units", "pipeline_units_nc", "pipeline_units_pres"]:
    area_pipeline[c] = area_pipeline[c].astype(int)
check("pipeline_type_split", (area_pipeline["pipeline_units_nc"] + area_pipeline["pipeline_units_pres"] == area_pipeline["pipeline_units"]).all(),
      "new-construction + preservation pipeline units = all pipeline units in every area", "pipeline type split does not add up",
      stage="post-integration cleaning")
print(f"{len(area_pipeline)} areas have unfinished affordable buildings ({int(area_pipeline['pipeline_units'].sum()):,} units)")

[PASS] pipeline_type_split: new-construction + preservation pipeline units = all pipeline units in every area
135 areas have unfinished affordable buildings (78,031 units)


### 9.3 Record ledger — from the raw files to the final table
This table follows the records step by step, so it is clear how many records reach the area table and why the others do not.

In [62]:
# Count the records (and housing units) at each step from the raw files to the integrated table.
h_completed_window = h[h["completion_status"].eq("completed") & h["completion_year"].isin(ANALYSIS_YEARS)]
h_in_area = h_completed_window[h_completed_window["modzcta"].notna()]
t_in_area = t_window_all[t_window_all["modzcta"].notna()]
ledger = pd.DataFrame([
    ("housing", "raw rows", len(h_raw), int(h_typed["all_counted_units"].sum())),
    ("housing", "excluded during cleaning", 0, 0),
    ("housing", "completed buildings (any year)", int(h["completion_status"].eq("completed").sum()),
     int(h.loc[h["completion_status"].eq("completed"), "units_counted"].sum())),
    ("housing", "completed in 2022-2025", len(h_completed_window), int(h_completed_window["units_counted"].sum())),
    ("housing", "  ... confidential (borough totals only)", int(h_completed_window["is_confidential"].sum()),
     int(h_completed_window.loc[h_completed_window["is_confidential"], "units_counted"].sum())),
    ("housing", "  ... published but unassigned", int((~h_completed_window["is_confidential"] & h_completed_window["modzcta"].isna()).sum()),
     int(h_completed_window.loc[~h_completed_window["is_confidential"] & h_completed_window["modzcta"].isna(), "units_counted"].sum())),
    ("housing", "  ... assigned to an area (enters the panel)", len(h_in_area), int(h_in_area["units_counted"].sum())),
    ("housing", "panel totals", int(panel["buildings_completed"].sum()), int(panel["units_completed"].sum())),
    ("housing", "in progress (pipeline, any area)", int(h["completion_status"].eq("in_progress").sum()),
     int(h.loc[h["completion_status"].eq("in_progress"), "units_counted"].sum())),
    ("sirta", "raw rows", T_ROWS_START, None),
    ("sirta", "excluded during cleaning", len(t_excluded), None),
    ("sirta", "retained after cleaning", len(t), None),
    ("sirta", "crash year 2022-2025", len(t_window_all), None),
    ("sirta", "  ... unassigned (borough totals only)", int(t_window_all["modzcta"].isna().sum()), None),
    ("sirta", "  ... assigned to an area (enters the panel)", len(t_in_area), None),
    ("sirta", "panel totals (crashes_all)", int(panel["crashes_all"].sum()), None),
    ("panel", "final panel rows (areas x years)", len(panel), None),
], columns=["dataset", "stage", "records", "counted_units"])
ledger["counted_units"] = ledger["counted_units"].astype("Int64")
ledger

,dataset,stage,records,counted_units
0,housing,raw rows,9551,320946
1,housing,excluded during cleaning,0,0
2,housing,completed buildings (any year),7885,242256
3,housing,completed in 2022-2025,2957,101249
4,housing,... confidential (borough totals only),716,1192
5,housing,... published but unassigned,0,0
6,housing,... assigned to an area (enters the panel),2241,100057
7,housing,panel totals,2241,100057
8,housing,"in progress (pipeline, any area)",1666,78690
9,sirta,raw rows,13754,<NA>


### 9.4 Checks on the integrated table
These checks confirm that the joins did not multiply or lose any records and that the totals match the cleaned data.

In [63]:
# Checks after the join: no repeated keys, the full area x year grid, and totals equal to the cleaned records.
stage = "integrated validation"
check("panel_key_unique", not panel.duplicated(["modzcta", "year"]).any(), "(modzcta, year) is unique", "duplicate panel keys", stage=stage)
check("panel_complete_grid", len(panel) == len(mz) * len(ANALYSIS_YEARS),
      f"{len(panel)} rows = {len(mz)} areas x {len(ANALYSIS_YEARS)} years", "grid rows are missing or duplicated", stage=stage)
check("panel_years", sorted(panel["year"].unique()) == ANALYSIS_YEARS, f"years are exactly {ANALYSIS_YEARS}",
      f"unexpected years {sorted(panel['year'].unique())}", stage=stage)
check("panel_units_total", int(panel["units_completed"].sum()) == int(h_in_area["units_counted"].sum()),
      f"panel units ({int(panel['units_completed'].sum()):,}) = assigned in-window units", "unit totals changed during integration", stage=stage)
check("panel_buildings_total", int(panel["buildings_completed"].sum()) == len(h_in_area),
      f"panel buildings ({len(h_in_area):,}) = assigned in-window buildings", "building totals changed", stage=stage)
check("panel_crashes_total", int(panel["crashes_all"].sum()) == len(t_in_area),
      f"panel crashes ({len(t_in_area):,}) = assigned in-window crashes", "crash totals changed during integration", stage=stage)
check("panel_local_plus_highway", (panel["crashes_local"] + panel["crashes_highway"] == panel["crashes_all"]).all(),
      "local + highway = all crashes in every row", "crash scope split does not add up", stage=stage)
check("panel_fatalities_total", int(panel["fatalities_all"].sum()) == int(t_in_area["total_fatalities"].sum()),
      "fatality totals preserved", "fatality totals changed", stage=stage)
check("panel_repeat_split", (panel["crashes_local_repeat"] + panel["crashes_local_first"] + panel["crashes_local_repeat_unknown"]
                             == panel["crashes_local"]).all(), "repeat + first + unknown = local crashes", "repeat split does not add up", stage=stage)
counts = panel[HOUSING_COUNT_COLS + CRASH_COUNT_COLS]
check("panel_no_negative_counts", (counts >= 0).all().all(), "no negative counts", "negative counts in the panel", stage=stage)
check("panel_count_types", all(str(panel[c].dtype) == "Int64" for c in counts.columns), "all counts are integers", "non-integer count columns", stage=stage)
rates = panel[["units_per_1k_residents", "local_crash_rate_per_100k"]]
check("panel_rates_finite", not np.isinf(rates.astype(float).to_numpy()).any(), "no infinite rates", "infinite rates found", stage=stage)
check("panel_rates_undefined_without_population", rates[panel["population_2018"].isna()].isna().all().all(),
      "rates are missing exactly where population is missing", "a rate exists without a population", stage=stage)
check("panel_no_unexpected_missing", panel.drop(columns=["population_2018", "crash_assignment_rate",
                                                         "units_per_1k_residents", "local_crash_rate_per_100k", "fatality_rate_per_100k"]).notna().all().all(),
      "no missing values outside population-dependent and coverage columns", "unexpected missing values in the panel", stage=stage)
rebuilt = build_panel(h, t)
try:
    pd.testing.assert_frame_equal(rebuilt, panel)
    reproducible = True
except AssertionError as err:
    reproducible, repro_msg = False, str(err)[:300]
check("panel_reproducible", reproducible, "rebuilding the panel from the cleaned records gives an identical table",
      f"rebuilt panel differs: {'' if reproducible else repro_msg}", stage=stage)
print(f"Area-years with zero housing completions: {100 * (panel['units_completed'] == 0).mean():.1f}% | "
      f"with zero assigned serious crashes: {100 * (panel['crashes_all'] == 0).mean():.1f}%")

[PASS] panel_key_unique: (modzcta, year) is unique
[PASS] panel_complete_grid: 708 rows = 177 areas x 4 years
[PASS] panel_years: years are exactly [2022, 2023, 2024, 2025]
[PASS] panel_units_total: panel units (100,057) = assigned in-window units
[PASS] panel_buildings_total: panel buildings (2,241) = assigned in-window buildings
[PASS] panel_crashes_total: panel crashes (10,777) = assigned in-window crashes
[PASS] panel_local_plus_highway: local + highway = all crashes in every row
[PASS] panel_fatalities_total: fatality totals preserved
[PASS] panel_repeat_split: repeat + first + unknown = local crashes
[PASS] panel_no_negative_counts: no negative counts
[PASS] panel_count_types: all counts are integers
[PASS] panel_rates_finite: no infinite rates
[PASS] panel_rates_undefined_without_population: rates are missing exactly where population is missing
[PASS] panel_no_unexpected_missing: no missing values outside population-dependent and coverage columns
[PASS] panel_reproducible: rebui

## 10. Research questions and eight visualizations
**Definitions used in all eight questions**
* **Area** = a MODZCTA ZIP-code area. **Period** = 2022–2025.
* **Affordable units completed** = affordable units in buildings completed in the period — **new construction and preservation together**, unless a chart says otherwise.
* **Serious-crash rate** = serious crashes on local streets per 100,000 residents **per year** (the total for the selected years ÷ the number of years ÷ population × 100,000). Highway crashes are left out by default, because expressways cross areas they do not serve.
* **Population** = DOHMH's 2018 estimate, the only population figure available for these areas. It is older than the study period and is used for both the housing and the crash rates.
* **Production tier** = *No completions*, or the *Lower half* / *Upper half* of the areas that completed units (split at their median units per 1,000 residents).

All results describe **areas**, not individual residents, and they show associations, not causes. We only use simple statistics: counts, percentages, medians, one rank correlation and percentile ranks.

### 10.1 Calculation and chart functions (shared with the website)
The two code cells below are **the same as `src/analysis.py` and `src/figures.py`** in the GitHub repository. The Dash website uses those files, so the website and this notebook calculate and draw everything in the same way.

| Function | Used for |
|---|---|
| `compute_area_metrics` | One row per area for any filter (years, boroughs, construction type, crash measure, road type) |
| `spearman`, `describe_strength` | RQ1 rank correlation and its description in words |
| `add_production_tier`, `classify_quadrants` | Production tiers; RQ2 area groups |
| `indexed_trend`, `coverage_by_year` | RQ3 change over time and location coverage |
| `time_block_shares` | RQ4 weekday × time-of-day shares |
| `classify_dominant_type`, `type_summary` | RQ5 construction-type groups |
| `weighted_band_rates` | RQ6 crash rate where each income band's units were completed |
| `lookback_shares` | RQ7 shares, with unknown values kept apart |
| `priority_ranking` | RQ8 list of areas to look at more closely |
| `parse_search`, `merge_filters`, `build_report`, `findings_text` | Website search, filters and report |

In [64]:
"""Calculations for the eight research questions, used by both the notebook and the Dash website.

Keeping them in one file means the notebook and the website always give the same numbers.
The functions take tables and return tables; only load_app_data reads files.

The tables come from the files the notebook saves in data/processed/:
  panel     -- one row per ZIP area (MODZCTA) and year, 2022-2025 (panel_modzcta_year.csv)
  crashes   -- one row per serious crash placed in a ZIP area, 2022-2025 (crashes_for_app.csv)
  pipeline  -- one row per ZIP area with unfinished affordable units (area_pipeline.csv)
  coverage  -- per borough and year: all crashes and crashes placed in an area (borough_year_summary.csv)
  buildings -- completed buildings 2022-2025 with coordinates, for the RQ2 map (buildings_for_app.csv)
"""
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

YEARS = [2022, 2023, 2024, 2025]
BOROUGHS = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]
HOUSING_TYPES = {"All": "units_completed", "New Construction": "units_new_construction", "Preservation": "units_preservation"}
PIPELINE_COLUMNS = {"All": "pipeline_units", "New Construction": "pipeline_units_nc", "Preservation": "pipeline_units_pres"}
CRASH_MEASURES = {"crashes": "Serious crashes", "severe_injuries": "Severe injuries", "fatalities": "Fatalities"}
CRASH_SCOPES = {"local": "Local streets", "all": "All roads (incl. highways)"}
TIER_ORDER = ["No completions", "Lower half", "Upper half"]
QUADRANTS = ["High production · high crash rate", "High production · lower crash rate",
             "Lower/no production · high crash rate", "Lower/no production · lower crash rate"]
TYPE_ORDER = ["New-construction dominant", "Mixed", "Preservation dominant", "No completions"]
BAND_LABELS = {"units_eli": "Extremely low income", "units_vli": "Very low income", "units_li": "Low income",
               "units_mod": "Moderate income", "units_mid": "Middle income", "units_other_income": "Other income units"}
BLOCKS = ["00–03", "03–06", "06–09", "09–12", "12–15", "15–18", "18–21", "21–24"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MIN_AREAS_FOR_CORRELATION = 10
DEFAULT_FILTERS = {"boroughs": list(BOROUGHS), "years": list(YEARS), "housing_type": "All",
                   "crash_measure": "crashes", "crash_scope": "local"}


def as_bool(values):
    """Turn a True/False column that may contain missing values into plain True/False (missing counts as False)."""
    return pd.Series(values).fillna(False).to_numpy(dtype=bool)


# Area numbers used by every research question

def compute_area_metrics(panel, years=YEARS, boroughs=None, housing_type="All", crash_measure="crashes", crash_scope="local"):
    """One row per ZIP area for the selected filters.

    units_per_1k = affordable units completed in the selected years per 1,000 residents (2018 population)
    crash_rate   = crashes (or injuries / deaths) per 100,000 residents per year:
                   the total is divided by the number of selected years, so one year and four years are comparable
    """
    years = sorted(int(y) for y in years)
    if not years or not set(years) <= set(YEARS):
        raise ValueError(f"years must be a non-empty subset of {YEARS}")
    if housing_type not in HOUSING_TYPES or crash_measure not in CRASH_MEASURES or crash_scope not in CRASH_SCOPES:
        raise ValueError("unknown housing_type, crash_measure or crash_scope")
    p = panel[panel["year"].isin(years)]
    if boroughs:
        p = p[p["area_borough"].isin(boroughs)]
    crash_col = f"{crash_measure}_{crash_scope}"
    sums = ["units_completed", "units_new_construction", "units_preservation", *BAND_LABELS, "crashes_local", "crashes_all",
            "fatalities_local", "crashes_local_repeat", "crashes_local_first", "crashes_local_repeat_unknown"]
    area = (p.assign(units=p[HOUSING_TYPES[housing_type]].astype(float), crash_total=p[crash_col].astype(float))
              .groupby("modzcta", as_index=False)
              .agg(area_borough=("area_borough", "first"), population_2018=("population_2018", "first"),
                   units=("units", "sum"), crash_total=("crash_total", "sum"), **{c: (c, "sum") for c in sums}))
    population = area["population_2018"].astype(float)
    area["n_years"] = len(years)
    area["units_per_1k"] = 1000 * area["units"] / population
    area["crash_rate"] = 100000 * area["crash_total"] / len(years) / population
    return area


def spearman(x, y, min_n=MIN_AREAS_FOR_CORRELATION):
    """Spearman rank correlation and the number of areas used. With fewer than 10 areas we return NaN instead of a number."""
    d = pd.DataFrame({"x": pd.to_numeric(x, errors="coerce"), "y": pd.to_numeric(y, errors="coerce")}).dropna()
    if len(d) < min_n or d["x"].nunique() < 2 or d["y"].nunique() < 2:
        return np.nan, len(d)
    return float(d["x"].rank().corr(d["y"].rank())), len(d)


def describe_strength(rho):
    """Describe a correlation in words (negligible, weak, moderate or strong). It never says anything about cause."""
    if pd.isna(rho):
        return "not enough areas to measure"
    size = abs(rho)
    strength = "negligible" if size < 0.1 else "weak" if size < 0.3 else "moderate" if size < 0.5 else "strong"
    return f"{strength} {'positive' if rho > 0 else 'negative'} association"


def add_production_tier(area):
    """Put each area in a production tier: No completions (0 units), or the lower or upper half of the areas
    that did complete units (split at their median units per 1,000 residents)."""
    a = area.copy()
    has = a["units"].gt(0) & a["units_per_1k"].notna()
    median = a.loc[has, "units_per_1k"].median()
    tier = np.select([as_bool(a["units"].eq(0)), as_bool(has & a["units_per_1k"].gt(median)), as_bool(has)],
                     ["No completions", "Upper half", "Lower half"], default="")
    a["production_tier"] = pd.Series(tier, index=a.index).replace("", np.nan)
    a.loc[a["population_2018"].isna(), "production_tier"] = np.nan
    a.attrs["tier_median"] = float(median) if pd.notna(median) else np.nan
    return a


def classify_quadrants(area):
    """RQ2: combine the production tier (upper half = high) with the crash rate (at or above the median = high)."""
    crash_median = area["crash_rate"].median()
    high_prod = as_bool(area["production_tier"].eq("Upper half"))
    high_crash = as_bool(area["crash_rate"] >= crash_median)
    quadrant = np.select([high_prod & high_crash, high_prod, high_crash], QUADRANTS[:3], default=QUADRANTS[3])
    return pd.Series(quadrant, index=area.index).where(area["crash_rate"].notna()), float(crash_median)


def indexed_trend(panel, tier_by_area, crash_col):
    """RQ3: crashes per tier and year, shown as an index where the first selected year = 100."""
    p = panel.assign(tier=panel["modzcta"].map(tier_by_area)).dropna(subset=["tier"])
    trend = p.groupby(["tier", "year"], as_index=False).agg(value=(crash_col, "sum"), areas=("modzcta", "nunique"))
    trend["value"] = trend["value"].astype(float)
    base = trend[trend["year"] == trend["year"].min()].set_index("tier")["value"]
    trend["index"] = 100 * trend["value"] / trend["tier"].map(base).replace(0, np.nan)
    return trend


def time_block_shares(crashes, upper_areas):
    """RQ4: for upper-half areas and for all other areas, the share of crashes in each weekday x 3-hour cell
    (each group adds up to 100%), and the difference between the two groups in percentage points."""
    c = crashes.copy()
    c["group"] = np.where(c["modzcta"].isin(upper_areas), "upper", "other")
    c["time_block"] = np.array(BLOCKS)[(c["crash_hour"].astype(int) // 3).to_numpy()]     # hour 0-2 -> "00–03", 3-5 -> "03–06", ...
    counts = (c.groupby(["group", "time_block", "weekday"]).size().unstack("group")
                .reindex(pd.MultiIndex.from_product([BLOCKS, DAYS], names=["time_block", "weekday"]))
                .reindex(columns=["upper", "other"]).fillna(0))
    n_upper, n_other = counts["upper"].sum(), counts["other"].sum()
    cells = counts.reset_index().rename(columns={"upper": "count_upper", "other": "count_other"})
    cells["share_upper_pct"] = 100 * cells["count_upper"] / n_upper if n_upper else np.nan
    cells["share_other_pct"] = 100 * cells["count_other"] / n_other if n_other else np.nan
    cells["diff_pp"] = cells["share_upper_pct"] - cells["share_other_pct"]
    cells.attrs.update({"n_upper": int(n_upper), "n_other": int(n_other)})
    return cells


def classify_dominant_type(area, nc_high=0.6, nc_low=0.4):
    """RQ5: new-construction dominant if more than 60% of the completed units are new construction,
    preservation dominant if less than 40%, otherwise mixed."""
    total = area["units_new_construction"].astype(float) + area["units_preservation"].astype(float)
    nc_share = area["units_new_construction"].astype(float) / total.where(total > 0)
    kind = np.select([as_bool(total.eq(0)), as_bool(nc_share > nc_high), as_bool(nc_share < nc_low)],
                     ["No completions", "New-construction dominant", "Preservation dominant"], default="Mixed")
    return pd.Series(kind, index=area.index), nc_share


def type_summary(area):
    """RQ5 table: number of areas, median crash rate and the middle 50% range for each group."""
    d = area.dropna(subset=["crash_rate"])
    s = d.groupby("dominant_type").agg(areas=("modzcta", "size"), median_crash_rate=("crash_rate", "median"),
                                       q1=("crash_rate", lambda v: v.quantile(0.25)), q3=("crash_rate", lambda v: v.quantile(0.75)),
                                       units=("units", "sum"))
    return s.reindex(TYPE_ORDER)


def weighted_band_rates(area):
    """RQ6: for each income band, the crash rate of the area where its average unit was completed
    (each area's rate weighted by the band's units there). For comparison, the rate of the average resident
    (each area's rate weighted by its population)."""
    v = area.dropna(subset=["crash_rate", "population_2018"])
    rows = []
    for col, label in BAND_LABELS.items():
        units = v[col].astype(float)
        rate = (units * v["crash_rate"]).sum() / units.sum() if units.sum() > 0 else np.nan   # weighted average
        rows.append({"band": label, "units": int(units.sum()), "weighted_crash_rate": rate})
    pop = v["population_2018"].astype(float)
    resident_avg = float((pop * v["crash_rate"]).sum() / pop.sum()) if pop.sum() > 0 else np.nan
    return pd.DataFrame(rows), resident_avg


def lookback_shares(area):
    """RQ7: per production tier, count crashes with an earlier similar serious injury recorded, with none recorded,
    and with a blank (unknown) value. The share only uses the known values; a blank is never counted as "none"."""
    s = (area.dropna(subset=["production_tier"]).groupby("production_tier")
             [["crashes_local_repeat", "crashes_local_first", "crashes_local_repeat_unknown"]].sum().astype(int))
    s.columns = ["earlier_recorded", "none_recorded", "unknown"]
    known = s["earlier_recorded"] + s["none_recorded"]
    s["share_known_pct"] = 100 * s["earlier_recorded"] / known.where(known > 0)
    return s.reindex(TIER_ORDER)


def priority_ranking(area, pipeline, housing_type="All", top_n=15):
    """RQ8: rank the areas by unfinished units per resident and by crash rate (as percentiles); the score is the
    average of the two. A higher score puts the area higher on the list. It is a list of places to look at more
    closely, not a prediction. Equal scores share a rank and are ordered by the crash percentile."""
    col = PIPELINE_COLUMNS[housing_type]
    a = area.merge(pipeline[["modzcta", "pipeline_buildings", "pipeline_units", "pipeline_units_nc", "pipeline_units_pres"]],
                   on="modzcta", how="left", validate="one_to_one")
    for c in ["pipeline_buildings", "pipeline_units", "pipeline_units_nc", "pipeline_units_pres"]:
        a[c] = a[c].fillna(0)
    a["pipeline_selected"] = a[col].astype(float)
    a["pipeline_per_1k"] = 1000 * a["pipeline_selected"] / a["population_2018"].astype(float)
    # Percentile ranks (0-100) among the areas that have both values.
    valid = a["pipeline_per_1k"].notna() & a["crash_rate"].notna()
    a["pipeline_pct"] = a.loc[valid, "pipeline_per_1k"].rank(pct=True) * 100
    a["crash_pct"] = a.loc[valid, "crash_rate"].rank(pct=True) * 100
    a["priority_score"] = (a["pipeline_pct"] + a["crash_pct"]) / 2
    a["nc_pipeline_share"] = a["pipeline_units_nc"] / a["pipeline_units"].where(a["pipeline_units"] > 0)
    # Only areas with unfinished units can be on the list. Ties are sorted in a fixed order, so the list is always the same.
    top = a[valid & a["pipeline_selected"].gt(0)].sort_values(
        ["priority_score", "crash_pct", "pipeline_pct", "modzcta"], ascending=[False, False, False, True]).head(top_n).copy()
    top["priority_rank"] = top["priority_score"].rank(method="min", ascending=False).astype(int)
    return top


# Website search (simple keyword rules) and the full report

BOROUGH_WORDS = {"bronx": "Bronx", "bx": "Bronx", "brooklyn": "Brooklyn", "bk": "Brooklyn", "bklyn": "Brooklyn",
                 "manhattan": "Manhattan", "mn": "Manhattan", "queens": "Queens", "qns": "Queens",
                 "staten island": "Staten Island", "staten": "Staten Island", "si": "Staten Island"}
HOUSING_WORDS = {"new construction": "New Construction", "new-construction": "New Construction", "new buildings": "New Construction",
                 "new building": "New Construction", "new build": "New Construction", "new builds": "New Construction",
                 "newly built": "New Construction", "new homes": "New Construction", "preservation": "Preservation",
                 "preserved": "Preservation", "rehabilitation": "Preservation", "rehab": "Preservation",
                 "all housing": "All", "all construction": "All", "all types": "All"}
MEASURE_WORDS = {"fatalities": "fatalities", "fatality": "fatalities", "fatal": "fatalities", "deaths": "fatalities", "killed": "fatalities",
                 "severe injuries": "severe_injuries", "severe": "severe_injuries", "injuries": "severe_injuries",
                 "crashes": "crashes", "crash": "crashes", "collisions": "crashes"}
SCOPE_WORDS = {"including highways": "all", "incl highways": "all", "all roads": "all", "highways": "all", "highway": "all",
               "local streets": "local", "local": "local"}
STOP_WORDS = {"in", "and", "the", "of", "for", "show", "me", "with", "units", "unit", "housing", "affordable", "nyc", "year", "years",
              "data", "on", "to", "from", "report", "areas", "area", "&", ","}


def parse_search(text):
    """Turn a search such as 'Brooklyn 2023 new construction fatal' into filter values.

    Returns {'filters': {...only the filters that were mentioned...}, 'recognized': [...], 'ignored': [...]}.
    Longer phrases are matched first. Years 2022-2025 and ranges such as 2022-2024 are recognised;
    other years (for example 2026, which is not complete) are listed as ignored instead of being used.
    """
    remaining = f" {str(text or '').lower()} "
    filters, recognized = {}, []

    def take(phrases, key, multi=False):
        # Look for each phrase as a whole word; when found, save the filter value and remove the phrase from the text.
        nonlocal remaining
        for phrase in sorted(phrases, key=len, reverse=True):
            pattern = rf"(?<![a-z0-9]){re.escape(phrase)}(?![a-z0-9])"
            if re.search(pattern, remaining):
                value = phrases[phrase]
                if multi:
                    filters.setdefault(key, [])
                    if value not in filters[key]:
                        filters[key].append(value)
                elif key not in filters:
                    filters[key] = value
                recognized.append(phrase)
                remaining = re.sub(pattern, " ", remaining)

    # Years: first ranges such as "2022-2024", then single years.
    for a, b in re.findall(r"(20\d\d)\s*[-–to]+\s*(20\d\d)", remaining):
        span = [y for y in range(int(a), int(b) + 1) if y in YEARS]
        if span:
            filters["years"] = sorted(set(filters.get("years", []) + span))
            recognized.append(f"{a}-{b}")
    remaining = re.sub(r"(20\d\d)\s*[-–to]+\s*(20\d\d)", " ", remaining)
    outside_window = []
    for y in re.findall(r"(?<!\d)(20\d\d)(?!\d)", remaining):
        if int(y) in YEARS:
            filters["years"] = sorted(set(filters.get("years", []) + [int(y)]))
            recognized.append(y)
        else:
            outside_window.append(y)          # e.g. 2026: shown back to the user, not used
        remaining = re.sub(rf"(?<!\d){y}(?!\d)", " ", remaining)
    if re.search(r"(?<![a-z])all years(?![a-z])", remaining):
        filters["years"] = list(YEARS)
        recognized.append("all years")
        remaining = re.sub(r"all years", " ", remaining)
    # Then the keyword lists: boroughs, construction type, crash measure and road type.
    take(BOROUGH_WORDS, "boroughs", multi=True)
    take(HOUSING_WORDS, "housing_type")
    take(MEASURE_WORDS, "crash_measure")
    take(SCOPE_WORDS, "crash_scope")
    if re.search(r"(?<![a-z])(citywide|all boroughs)(?![a-z])", remaining):
        filters["boroughs"] = list(BOROUGHS)
        recognized.append("citywide")
        remaining = re.sub(r"(citywide|all boroughs)", " ", remaining)
    # Anything left (except small words such as "in" or "and") is reported back as not recognised.
    ignored = outside_window + [w for w in re.findall(r"[a-z0-9\-]+", remaining) if w not in STOP_WORDS]
    return {"filters": filters, "recognized": recognized, "ignored": ignored}


def merge_filters(dropdowns, parsed):
    """Words from the search replace the matching dropdown values; all other dropdowns stay as they are.
    An empty dropdown falls back to all boroughs or all years."""
    merged = {**DEFAULT_FILTERS, **{k: v for k, v in dropdowns.items() if v}}
    merged.update(parsed.get("filters", {}))
    merged["boroughs"] = [b for b in BOROUGHS if b in (merged.get("boroughs") or BOROUGHS)]
    merged["years"] = sorted(int(y) for y in (merged.get("years") or YEARS))
    return merged


def coverage_by_year(coverage, years, boroughs):
    """Share of crashes that could be placed in a ZIP area, per year, for the selected boroughs.
    Crashes with an unknown borough only belong to the citywide view, so they are included
    only when all five boroughs are selected."""
    selected = list(boroughs) + (["Unknown"] if set(BOROUGHS) <= set(boroughs) else [])
    c = coverage[coverage["year"].isin(years) & coverage["borough"].isin(selected)]
    out = c.groupby("year", as_index=False)[["crashes_all", "crashes_assigned"]].sum()
    out["assignment_rate_pct"] = 100 * out["crashes_assigned"] / out["crashes_all"].where(out["crashes_all"] > 0)
    return out


def select_buildings(buildings, filters):
    """Completed buildings for the RQ2 map that match the selected years, boroughs and construction type."""
    b = buildings[buildings["completion_year"].astype(int).isin(filters["years"]) & buildings["area_borough"].isin(filters["boroughs"])]
    if filters["housing_type"] != "All":
        b = b[b["construction_type"].eq(filters["housing_type"])]
    return b


def build_report(data, filters):
    """Calculate everything the website shows for one set of filters (the notebook uses the default filters).
    Returns a dictionary of tables; "empty" is True when no area matches."""
    f = merge_filters(filters, {})
    area = compute_area_metrics(data["panel"], f["years"], f["boroughs"], f["housing_type"], f["crash_measure"], f["crash_scope"])
    report = {"filters": f, "area": area, "empty": area.empty or area["population_2018"].notna().sum() == 0}
    if report["empty"]:
        return report
    area = add_production_tier(area)
    area["quadrant"], report["crash_median"] = classify_quadrants(area)
    area["dominant_type"], area["nc_share"] = classify_dominant_type(area)
    tier_by_area = area.set_index("modzcta")["production_tier"]
    panel_sel = data["panel"][data["panel"]["year"].isin(f["years"]) & data["panel"]["area_borough"].isin(f["boroughs"])]
    crash_col = f"{f['crash_measure']}_{f['crash_scope']}"
    report["trend"] = indexed_trend(panel_sel, tier_by_area, crash_col)
    report["coverage"] = coverage_by_year(data["coverage"], f["years"], f["boroughs"])
    # RQ4 uses crashes with a known time; 00:00 is left out because it usually means "time unknown".
    cr = data["crashes"]
    cr = cr[cr["crash_year"].isin(f["years"]) & cr["area_borough"].isin(f["boroughs"]) & cr["crash_hour"].notna()
            & ~cr["time_exact_midnight"].astype(bool)]
    if f["crash_scope"] == "local":
        cr = cr[~cr["is_highway"].astype(bool)]
    upper = set(area.loc[area["production_tier"].eq("Upper half"), "modzcta"])
    report["time_cells"] = time_block_shares(cr, upper) if len(cr) else None
    report["types"] = type_summary(area)
    report["bands"], report["resident_avg"] = weighted_band_rates(area)
    report["lookback"] = lookback_shares(area)
    report["priority"] = priority_ranking(area, data["pipeline"], f["housing_type"])
    report["buildings"] = select_buildings(data["buildings"], f) if "buildings" in data else None
    rho, n = spearman(area["units_per_1k"], area["crash_rate"])
    report.update({"area": area, "rho": rho, "n_corr": n})
    report["kpis"] = {"areas": int(area["population_2018"].notna().sum()), "units": float(area["units"].sum()),
                      "crash_total": float(area["crash_total"].sum()), "rho": rho}
    return report


def findings_text(report, measure_label, housing_label):
    """A few short sentences about the current report. They describe patterns and never claim a cause."""
    if report.get("empty"):
        return ["No ZIP areas match this selection."]
    out = []
    rho, n = report["rho"], report["n_corr"]
    if pd.isna(rho):
        out.append(f"Only {n} areas are selected, which is too few to describe an association reliably.")
    else:
        out.append(f"Across {n} ZIP areas, {housing_label.lower()} affordable units completed per resident and "
                   f"{measure_label.lower()} per 100,000 residents per year show a {describe_strength(rho)} (Spearman ρ = {rho:.2f}). "
                   "This describes areas, not individual residents, and does not show cause and effect.")
    q = report["area"]["quadrant"].value_counts()
    out.append(f"{int(q.get(QUADRANTS[0], 0))} areas combine upper-half affordable production with an at-or-above-median crash rate.")
    t = report["types"]["median_crash_rate"]
    if pd.notna(t.get("New-construction dominant")) and pd.notna(t.get("Preservation dominant")):
        out.append(f"Median crash rate: {t['New-construction dominant']:.1f} in new-construction-dominant areas vs "
                   f"{t['Preservation dominant']:.1f} in preservation-dominant areas (a descriptive difference).")
    top = report["priority"]
    if len(top):
        out.append("Highest-scoring candidate areas for further investigation: " + ", ".join(top["modzcta"].head(3)) + ".")
    return out


# Reading the files saved by the notebook (used by the website and by the check in notebook Section 13.3)

APP_FILES = {"panel": "panel_modzcta_year.csv", "crashes": "crashes_for_app.csv", "pipeline": "area_pipeline.csv",
             "coverage": "borough_year_summary.csv", "buildings": "buildings_for_app.csv",
             "geojson": "modzcta_simplified.geojson", "meta": "metadata.json"}


def missing_app_files(data_dir):
    """Names of the required data files that are missing from data_dir."""
    return [name for name in APP_FILES.values() if not (Path(data_dir) / name).exists()]


def load_app_data(data_dir):
    """Read the processed files once. ZIP-area codes are read as text, so "10001" stays "10001"."""
    d = Path(data_dir)
    as_text = {"modzcta": str}
    return {"panel": pd.read_csv(d / APP_FILES["panel"], dtype=as_text),
            "crashes": pd.read_csv(d / APP_FILES["crashes"], dtype=as_text),
            "pipeline": pd.read_csv(d / APP_FILES["pipeline"], dtype=as_text),
            "coverage": pd.read_csv(d / APP_FILES["coverage"]),
            "buildings": pd.read_csv(d / APP_FILES["buildings"], dtype=as_text),
            "geojson": json.loads((d / APP_FILES["geojson"]).read_text()),
            "meta": json.loads((d / APP_FILES["meta"]).read_text())}

In [65]:
"""The eight research-question charts, used by both the notebook and the Dash website.

Each function takes tables from src/analysis.py and returns a Plotly figure.
If a selection leaves nothing to show, the function returns an empty chart with a short message.
"""
import textwrap

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# In the repository this file imports BAND_LABELS, BLOCKS, CRASH_MEASURES, DAYS, QUADRANTS, TIER_ORDER, TYPE_ORDER and
# describe_strength from src/analysis.py; in the notebook they are defined in the cell above.

# Chart colours. They are easy to tell apart, also for colour-blind readers.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
NEUTRAL, INK, INK_SOFT, SURFACE, GRID = "#b8b6ae", "#0b0b0b", "#52514e", "#fcfcfb", "#e7e6e2"
TIER_COLORS = {"No completions": NEUTRAL, "Lower half": SERIES[0], "Upper half": SERIES[1]}
QUADRANT_COLORS = dict(zip(QUADRANTS, [SERIES[7], SERIES[0], SERIES[1], "#e7e6e2"]))
DIVERGING = [[0.0, "#2a78d6"], [0.5, "#f0efec"], [1.0, "#e34948"]]
RATE_WORDING = {"Serious crashes": "serious-crash rate", "Severe injuries": "severe-injury rate", "Fatalities": "fatality rate"}
HOUSING_WORDING = {"All": "new construction + preservation", "New Construction": "new construction only", "Preservation": "preservation only"}


def style(fig, title, subtitle=None, height=440):
    """Common look for all charts: a bold title, a grey subtitle (split into lines so it fits), light background."""
    title_lines = textwrap.wrap(title, 72, break_on_hyphens=False)
    sub_lines = textwrap.wrap(subtitle, 100, break_on_hyphens=False) if subtitle else []
    text = "<br>".join(f"<b>{line}</b>" for line in title_lines)
    if sub_lines:
        text += "<br>" + "<br>".join(f"<span style='font-size:12px;color:{INK_SOFT}'>{line}</span>" for line in sub_lines)
    top = 36 + 19 * len(title_lines) + 16 * len(sub_lines)
    fig.update_layout(title=dict(text=text, x=0, xanchor="left", xref="container", y=1, yanchor="top", yref="container",
                                 pad=dict(t=14, l=12), font=dict(size=15, color=INK)),
                      height=height + top - 78, template="plotly_white",
                      font=dict(family="Arial, Helvetica, sans-serif", size=12, color=INK),
                      paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, margin=dict(l=60, r=24, t=top + 8, b=56),
                      legend=dict(bgcolor="rgba(0,0,0,0)"))
    fig.update_xaxes(gridcolor=GRID, zerolinecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID)
    return fig


def empty_figure(title, message="No data for this selection"):
    """A blank chart with a message, used when the selection has nothing to show."""
    fig = go.Figure()
    fig.add_annotation(text=message, x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False, font=dict(size=14, color=INK_SOFT))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return style(fig, title, height=320)


def rate_words(measure_label):
    """Words for the selected measure, e.g. "serious-crash rate" or "fatality rate"."""
    return RATE_WORDING.get(measure_label, f"{measure_label.lower()} rate")


def fig_rq1(area, rho, n, measure_label="Serious crashes", housing_label="All"):
    """RQ1 scatter plot: units completed per 1,000 residents (x, log scale) against the crash rate (y), one bubble per area."""
    title = f"RQ1 · Affordable units completed per resident vs {rate_words(measure_label)}"
    d = area.dropna(subset=["crash_rate", "production_tier"])
    if d.empty:
        return empty_figure(title)
    pos = d.loc[d["units_per_1k"] > 0, "units_per_1k"]
    lo = int(np.floor(np.log10(pos.min()))) if len(pos) else -1
    hi = int(np.ceil(np.log10(pos.max()))) if len(pos) else 1
    # A log axis cannot show 0, so areas with no completed units are drawn just left of the
    # smallest value, and that position is labelled "0" on the axis.
    zero_x = 10 ** (lo - 0.5)
    d = d.assign(x_plot=d["units_per_1k"].where(d["units_per_1k"] > 0, zero_x), population_2018=d["population_2018"].astype(float))
    fig = px.scatter(d, x="x_plot", y="crash_rate", color="production_tier", size="population_2018", size_max=20, log_x=True,
                     category_orders={"production_tier": TIER_ORDER}, color_discrete_map=TIER_COLORS,
                     custom_data=["modzcta", "area_borough", "units", "crash_total", "population_2018", "units_per_1k"],
                     labels={"x_plot": f"Units completed per 1,000 residents ({HOUSING_WORDING.get(housing_label, housing_label)}, log)",
                             "crash_rate": f"{measure_label} per 100,000 residents per year", "production_tier": "Production tier"})
    fig.update_traces(marker=dict(line=dict(width=1, color=SURFACE), opacity=0.85),
                      hovertemplate="<b>ZIP area %{customdata[0]}</b> (%{customdata[1]})<br>Units completed: %{customdata[2]:,.0f}"
                                    "<br>Units per 1,000 residents: %{customdata[5]:.2f}<br>" + measure_label +
                                    ": %{customdata[3]:,.0f}<br>Rate: %{y:.1f} per 100k per year<br>Population (2018): %{customdata[4]:,.0f}<extra></extra>")
    ticks = [zero_x] + [10 ** k for k in range(lo, hi + 1)]
    fig.update_xaxes(tickvals=ticks, ticktext=["0"] + [f"{v:g}" for v in ticks[1:]])
    sub = (f"Spearman ρ = {rho:.2f} across {n} areas ({describe_strength(rho)}). Bubble size = 2018 population; "
           "areas with no completed units are shown at '0'." if pd.notna(rho) else f"Too few areas ({n}) to measure an association.")
    return style(fig, title, sub, height=480)


def map_view(geojson, codes, width_px=600, height_px=470):
    """Centre and zoom of the map, so the selected areas fill it (one borough is shown close up)."""
    lons, lats = [], []
    for feature in geojson["features"]:
        if feature["properties"].get("modzcta") in codes:
            geom = feature["geometry"]
            polygons = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
            points = np.array([pt for polygon in polygons for ring in polygon for pt in ring], dtype=float)
            lons += [points[:, 0].min(), points[:, 0].max()]
            lats += [points[:, 1].min(), points[:, 1].max()]
    if not lons:
        return {"lat": 40.705, "lon": -73.94}, 9.0
    lon_span, lat_span = max(lons) - min(lons), max(lats) - min(lats)
    mid_lat = (max(lats) + min(lats)) / 2
    zoom_lon = np.log2(width_px * 360 / (512 * max(lon_span, 0.01)))
    zoom_lat = np.log2(height_px * 360 * np.cos(np.radians(mid_lat)) / (512 * max(lat_span, 0.01)))
    return {"lat": mid_lat, "lon": (max(lons) + min(lons)) / 2}, float(min(zoom_lon, zoom_lat) - 0.2)


def fig_rq2(area, crash_median, geojson, buildings=None, measure_label="Serious crashes"):
    """RQ2 map: areas coloured by production tier and crash rate, with completed buildings drawn on top."""
    title = f"RQ2 · Where high affordable production meets a high {rate_words(measure_label)}"
    d = area.dropna(subset=["quadrant"])
    if d.empty:
        return empty_figure(title)
    d = d.assign(units_per_1k_r=d["units_per_1k"].round(2), crash_rate_r=d["crash_rate"].round(1))
    center, zoom = map_view(geojson, set(d["modzcta"]))
    # The map has a plain background instead of map tiles, so it works without an outside map service.
    common = dict(geojson=geojson, locations="modzcta", featureidkey="properties.modzcta", color="quadrant",
                  category_orders={"quadrant": QUADRANTS}, color_discrete_map=QUADRANT_COLORS, hover_name="modzcta",
                  hover_data={"area_borough": True, "units": ":,.0f", "units_per_1k_r": True, "crash_rate_r": True,
                              "quadrant": False, "modzcta": False},
                  labels={"quadrant": "Area type", "area_borough": "Borough", "units": "Units completed",
                          "units_per_1k_r": "Units per 1,000 residents", "crash_rate_r": f"{measure_label} per 100k per year"},
                  center=center, zoom=zoom, opacity=0.8)
    use_maplibre = hasattr(px, "choropleth_map")          # newer Plotly (5.24+); older versions use the Mapbox version
    fig = (px.choropleth_map(d, map_style="white-bg", **common) if use_maplibre
           else px.choropleth_mapbox(d, mapbox_style="white-bg", **common))
    if buildings is not None and len(buildings):
        trace = go.Scattermap if use_maplibre else go.Scattermapbox
        max_units = float(buildings["units_counted"].max())
        for ctype, colour in [("New Construction", INK), ("Preservation", SERIES[6])]:
            b = buildings[buildings["construction_type"].eq(ctype)]
            if b.empty:
                continue
            size = 3 + 9 * np.sqrt(b["units_counted"].fillna(1).astype(float) / max_units)
            fig.add_trace(trace(lat=b["latitude"].astype(float), lon=b["longitude"].astype(float), mode="markers",
                                name=f"{ctype} buildings", marker=dict(size=size, color=colour, opacity=0.6), hoverinfo="text",
                                text=[f"{ctype}: {int(u):,} units · completed {int(y)}" for u, y in
                                      zip(b["units_counted"].fillna(0), b["completion_year"])]))
    fig.update_traces(marker_line_color="#ffffff", marker_line_width=0.6, selector=dict(type="choroplethmap"))
    fig = style(fig, title, f"High production = upper half of areas with completions; high rate = at or above the median "
                            f"area ({crash_median:.1f} per 100k per year). Scroll to zoom, drag to pan.", height=600)
    fig.update_layout(margin=dict(l=8, r=8, t=fig.layout.margin.t, b=8),
                      legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="left", x=0, title_text=""))
    return fig


def fig_rq3(trend, coverage, measure_label="Serious crashes"):
    """RQ3 line chart: crashes per production tier, indexed to the first year = 100, with the yearly placement rate below."""
    title = f"RQ3 · Change in {measure_label.lower()} by affordable production tier"
    if trend is None or trend.empty or trend["year"].nunique() < 2:
        return empty_figure(title, "Select at least two years to show change over time")
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.1,
                        subplot_titles=(f"{measure_label}, indexed to {int(trend['year'].min())} = 100",
                                        "Share of crashes placed in a ZIP area (%)"))
    for tier in TIER_ORDER:
        d = trend[trend["tier"] == tier]
        if d.empty:
            continue
        fig.add_trace(go.Scatter(x=d["year"], y=d["index"], mode="lines+markers", name=tier, line=dict(width=2, color=TIER_COLORS[tier]),
                                 marker=dict(size=8), customdata=d[["value", "areas"]],
                                 hovertemplate=f"<b>{tier}</b><br>%{{x}}: index %{{y:.1f}}<br>%{{customdata[0]:,.0f}} in %{{customdata[1]}} areas<extra></extra>"),
                      row=1, col=1)
    if coverage is not None and len(coverage):
        fig.add_trace(go.Bar(x=coverage["year"], y=coverage["assignment_rate_pct"], marker_color=NEUTRAL, showlegend=False,
                             hovertemplate="%{x}: %{y:.1f}% of crashes placed in an area<extra></extra>"), row=2, col=1)
    fig.add_hline(y=100, line_dash="dot", line_color=INK_SOFT, row=1, col=1)
    fig.update_xaxes(dtick=1, row=2, col=1, title_text="Year")
    fig.update_yaxes(range=[0, 100], row=2, col=1)
    fig.update_layout(legend_title_text="Production tier")
    return style(fig, title, "Tiers are based on units completed in the same years, so this shows change, not cause", height=500)


def fig_rq4(cells):
    """RQ4 heatmap: difference in crash shares per weekday and time of day (upper-half areas minus other areas)."""
    title = "RQ4 · When serious crashes happen: upper-half production areas minus other areas"
    if cells is None or cells.attrs.get("n_upper", 0) == 0 or cells.attrs.get("n_other", 0) == 0:
        return empty_figure(title, "Not enough crashes with a known time in both groups")
    diff = cells.pivot(index="time_block", columns="weekday", values="diff_pp").loc[BLOCKS, DAYS]
    # Use the same colour range above and below zero, so white always means "no difference".
    zmax = float(np.nanmax(np.abs(diff.to_numpy()))) or 1.0
    fig = px.imshow(diff, color_continuous_scale=DIVERGING, zmin=-zmax, zmax=zmax, aspect="auto", text_auto=".1f",
                    labels=dict(x="Day of week", y="Time of day", color="Difference (pp)"))
    fig.update_traces(hovertemplate="%{x}, %{y}<br>Difference in share: %{z:.2f} percentage points<extra></extra>")
    return style(fig, title, f"Share of each group's crashes in each cell (each group adds up to 100%); {cells.attrs['n_upper']:,} vs "
                             f"{cells.attrs['n_other']:,} crashes. Small differences can be due to chance.", height=470)


def fig_rq5(area, types, measure_label="Serious crashes"):
    """RQ5 box plot: crash rates of the areas in each construction-type group."""
    title = f"RQ5 · {rate_words(measure_label).capitalize()} by the dominant type of affordable production"
    d = area.dropna(subset=["crash_rate"])
    if d.empty:
        return empty_figure(title)
    labels = {k: f"{k}<br>(n = {int(v)})" for k, v in types["areas"].fillna(0).items()}
    d = d.assign(group=d["dominant_type"].map(labels))
    fig = px.box(d, x="group", y="crash_rate", points="all", category_orders={"group": [labels[k] for k in TYPE_ORDER]},
                 color_discrete_sequence=[SERIES[0]], hover_data={"modzcta": True, "area_borough": True, "units": ":,.0f",
                                                                  "nc_share": ":.0%", "group": False},
                 labels={"group": "", "crash_rate": f"{measure_label} per 100,000 residents per year", "nc_share": "New-construction share"})
    fig.update_traces(marker=dict(size=6, opacity=0.6), line=dict(width=2))
    return style(fig, title, "New-construction dominant: > 60% of completed units are new construction; preservation dominant: < 40%")


def fig_rq6(bands, resident_avg, measure_label="Serious crashes", housing_type="All"):
    """RQ6 bar chart: crash rate where each income band's units were completed, with the average resident as a line."""
    title = f"RQ6 · {rate_words(measure_label).capitalize()} in the areas where units of each income band were completed"
    d = bands.dropna(subset=["weighted_crash_rate"])
    if d.empty:
        return empty_figure(title)
    fig = px.bar(d, x="weighted_crash_rate", y="band", orientation="h", color_discrete_sequence=[SERIES[0]], custom_data=["units"],
                 category_orders={"band": list(BAND_LABELS.values())}, text=d["weighted_crash_rate"].round(1),
                 labels={"weighted_crash_rate": f"{measure_label} per 100k residents per year (unit-weighted area average)", "band": ""})
    fig.update_traces(textposition="outside", cliponaxis=False,
                      hovertemplate="<b>%{y}</b><br>%{x:.1f} per 100k per year<br>Units: %{customdata[0]:,}<extra></extra>")
    if pd.notna(resident_avg):
        fig.add_vline(x=resident_avg, line_dash="dash", line_color=INK_SOFT, annotation_text=f"average resident: {resident_avg:.1f}",
                      annotation_position="top")
    note = "" if housing_type == "All" else " Income bands include both construction types."
    return style(fig, title, "Describes where units were completed, not the risk faced by residents." + note)


def fig_rq7(lookback):
    """RQ7 stacked bar chart: share of crashes with / without an earlier similar serious injury, per production tier."""
    title = "RQ7 · Crashes where DOT recorded earlier same-type serious injuries at the location (3-year look-back)"
    d = lookback.dropna(how="all", subset=["earlier_recorded", "none_recorded", "unknown"])
    if d.empty or d[["earlier_recorded", "none_recorded", "unknown"]].sum().sum() == 0:
        return empty_figure(title)
    plot = d[["earlier_recorded", "none_recorded", "unknown"]].rename(
        columns={"earlier_recorded": "Earlier similar serious injury recorded", "none_recorded": "None recorded", "unknown": "Unknown (blank)"})
    share = plot.div(plot.sum(axis=1), axis=0).mul(100)          # each tier adds up to 100%
    long = (share.reset_index(names="tier").melt(id_vars="tier", var_name="status", value_name="share")
            .merge(plot.reset_index(names="tier").melt(id_vars="tier", var_name="status", value_name="crashes"), on=["tier", "status"]))
    fig = px.bar(long, x="share", y="tier", color="status", orientation="h", barmode="stack", custom_data=["crashes"],
                 category_orders={"tier": TIER_ORDER[::-1], "status": list(plot.columns)},
                 color_discrete_map={"Earlier similar serious injury recorded": SERIES[1], "None recorded": SERIES[0], "Unknown (blank)": NEUTRAL},
                 text=[f"{v:.0f}%" if v >= 8 else "" for v in long["share"].fillna(0)],          # no labels in very thin segments
                 labels={"share": "Share of local-street serious crashes (%)", "tier": "Production tier", "status": ""})
    fig.update_traces(hovertemplate="%{y} · %{fullData.name}: %{x:.1f}% (%{customdata[0]:,} crashes)<extra></extra>",
                      marker_line_color=SURFACE, marker_line_width=2, textposition="inside", insidetextanchor="middle", textangle=0)
    return style(fig, title, "DOT's field counts deaths and severe injuries in same-type crashes at the same location in the previous 3 years",
                 height=380)


def fig_rq8(top, measure_label="Serious crashes"):
    """RQ8 ranked bar chart: the top areas with their two percentiles (unfinished units per resident and crash rate)."""
    title = "RQ8 · Candidate areas for further investigation: unfinished affordable projects and crash rates"
    if top is None or top.empty:
        return empty_figure(title, "No areas with unfinished affordable projects in this selection")
    t = top.assign(label=top["priority_rank"].astype(str) + ". " + top["modzcta"] + " · " + top["area_borough"])
    # One row per area and percentile, so the two bars of each area can be drawn side by side.
    long = t.melt(id_vars=["label", "priority_score", "pipeline_selected", "nc_pipeline_share", "crash_rate"],
                  value_vars=["pipeline_pct", "crash_pct"], var_name="component", value_name="percentile")
    long["component"] = long["component"].map({"pipeline_pct": "Unfinished units per resident", "crash_pct": rate_words(measure_label).capitalize()})
    fig = px.bar(long, x="percentile", y="label", color="component", orientation="h", barmode="group",
                 category_orders={"label": t["label"].tolist()}, color_discrete_sequence=SERIES[:2],
                 custom_data=["priority_score", "pipeline_selected", "nc_pipeline_share", "crash_rate"],
                 labels={"percentile": "Percentile among selected ZIP areas", "label": "", "component": ""})
    fig.update_traces(hovertemplate="<b>%{y}</b><br>%{fullData.name}: %{x:.0f}th percentile<br>Priority score %{customdata[0]:.1f}"
                                    "<br>Unfinished units: %{customdata[1]:,.0f} (%{customdata[2]:.0%} new construction overall)"
                                    "<br>Crash rate: %{customdata[3]:.1f} per 100k per year<extra></extra>",
                      marker_line_color=SURFACE, marker_line_width=2)
    fig.update_xaxes(range=[0, 100])
    return style(fig, title, "Score = average of the two percentiles. A list of places to look at more closely, not a prediction", height=560)


def report_figures(report, geojson):
    """The eight charts for one report from build_report (used by the website)."""
    f = report["filters"]
    measure = CRASH_MEASURES[f["crash_measure"]]
    if report.get("empty"):
        return [empty_figure(f"RQ{i}", "No ZIP areas match this selection") for i in range(1, 9)]
    return [fig_rq1(report["area"], report["rho"], report["n_corr"], measure, f["housing_type"]),
            fig_rq2(report["area"], report["crash_median"], geojson, report.get("buildings"), measure),
            fig_rq3(report["trend"], report["coverage"], measure),
            fig_rq4(report["time_cells"]),
            fig_rq5(report["area"], report["types"], measure),
            fig_rq6(report["bands"], report["resident_avg"], measure, f["housing_type"]),
            fig_rq7(report["lookback"]),
            fig_rq8(report["priority"], measure)]

### 10.2 Quick checks of the calculations
Five checks on a tiny hand-made table, where we can work out the right answer by hand. They confirm the formulas before we use them on the real data (the same checks are in `tests/` in the repository).

In [66]:
def toy_panel():
    # Four areas x four years with the same values every year, so every expected result is easy to work out by hand.
    spec = {"A": ("Manhattan", 10000, 10, 10, 2), "B": ("Manhattan", 20000, 0, 0, 1),
            "C": ("Brooklyn", 40000, 4, 0, 4), "D": ("Brooklyn", 5000, 10, 6, 1)}   # borough, population, units/yr, NC units/yr, crashes/yr
    rows = []
    for area_code, (borough, pop, units, nc, crashes) in spec.items():
        for year in YEARS:
            rows.append({"modzcta": area_code, "year": year, "area_borough": borough, "population_2018": pop,
                         "units_completed": units, "units_new_construction": nc, "units_preservation": units - nc,
                         "units_eli": units, "units_vli": 0, "units_li": 0, "units_mod": 0, "units_mid": 0, "units_other_income": 0,
                         "crashes_local": crashes, "crashes_all": crashes + 1, "severe_injuries_local": crashes, "severe_injuries_all": crashes + 1,
                         "fatalities_local": 0, "fatalities_all": 0, "crashes_local_repeat": 0, "crashes_local_first": crashes,
                         "crashes_local_repeat_unknown": 0})
    return pd.DataFrame(rows)

toy = add_production_tier(compute_area_metrics(toy_panel())).set_index("modzcta")
check("calc_rates", np.isclose(toy.loc["A", "units_per_1k"], 4.0) and np.isclose(toy.loc["A", "crash_rate"], 20.0),
      "area A: 40 units / 10,000 residents = 4.0 per 1,000; 8 crashes / 4 years / 10,000 = 20 per 100k per year",
      f"unexpected rates {toy.loc['A', ['units_per_1k', 'crash_rate']].tolist()}", stage="research checks")
check("calc_tiers", toy["production_tier"].to_dict() == {"A": "Lower half", "B": "No completions", "C": "Lower half", "D": "Upper half"},
      "tiers: 0 units -> No completions; above the median of areas with completions -> Upper half",
      f"unexpected tiers {toy['production_tier'].to_dict()}", stage="research checks")
lb = lookback_shares(pd.DataFrame({"production_tier": ["Upper half", "Upper half"], "crashes_local_repeat": [1, 0],
                                   "crashes_local_first": [1, 1], "crashes_local_repeat_unknown": [5, 0]}))
check("calc_lookback_unknown", np.isclose(lb.loc["Upper half", "share_known_pct"], 100 / 3),
      "unknown look-back values are excluded from the share, not counted as 'none' (1 of 3 known = 33.3%)",
      f"unexpected share {lb.loc['Upper half', 'share_known_pct']}", stage="research checks")
toy_top = priority_ranking(toy.reset_index(), pd.DataFrame({"modzcta": ["A", "C"], "pipeline_buildings": [1, 2], "pipeline_units": [50, 50],
                                                            "pipeline_units_nc": [50, 0], "pipeline_units_pres": [0, 50]}))
check("calc_priority", list(toy_top["modzcta"]) == ["A", "C"]
      and np.allclose(toy_top["priority_score"], (toy_top["pipeline_pct"] + toy_top["crash_pct"]) / 2),
      "priority score = average of the two percentiles; areas without unfinished units are not listed",
      f"unexpected ranking {toy_top[['modzcta', 'priority_score']].to_dict('records')}", stage="research checks")
parsed = parse_search("Brooklyn 2023 new construction")["filters"]
check("calc_search", parsed == {"years": [2023], "boroughs": ["Brooklyn"], "housing_type": "New Construction"},
      "search 'Brooklyn 2023 new construction' -> borough Brooklyn, year 2023, New Construction",
      f"unexpected parse {parsed}", stage="research checks")

[PASS] calc_rates: area A: 40 units / 10,000 residents = 4.0 per 1,000; 8 crashes / 4 years / 10,000 = 20 per 100k per year
[PASS] calc_tiers: tiers: 0 units -> No completions; above the median of areas with completions -> Upper half
[PASS] calc_lookback_unknown: unknown look-back values are excluded from the share, not counted as 'none' (1 of 3 known = 33.3%)
[PASS] calc_priority: priority score = average of the two percentiles; areas without unfinished units are not listed
[PASS] calc_search: search 'Brooklyn 2023 new construction' -> borough Brooklyn, year 2023, New Construction


True

### 10.3 Tables used by the research questions and the website
The website reads the same tables that are built here: the integrated **panel**, the **crashes** placed in an area (for the crash times in RQ4), the **unfinished projects** per area (RQ8) and the **borough-year coverage** (RQ3). `build_report` then calculates every research-question result for one set of filters. The notebook uses the default filters: all boroughs, 2022–2025, all construction types, serious crashes on local streets.

In [67]:
# Small tables for the website: crashes with their hour and weekday (RQ4) and completed buildings with coordinates (RQ2 map).
crashes_app = t.loc[t["crash_year"].isin(ANALYSIS_YEARS) & t["modzcta"].notna(),
                    ["modzcta", "area_borough", "crash_year", "crash_hour", "weekday", "is_highway", "time_exact_midnight"]].copy()
buildings_app = h.loc[h["completion_status"].eq("completed") & h["completion_year"].isin(ANALYSIS_YEARS) & h["modzcta"].notna()
                      & h["has_coords"],
                      ["modzcta", "area_borough", "construction_type", "completion_year", "units_counted", "latitude", "longitude"]].copy()
mz_simple = mz[["modzcta", "geometry"]].copy()
mz_simple["geometry"] = mz_simple.geometry.simplify(0.0002, preserve_topology=True)      # lighter boundaries for maps
GEOJSON = json.loads(mz_simple.to_json())

# Calculate all research-question results with the default filters.
APP_DATA = {"panel": panel, "crashes": crashes_app, "pipeline": area_pipeline, "coverage": borough_year, "buildings": buildings_app}
REPORT = build_report(APP_DATA, DEFAULT_FILTERS)
AREA = REPORT["area"]
RESULTS = {}
check("report_not_empty", not REPORT["empty"], f"default report covers {REPORT['kpis']['areas']} areas", "the default report is empty",
      stage="research checks")
check("report_area_totals", np.isclose(AREA["units"].sum(), panel["units_completed"].sum())
      and np.isclose(AREA["crash_total"].sum(), panel["crashes_local"].sum()),
      "area totals equal the panel totals (no units or crashes lost in the calculations)", "area totals differ from the panel",
      stage="research checks")
band_cols = list(BAND_LABELS)
check("report_band_totals", (panel[band_cols].sum(axis=1) == panel["units_income_known"]).all(),
      "income-band units add up to the units with a known income breakdown in every area-year",
      "income-band units do not add up to units_income_known", stage="research checks")

def fmt(value, spec=".1f", missing="n/a"):
    # Format a number for the written results. A missing value is shown as "n/a", never as 0.
    return missing if value is None or pd.isna(value) else format(value, spec)

# How many areas are in each production tier, and their median values.
tier_table = (AREA.groupby("production_tier")
              .agg(areas=("modzcta", "size"), units=("units", "sum"), new_construction=("units_new_construction", "sum"),
                   preservation=("units_preservation", "sum"), median_units_per_1k=("units_per_1k", "median"),
                   median_crash_rate=("crash_rate", "median")).reindex(TIER_ORDER))
display(tier_table.round(2))
print(f"Tier split: median of {fmt(AREA.attrs['tier_median'], '.2f')} units per 1,000 residents among areas with completions")

[PASS] report_not_empty: default report covers 177 areas
[PASS] report_area_totals: area totals equal the panel totals (no units or crashes lost in the calculations)
[PASS] report_band_totals: income-band units add up to the units with a known income breakdown in every area-year


,areas,units,new_construction,preservation,median_units_per_1k,median_crash_rate
production_tier,,,,,,
No completions,47,0.0,0,0,0.00,21.87
Lower half,65,8609.0,4591,4018,1.66,26.79
Upper half,65,91448.0,43314,48134,17.80,32.66


Tier split: median of 6.82 units per 1,000 residents among areas with completions


### Section A — Spatial relationship
#### RQ1 — Is affordable housing production associated with serious traffic-crash rates across NYC areas?
**Measure.** For each area: affordable units completed in 2022–2025 per 1,000 residents (x-axis) and serious crashes per 100,000 residents per year (y-axis). Both are divided by population, so large areas do not stand out just because they are large. The **Spearman rank correlation** (ρ, from −1 to +1) tells us whether areas that rank higher on one measure also tend to rank higher on the other. We use ranks because a few areas have very large values.

In [68]:
fig_rq1(AREA, REPORT["rho"], REPORT["n_corr"]).show()
# The same correlation for all units, for new construction only and for preservation only.
rq1_by_type = []
for housing_type in HOUSING_TYPES:
    a = compute_area_metrics(panel, housing_type=housing_type)
    rho, n = spearman(a["units_per_1k"], a["crash_rate"])
    rq1_by_type.append({"construction type": housing_type, "areas": n, "spearman_rho": round(rho, 3), "description": describe_strength(rho)})
rq1_by_type = pd.DataFrame(rq1_by_type)
display(rq1_by_type)
RESULTS["rq1"] = {"rho": REPORT["rho"], "areas": REPORT["n_corr"], "by_type": rq1_by_type.to_dict("records"),
                  "median_crash_rate_by_tier": tier_table["median_crash_rate"].to_dict()}
display(Markdown(
    f"**Result.** Across {REPORT['n_corr']} ZIP areas, affordable units completed per resident and the serious-crash rate show a "
    f"**{describe_strength(REPORT['rho'])}** (Spearman ρ = {fmt(REPORT['rho'], '.2f')}). New construction alone: ρ = "
    f"{fmt(rq1_by_type.loc[1, 'spearman_rho'], '.2f')}; preservation alone: ρ = {fmt(rq1_by_type.loc[2, 'spearman_rho'], '.2f')}. "
    f"Median crash rate by tier: " + ", ".join(f"{k} {fmt(v)}" for k, v in tier_table["median_crash_rate"].items()) + " per 100,000 residents per year."))

,construction type,areas,spearman_rho,description
0,All,177,0.330,moderate positive association
1,New Construction,177,0.421,moderate positive association
2,Preservation,177,0.099,negligible positive association


**Result.** Across 177 ZIP areas, affordable units completed per resident and the serious-crash rate show a **moderate positive association** (Spearman ρ = 0.33). New construction alone: ρ = 0.42; preservation alone: ρ = 0.10. Median crash rate by tier: No completions 21.9, Lower half 26.8, Upper half 32.7 per 100,000 residents per year.

**How to read it.** Each bubble is a ZIP area (bubble size = population). A positive ρ means that areas which completed more affordable units per resident *tend* to have higher serious-crash rates — for example because both are common in dense, busy parts of the city. It does not mean that the housing caused the crashes, and it says nothing about the risk for the people living in particular buildings. Both measures are divided by the same 2018 population, so areas with few residents but heavy traffic (such as Midtown) can have high values on both axes; hover over a bubble to see the actual counts.

#### RQ2 — Which areas have both relatively high affordable housing production and relatively high serious-crash rates?
*Relatively high production* = upper production tier; *relatively high crash rate* = at or above the median area. Completed buildings are drawn on top of the areas, coloured by construction type, so both datasets appear on the same map.

In [69]:
fig_rq2(AREA, REPORT["crash_median"], GEOJSON, REPORT["buildings"]).show()
# Count the areas in each group and list the areas with both high production and a high crash rate.
quadrant_counts = AREA["quadrant"].value_counts().reindex(QUADRANTS).fillna(0).astype(int)
display(quadrant_counts.to_frame("areas"))
high_high = AREA[AREA["quadrant"].eq(QUADRANTS[0])].sort_values("crash_rate", ascending=False)
display(high_high[["modzcta", "area_borough", "units", "units_new_construction", "units_preservation", "units_per_1k", "crash_rate"]]
        .round(2).head(15))
hh_nc_share = 100 * high_high["units_new_construction"].sum() / high_high["units"].sum() if high_high["units"].sum() > 0 else np.nan
RESULTS["rq2"] = {"high_high_areas": len(high_high), "by_borough": high_high["area_borough"].value_counts().to_dict(),
                  "crash_rate_median": REPORT["crash_median"], "new_construction_share_pct": hh_nc_share}
display(Markdown(
    f"**Result.** {len(high_high)} areas combine upper-tier affordable production with a serious-crash rate at or above the median "
    f"({fmt(REPORT['crash_median'])} per 100,000 residents per year). By borough: "
    + (", ".join(f"{b} {n}" for b, n in high_high["area_borough"].value_counts().items()) or "none")
    + f". New construction makes up {fmt(hh_nc_share, '.0f')}% of the units completed in these areas; the rest is preservation."))

,areas
quadrant,
High production · high crash rate,44
High production · lower crash rate,21
Lower/no production · high crash rate,45
Lower/no production · lower crash rate,67


,modzcta,area_borough,units,units_new_construction,units_preservation,units_per_1k,crash_rate
15,10018,Manhattan,177.0,177,0,18.27,147.10
82,11101,Queens,2451.0,2451,0,81.58,114.00
79,10474,Bronx,291.0,243,48,24.00,105.14
0,10001,Manhattan,618.0,618,0,26.79,94.27
32,10036,Manhattan,996.0,324,672,36.31,84.77
60,10455,Bronx,1135.0,955,180,26.84,63.86
59,10454,Bronx,1101.0,921,180,28.82,61.51
31,10035,Manhattan,780.0,572,208,21.82,52.46
56,10451,Bronx,3782.0,1799,1983,79.12,50.73
75,10470,Bronx,194.0,194,0,12.57,46.98


**Result.** 44 areas combine upper-tier affordable production with a serious-crash rate at or above the median (26.7 per 100,000 residents per year). By borough: Brooklyn 15, Bronx 14, Manhattan 11, Queens 3, Staten Island 1. New construction makes up 65% of the units completed in these areas; the rest is preservation.

**How to read it.** Red areas combine relatively high production with a relatively high crash rate — places where housing and street-safety planning overlap. Splitting at the median is a simplification: an area just below the median is not very different from one just above it, so hover over an area to see its actual values.

### Section B — Time patterns
#### RQ3 — How did serious crashes change between 2022 and 2025 across areas with different levels of affordable housing production?
Each line shows the serious crashes of one production tier, indexed to 2022 = 100, so tiers of different sizes can be compared. **This describes change; it does not show an effect.** The tiers are based on units completed in the *same* years, so an upper-tier area may have completed most of its units in 2025, after the 2022 crashes it is compared with. The lower panel shows the share of crashes that could be placed in an area each year, because better location data can move all the lines up together.

In [70]:
# Crashes per tier and year, and the percentage change from the first to the last year.
trend = REPORT["trend"]
fig_rq3(trend, REPORT["coverage"]).show()
first, last = ANALYSIS_YEARS[0], ANALYSIS_YEARS[-1]
TIER_WORDING = {"No completions": "areas with no completions", "Lower half": "lower-half areas", "Upper half": "upper-half areas"}
rq3_table = trend.pivot(index="tier", columns="year", values="value").reindex(TIER_ORDER)
rq3_table[f"change_{first}_{last}_pct"] = (100 * (rq3_table[last] / rq3_table[first].replace(0, np.nan) - 1)).round(1)
display(rq3_table)
display(REPORT["coverage"].round(1))
RESULTS["rq3"] = {"change_pct": rq3_table[f"change_{first}_{last}_pct"].to_dict(),
                  "citywide_dated_change_pct": 100 * change_dated, "citywide_assigned_change_pct": 100 * change_assigned}
display(Markdown(
    f"**Result.** From {first} to {last}, serious crashes on local streets changed by "
    + ", ".join(f"{fmt(v, '+.1f')}% in {TIER_WORDING[k]}" for k, v in rq3_table[f"change_{first}_{last}_pct"].items())
    + f". Citywide, all dated serious crashes changed by {fmt(100 * change_dated, '+.1f')}% and crashes placed in an area by "
    f"{fmt(100 * change_assigned, '+.1f')}%, so part of any change can come from location coverage. Four yearly values describe change; "
    "they are not a trend and do not show that housing production raised or lowered crashes."))

year,2022,2023,2024,2025,change_2022_2025_pct
tier,,,,,
No completions,255.0,277.0,279.0,310.0,21.6
Lower half,930.0,1029.0,1049.0,1091.0,17.3
Upper half,1158.0,1214.0,1174.0,1133.0,-2.2


,year,crashes_all,crashes_assigned,assignment_rate_pct
0,2022,2879,2557,88.8
1,2023,3016,2696,89.4
2,2024,3116,2729,87.6
3,2025,2947,2795,94.8


**Result.** From 2022 to 2025, serious crashes on local streets changed by +21.6% in areas with no completions, +17.3% in lower-half areas, -2.2% in upper-half areas. Citywide, all dated serious crashes changed by +2.4% and crashes placed in an area by +9.3%, so part of any change can come from location coverage. Four yearly values describe change; they are not a trend and do not show that housing production raised or lowered crashes.

#### RQ4 — How do crash times and weekdays differ between areas with different housing-production levels?
Every crash falls into one weekday × 3-hour cell. For each group — upper-tier production areas and all other areas — the shares of all cells add up to 100%. The heatmap shows the **difference in share** (upper tier minus other areas) in percentage points: a red cell holds a larger share of the upper-tier areas' crashes. Crashes recorded at exactly 00:00 (probably an unknown time) and highway crashes are left out.

In [71]:
# The heatmap, the three largest differences in each direction, and the weekend and night shares of each group.
cells = REPORT["time_cells"]
fig_rq4(cells).show()
largest = pd.concat([cells.nlargest(3, "diff_pp"), cells.nsmallest(3, "diff_pp")])[["weekday", "time_block", "share_upper_pct", "share_other_pct", "diff_pp"]]
display(largest.round(2))

def group_share(mask):
    # Share (%) of each group's crashes that fall in the chosen cells.
    return 100 * cells.loc[mask, "count_upper"].sum() / cells.attrs["n_upper"], 100 * cells.loc[mask, "count_other"].sum() / cells.attrs["n_other"]

weekend_upper, weekend_other = group_share(cells["weekday"].isin(["Saturday", "Sunday"]))
night_upper, night_other = group_share(cells["time_block"].isin(["21–24", "00–03", "03–06"]))
RESULTS["rq4"] = {"crashes_upper": cells.attrs["n_upper"], "crashes_other": cells.attrs["n_other"],
                  "weekend_share_pct": [weekend_upper, weekend_other], "night_share_pct": [night_upper, night_other],
                  "largest_cell_difference_pp": float(cells["diff_pp"].abs().max())}
display(Markdown(
    f"**Result.** {cells.attrs['n_upper']:,} crashes in upper-tier areas and {cells.attrs['n_other']:,} in other areas have a usable time. "
    f"Weekend share: {fmt(weekend_upper)}% vs {fmt(weekend_other)}%; night share (21:00–06:00): {fmt(night_upper)}% vs {fmt(night_other)}%. "
    f"The largest single-cell difference is {fmt(RESULTS['rq4']['largest_cell_difference_pp'], '.1f')} percentage points. With 56 cells, "
    "differences of one or two points can arise by chance, so this chart is exploratory."))

group,weekday,time_block,share_upper_pct,share_other_pct,diff_pp
25,Friday,09–12,2.00,1.25,0.76
54,Saturday,21–24,2.57,2.00,0.56
48,Sunday,18–21,2.68,2.24,0.44
21,Monday,09–12,1.13,1.87,-0.74
38,Thursday,15–18,2.61,3.31,-0.70
30,Wednesday,12–15,1.89,2.55,-0.66


**Result.** 4,596 crashes in upper-tier areas and 5,140 in other areas have a usable time. Weekend share: 27.9% vs 27.4%; night share (21:00–06:00): 29.4% vs 25.9%. The largest single-cell difference is 0.8 percentage points. With 56 cells, differences of one or two points can arise by chance, so this chart is exploratory.

### Section C — Housing characteristics
#### RQ5 — Do areas dominated by new construction and preservation show different serious-crash patterns?
An area is **new-construction dominant** when more than 60% of its 2022–2025 affordable units are new construction, **preservation dominant** when less than 40% are, and **mixed** otherwise. The box plot shows the spread of crash rates in each group (the line in the box is the median; every dot is an area).

In [72]:
# Median crash rate per group, and the borough mix of each group (boroughs differ in both housing and crashes).
types = REPORT["types"]
fig_rq5(AREA, types).show()
display(types.round(1))
borough_mix = (pd.crosstab(AREA["dominant_type"], AREA["area_borough"], normalize="index") * 100).reindex(TYPE_ORDER).round(0)
display(Markdown("**Borough mix of each group (% of its areas)**")); display(borough_mix)
med = types["median_crash_rate"]
RESULTS["rq5"] = {"median_crash_rate": med.to_dict(), "areas": types["areas"].to_dict()}
display(Markdown(
    f"**Result.** Median serious-crash rate: {fmt(med.get('New-construction dominant'))} in new-construction-dominant areas, "
    f"{fmt(med.get('Mixed'))} in mixed areas, {fmt(med.get('Preservation dominant'))} in preservation-dominant areas and "
    f"{fmt(med.get('No completions'))} in areas with no completions (per 100,000 residents per year). The groups differ in borough mix "
    "(table above), so borough differences may explain part of any gap. This compares areas; it does not show an effect of the construction type."))

,areas,median_crash_rate,q1,q3,units
dominant_type,,,,,
New-construction dominant,78,31.4,24.2,39.4,41852.0
Mixed,16,27.8,23.2,34.2,23639.0
Preservation dominant,36,23.2,19.4,32.3,34566.0
No completions,47,21.9,15.2,26.4,0.0


**Borough mix of each group (% of its areas)**

area_borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island
dominant_type,,,,,
New-construction dominant,15.0,35.0,15.0,31.0,4.0
Mixed,56.0,38.0,0.0,6.0,0.0
Preservation dominant,6.0,6.0,53.0,33.0,3.0
No completions,4.0,4.0,28.0,47.0,17.0


**Result.** Median serious-crash rate: 31.4 in new-construction-dominant areas, 27.8 in mixed areas, 23.2 in preservation-dominant areas and 21.9 in areas with no completions (per 100,000 residents per year). The groups differ in borough mix (table above), so borough differences may explain part of any gap. This compares areas; it does not show an effect of the construction type.

#### RQ6 — Are affordable housing units in different income categories located in areas with different serious-crash rates?
For each income band, the bar shows the crash rate of the area **where the average unit in that band was completed** (each area's crash rate weighted by the number of that band's units there). The dashed line is the rate for the average resident (weighted by population). Only buildings whose income bands add up exactly to their total are used.

**This describes where units were completed, not the risk faced by the people who live in them.**

In [73]:
# Crash rate per income band, compared with the average resident.
bands, resident_avg = REPORT["bands"], REPORT["resident_avg"]
fig_rq6(bands, resident_avg).show()
bands_table = bands.assign(difference_vs_average_resident=bands["weighted_crash_rate"] - resident_avg)
display(bands_table.round(1))
RESULTS["rq6"] = {"band_rates": bands.set_index("band")["weighted_crash_rate"].to_dict(), "resident_avg": resident_avg,
                  "units": bands.set_index("band")["units"].to_dict()}
known = bands.dropna(subset=["weighted_crash_rate"])
hi_band = known.loc[known["weighted_crash_rate"].idxmax()] if len(known) else None
lo_band = known.loc[known["weighted_crash_rate"].idxmin()] if len(known) else None
display(Markdown(
    f"**Result.** The average resident lives in an area with {fmt(resident_avg)} serious crashes per 100,000 residents per year. "
    + (f"The highest band value is {hi_band['band'].lower()} units ({fmt(hi_band['weighted_crash_rate'])}) and the lowest is "
       f"{lo_band['band'].lower()} units ({fmt(lo_band['weighted_crash_rate'])}). " if hi_band is not None else "")
    + "Bands with few units can move a lot when a single large building is added, so small differences should not be over-interpreted."))

,band,units,weighted_crash_rate,difference_vs_average_resident
0,Extremely low income,15451,32.6,3.3
1,Very low income,30111,27.9,-1.4
2,Low income,29098,32.4,3.1
3,Moderate income,5668,34.4,5.1
4,Middle income,19360,42.4,13.1
5,Other income units,369,32.8,3.5


**Result.** The average resident lives in an area with 29.3 serious crashes per 100,000 residents per year. The highest band value is middle income units (42.4) and the lowest is very low income units (27.9). Bands with few units can move a lot when a single large building is added, so small differences should not be over-interpreted.

### Section D — Road-safety priorities
#### RQ7 — Do areas with greater affordable housing production also have higher shares of crashes with reported historical similar serious-injury or fatality events?
**What the SIRTA field means.** DOT describes `similar_fatalities_severe` as the *number of fatalities and severe injuries from crashes of the same type at the same location in the past three years*. So it counts **people hurt in earlier similar crashes**. The column descriptions do not explain exactly how DOT decides "same type" and "same location", so this field does **not** prove that crashes happened at exactly the same spot. A crash is labelled *earlier similar serious injury recorded* when the value is 1 or more, *none recorded* when it is 0, and *unknown* when it is blank. Unknown values are shown separately and are never counted as zero. Only local-street crashes are used.

In [74]:
# Look-back status per production tier; the share uses known values only.
lookback = REPORT["lookback"]
fig_rq7(lookback).show()
display(lookback.round(1))
RESULTS["rq7"] = {"share_known_pct": lookback["share_known_pct"].to_dict(), "unknown": int(lookback["unknown"].sum())}
display(Markdown(
    "**Result.** Share of local serious crashes with an earlier similar serious injury or fatality recorded (among known values): "
    + "; ".join(f"{k} {fmt(v)}%" for k, v in lookback["share_known_pct"].items())
    + f". {int(lookback['unknown'].sum()):,} crashes have a blank look-back value and are excluded from these shares."))

,earlier_recorded,none_recorded,unknown,share_known_pct
production_tier,,,,
No completions,44,940,137,4.5
Lower half,178,3429,492,4.9
Upper half,348,3793,538,8.4


**Result.** Share of local serious crashes with an earlier similar serious injury or fatality recorded (among known values): No completions 4.5%; Lower half 4.9%; Upper half 8.4%. 1,167 crashes have a blank look-back value and are excluded from these shares.

#### RQ8 — Which areas combine unfinished affordable housing projects with relatively high serious-crash rates?
**Unfinished projects** = buildings in projects that have started but that have **no building completion date** in the downloaded data. A missing completion date does not tell us how close a building is to being finished, and preservation work does not necessarily add residents, so the hover text also shows the new-construction share.

**Priority score** = the average of two percentile ranks among the selected areas: unfinished units per 1,000 residents, and the 2022–2025 serious-crash rate. A higher score puts an area higher on the list. **The list only shows where it may be worth looking more closely. It is not a prediction, and it does not prove that a street needs to be redesigned.**

In [75]:
# The 15 highest-scoring areas, and how much of the unfinished housing started before 2022.
top = REPORT["priority"]
fig_rq8(top).show()
display(top[["priority_rank", "modzcta", "area_borough", "pipeline_selected", "nc_pipeline_share", "pipeline_per_1k", "crash_rate",
             "pipeline_pct", "crash_pct", "priority_score"]].round(2))
started_before = pipe_records["start_year"] < ANALYSIS_YEARS[0]
old_share = 100 * pipe_records.loc[started_before, "units_counted"].sum() / pipe_records["units_counted"].sum()
RESULTS["rq8"] = {"top5": top["modzcta"].head(5).tolist(), "boroughs_top15": top["area_borough"].value_counts().to_dict(),
                  "pipeline_units_started_before_window_pct": old_share}
display(Markdown(
    "**Result.** Highest-scoring areas for further investigation: "
    + ", ".join(f"{m} ({b})" for m, b in zip(top["modzcta"].head(5), top["area_borough"].head(5)))
    + f". {fmt(old_share, '.0f')}% of unfinished units are in projects that started before {ANALYSIS_YEARS[0]}, so some of this pipeline may be "
    "long-running or stalled. The list identifies places worth a closer look by housing and transportation planners; it is not a forecast."))

,priority_rank,modzcta,area_borough,pipeline_selected,nc_pipeline_share,pipeline_per_1k,crash_rate,pipeline_pct,crash_pct,priority_score
82,1,11101,Queens,925.0,1.00,30.79,114.00,92.66,99.44,96.05
31,1,10035,Manhattan,1535.0,0.41,42.95,52.46,98.87,93.22,96.05
59,3,10454,Bronx,1304.0,0.00,34.13,61.51,96.05,94.92,95.48
79,4,10474,Bronx,310.0,0.87,25.56,105.14,89.83,98.87,94.35
104,5,11217,Brooklyn,1290.0,0.93,31.15,44.07,93.22,90.40,91.81
75,6,10470,Bronx,455.0,0.72,29.48,46.98,91.53,91.53,91.53
56,7,10451,Bronx,1051.0,0.83,21.99,50.73,88.70,92.66,90.68
93,7,11206,Brooklyn,3479.0,0.21,39.38,39.33,98.31,83.05,90.68
60,9,10455,Bronx,771.0,0.85,18.23,63.86,83.62,96.05,89.83
32,10,10036,Manhattan,413.0,0.05,15.06,84.77,80.23,97.74,88.98


**Result.** Highest-scoring areas for further investigation: 11101 (Queens), 10035 (Manhattan), 10454 (Bronx), 10474 (Bronx), 11217 (Brooklyn). 26% of unfinished units are in projects that started before 2022, so some of this pipeline may be long-running or stalled. The list identifies places worth a closer look by housing and transportation planners; it is not a forecast.

## 11. Findings and data story
The story below is written by the code from the calculated results, so every number updates when the notebook is run again. The website shows the same story for the default selection and updates the findings when the filters change.

In [76]:
# Build the data story from the results above (no number in it is typed by hand).
r = RESULTS
conf_window = h[h["completion_status"].eq("completed") & h["completion_year"].isin(ANALYSIS_YEARS) & h["is_confidential"]]
unplaced_pct = 100 - coverage_year["assignment_rate_pct"]
tier_changes = ", ".join(f"{fmt(v, '+.1f')}% in {TIER_WORDING[k]}" for k, v in r["rq3"]["change_pct"].items())
med5 = r["rq5"]["median_crash_rate"]
band_values = [v for v in r["rq6"]["band_rates"].values() if pd.notna(v)]
band_min, band_max = (min(band_values), max(band_values)) if band_values else (np.nan, np.nan)

# "What surprised us": the approved wording. The numbers come from RQ1 (by construction type) and RQ5.
rho_type = {row["construction type"]: row["spearman_rho"] for row in r["rq1"]["by_type"]}
rho_nc, rho_pres = rho_type.get("New Construction", np.nan), rho_type.get("Preservation", np.nan)
med_nc, med_pres = med5.get("New-construction dominant", np.nan), med5.get("Preservation dominant", np.nan)
SURPRISE = ("One finding that stood out was the difference between new construction and preservation. Areas with more "
            f"new-construction units per resident tended to have higher serious-crash rates (ρ = {fmt(rho_nc, '.2f')}), while "
            f"preservation showed a much weaker relationship (ρ = {fmt(rho_pres, '.2f')}). We also found that new-construction-dominant "
            f"areas had a higher median crash rate ({fmt(med_nc)}) than preservation-dominant areas ({fmt(med_pres)}). This is an "
            "interesting difference, but it does not mean that new construction causes more crashes.")
# The wording assumes this pattern. If a future download changes it, print a reminder to review the paragraph.
if not (rho_nc > rho_pres and med_nc > med_pres):
    print("Note: the latest results no longer show this pattern, so the 'What surprised us' paragraph should be reviewed.")
STORY_MARKDOWN = f'''
### The question
Do New York City areas that produce more affordable housing also have more serious traffic crashes per resident — and how does this differ by place, year, construction type and income group? Affordable homes are built or preserved to house people for decades. If they are concentrated where streets are most dangerous, housing and road-safety programmes could benefit from being planned together.

### The data
* **Housing:** HPD's *Affordable Housing Production by Building* (H2) — {len(h):,} building records, of which {len(conf_window):,} buildings completed in {ANALYSIS_YEARS[0]}–{ANALYSIS_YEARS[-1]} are confidential and cannot be placed on a map.
* **Transportation:** DOT's *Serious Injury Response, Tracking & Analysis Program* (SIRTA) — {len(t):,} serious crashes kept after removing {len(t_excluded):,} unusable or duplicate rows.
* **Geographic reference:** MODZCTA ZIP-code areas (boundaries and 2018 population), used only to place records and compute per-resident rates.
* Snapshot: {SNAPSHOT_DATE} ({SOURCE_MODE}). The partial year 2026 is excluded.

### How the datasets were connected
Both datasets were aggregated to **ZIP-code area × year** ({panel['modzcta'].nunique()} areas × {len(ANALYSIS_YEARS)} years = {len(panel)} rows) and joined one-to-one, so no record was duplicated. Records were placed in areas by their coordinates first and by ZIP code otherwise; {fmt(unplaced_pct.min())}–{fmt(unplaced_pct.max())}% of each year's crashes could not be placed and are kept in borough totals only.

### What we found
* **RQ1 — association:** across {r['rq1']['areas']} areas, units completed per resident and the serious-crash rate show a {describe_strength(r['rq1']['rho'])} (Spearman ρ = {fmt(r['rq1']['rho'], '.2f')}).
* **RQ2 — overlap:** {r['rq2']['high_high_areas']} areas combine relatively high production with a relatively high crash rate (by borough: {', '.join(f'{b} {n}' for b, n in r['rq2']['by_borough'].items()) or 'none'}).
* **RQ3 — change over time:** from {ANALYSIS_YEARS[0]} to {ANALYSIS_YEARS[-1]}, local serious crashes changed by {tier_changes} (citywide: {fmt(r['rq3']['citywide_dated_change_pct'], '+.1f')}%).
* **RQ4 — timing:** weekend share {fmt(r['rq4']['weekend_share_pct'][0])}% in upper-tier areas vs {fmt(r['rq4']['weekend_share_pct'][1])}% elsewhere; night share {fmt(r['rq4']['night_share_pct'][0])}% vs {fmt(r['rq4']['night_share_pct'][1])}% (exploratory).
* **RQ5 — construction type:** median crash rate {fmt(med5.get('New-construction dominant'))} in new-construction-dominant areas vs {fmt(med5.get('Preservation dominant'))} in preservation-dominant areas (a descriptive difference; the groups differ in borough mix).
* **RQ6 — income bands:** band values range from {fmt(band_min)} to {fmt(band_max)} serious crashes per 100,000 residents per year, compared with {fmt(r['rq6']['resident_avg'])} for the average resident. These describe where units were built, not personal risk.
* **RQ7 — earlier similar serious injuries:** shares (known values only): {'; '.join(f'{k} {fmt(v)}%' for k, v in r['rq7']['share_known_pct'].items())}.
* **RQ8 — screening list:** highest-scoring areas for further investigation: {', '.join(r['rq8']['top5']) or 'none'} (not a prediction).

### What surprised us
{SURPRISE}

### What the data cannot tell us
These are **area-level associations**: they do not show that housing production causes crashes, and they say nothing about the risk faced by individual residents. Preservation does not necessarily add residents. Rates use 2018 population estimates, four years are too few for trend claims, some crashes could not be placed in an area, and SIRTA covers serious crashes only.

### Why it matters for New York City
The results can help **DOT** and **HPD** see where affordable housing and serious crashes overlap, so that street-safety reviews can be considered alongside housing projects — especially in the RQ2 overlap areas and the RQ8 candidate areas, where unfinished projects will bring new or renovated homes. Community boards and housing advocates can use the website to look up their own borough, year and housing type. Any action should start with a closer local look, because these figures identify places to investigate, not causes.
'''
display(Markdown(STORY_MARKDOWN))


### The question
Do New York City areas that produce more affordable housing also have more serious traffic crashes per resident — and how does this differ by place, year, construction type and income group? Affordable homes are built or preserved to house people for decades. If they are concentrated where streets are most dangerous, housing and road-safety programmes could benefit from being planned together.

### The data
* **Housing:** HPD's *Affordable Housing Production by Building* (H2) — 9,551 building records, of which 716 buildings completed in 2022–2025 are confidential and cannot be placed on a map.
* **Transportation:** DOT's *Serious Injury Response, Tracking & Analysis Program* (SIRTA) — 13,309 serious crashes kept after removing 445 unusable or duplicate rows.
* **Geographic reference:** MODZCTA ZIP-code areas (boundaries and 2018 population), used only to place records and compute per-resident rates.
* Snapshot: 2026-10-16 (fresh-download). The partial year 2026 is excluded.

### How the datasets were connected
Both datasets were aggregated to **ZIP-code area × year** (177 areas × 4 years = 708 rows) and joined one-to-one, so no record was duplicated. Records were placed in areas by their coordinates first and by ZIP code otherwise; 5.2–12.4% of each year's crashes could not be placed and are kept in borough totals only.

### What we found
* **RQ1 — association:** across 177 areas, units completed per resident and the serious-crash rate show a moderate positive association (Spearman ρ = 0.33).
* **RQ2 — overlap:** 44 areas combine relatively high production with a relatively high crash rate (by borough: Brooklyn 15, Bronx 14, Manhattan 11, Queens 3, Staten Island 1).
* **RQ3 — change over time:** from 2022 to 2025, local serious crashes changed by +21.6% in areas with no completions, +17.3% in lower-half areas, -2.2% in upper-half areas (citywide: +2.4%).
* **RQ4 — timing:** weekend share 27.9% in upper-tier areas vs 27.4% elsewhere; night share 29.4% vs 25.9% (exploratory).
* **RQ5 — construction type:** median crash rate 31.4 in new-construction-dominant areas vs 23.2 in preservation-dominant areas (a descriptive difference; the groups differ in borough mix).
* **RQ6 — income bands:** band values range from 27.9 to 42.4 serious crashes per 100,000 residents per year, compared with 29.3 for the average resident. These describe where units were built, not personal risk.
* **RQ7 — earlier similar serious injuries:** shares (known values only): No completions 4.5%; Lower half 4.9%; Upper half 8.4%.
* **RQ8 — screening list:** highest-scoring areas for further investigation: 11101, 10035, 10454, 10474, 11217 (not a prediction).

### What surprised us
One finding that stood out was the difference between new construction and preservation. Areas with more new-construction units per resident tended to have higher serious-crash rates (ρ = 0.42), while preservation showed a much weaker relationship (ρ = 0.10). We also found that new-construction-dominant areas had a higher median crash rate (31.4) than preservation-dominant areas (23.2). This is an interesting difference, but it does not mean that new construction causes more crashes.

### What the data cannot tell us
These are **area-level associations**: they do not show that housing production causes crashes, and they say nothing about the risk faced by individual residents. Preservation does not necessarily add residents. Rates use 2018 population estimates, four years are too few for trend claims, some crashes could not be placed in an area, and SIRTA covers serious crashes only.

### Why it matters for New York City
The results can help **DOT** and **HPD** see where affordable housing and serious crashes overlap, so that street-safety reviews can be considered alongside housing projects — especially in the RQ2 overlap areas and the RQ8 candidate areas, where unfinished projects will bring new or renovated homes. Community boards and housing advocates can use the website to look up their own borough, year and housing type. Any action should start with a closer local look, because these figures identify places to investigate, not causes.


## 12. Limitations
* **Correlation is not causation.** Every result is an association between areas. None of them shows that housing production raises or lowers crashes, and none describes individual residents.
* **Built or preserved units are not new residents.** Preservation repairs or refinances existing homes, so completed units are not a count of new people.
* **What the datasets cover.** SIRTA only has serious crashes reviewed by DOT (from 2022), not all collisions, and its ZIP column is much less complete in 2022. H2 only covers HPD-financed affordable housing, and confidential buildings have no location.
* **Old population numbers, used twice.** The 2018 population estimate is used for both the housing rate and the crash rate. Areas with few residents but a lot of traffic (for example Midtown or the Financial District) get very high rates per resident, which does not measure anyone's personal exposure to traffic.
* **Short period.** 2022–2025 gives four yearly values — enough to describe change, not to prove a trend. 2026 is not complete and is left out.
* **Simple cut-offs.** The production tiers, the RQ2 groups and the RQ5 limits (60% / 40%) are simple splits; areas just above and just below a split are similar.
* **Geographic matching.** Some records cannot be placed in a ZIP area. Whether MODZCTA may be used as a reference file still needs to be confirmed by the TA.

The table below lists the data problems that remain after cleaning. They are properties of the source data, not errors in the code.

In [77]:
# Remaining data problems, how we handled them and how they could affect the results.
n_units_unknown = int(h["units_unknown"].sum())
n_ns_window = int((t["id_nonstandard"] & t["crash_year"].isin(ANALYSIS_YEARS)).sum())
repeated_ids = ", ".join(str(c) for c in id_resolution["crash_id"]) if len(id_resolution) else "none"
limitations = pd.DataFrame([
    ("Housing", "Confidential buildings (no published location)", int(h["is_confidential"].sum()),
     "kept in citywide and borough totals", f"{len(conf_window):,} buildings completed in 2022–2025 ({int(conf_window['units_counted'].sum()):,} units) are missing from area metrics"),
    ("Housing", "Counted units unknown", n_units_unknown, "buildings counted; units left out of sums", "unit totals slightly understated" if n_units_unknown else "none"),
    ("Housing", "Completion before project start", int(h["date_order_issue"].sum()), "kept and flagged", "completion year may be wrong for these buildings"),
    ("Housing", "Building in more than one project", int(h["building_in_multiple_projects"].sum()), "kept (HPD counts units per project) and flagged",
     "a physical building may be counted more than once"),
    ("SIRTA", "Orphan rows (no id, date or injuries)", int(is_orphan.sum()), "excluded with the reason recorded", "none (cannot be tied to a crash or year)"),
    ("SIRTA", f"Repeated collision id ({repeated_ids})", int(id_resolution["records"].str.split(", ").str.len().sum()) if len(id_resolution) else 0,
     "resolved with the id-sequence evidence where possible; fatality conflicts left as recorded and flagged",
     "a fatality total may differ by the conflicting amount" if t["fatality_conflict"].any() else "none"),
    ("SIRTA", "Collision ids outside the regular sequence", int(t["id_nonstandard"].sum()), "kept; not treated as date errors",
     f"{n_ns_window} of them fall inside 2022–2025"),
    ("SIRTA", "Possible date problems (id check)", int(t["date_suspect"].sum()), "kept and flagged", "a few crashes may belong to another year"),
    ("SIRTA", "2022–2025 crashes not placed in an area", int(t_window_all["modzcta"].isna().sum()), "kept in borough totals",
     f"{fmt(unplaced_pct.min())}–{fmt(unplaced_pct.max())}% per year; coverage shifts the 2022–2025 change by {fmt(COVERAGE_BIAS_PP, '+.1f')} points"),
    ("SIRTA", "Reported ZIP disagrees with coordinates", int(t["zip_coord_agree"].eq(False).sum()), "coordinates used first",
     f"{fmt(dis.loc['sirta', 'neighbouring_area_pct'], '.0f')}% of disagreements are neighbouring areas"),
    ("SIRTA", "Blank 3-year look-back value", int(t["repeat_location_status"].eq("unknown").sum()), "kept as unknown, never zero", "RQ7 shares use known values only"),
    ("SIRTA", "Crash time recorded exactly 00:00", int(t["time_exact_midnight"].sum()), "excluded from RQ4 only", "small"),
    ("Reference", "2018 population for all areas", len(mz), "used for all per-resident rates", "rates may be off where population changed"),
], columns=["dataset", "limitation", "records", "handling", "possible effect on the analysis"])
limitations

,dataset,limitation,records,handling,possible effect on the analysis
0,Housing,Confidential buildings (no published location),1949,kept in citywide and borough totals,"716 buildings completed in 2022–2025 (1,192 units) are missing from area metrics"
1,Housing,Counted units unknown,29,buildings counted; units left out of sums,unit totals slightly understated
2,Housing,Completion before project start,20,kept and flagged,completion year may be wrong for these buildings
3,Housing,Building in more than one project,247,kept (HPD counts units per project) and flagged,a physical building may be counted more than once
4,SIRTA,"Orphan rows (no id, date or injuries)",444,excluded with the reason recorded,none (cannot be tied to a crash or year)
5,SIRTA,Repeated collision id (4597666),2,resolved with the id-sequence evidence where possible; fatality conflicts left as recorded and flagged,a fatality total may differ by the conflicting amount
6,SIRTA,Collision ids outside the regular sequence,5,kept; not treated as date errors,0 of them fall inside 2022–2025
7,SIRTA,Possible date problems (id check),0,kept and flagged,a few crashes may belong to another year
8,SIRTA,2022–2025 crashes not placed in an area,1181,kept in borough totals,5.2–12.4% per year; coverage shifts the 2022–2025 change by +6.9 points
9,SIRTA,Reported ZIP disagrees with coordinates,559,coordinates used first,100% of disagreements are neighbouring areas


## 13. Validation and saved project outputs
### 13.1 Validation summary
**A. Checks** — did every check in the notebook pass? **B. Known data problems** — problems in the source data that we kept or resolved on purpose (these are not failures). **C. Cleaning log** — what each cleaning step changed.

In [78]:
# Summary of all checks, known data problems and cleaning steps recorded in the notebook.
validation_df = pd.DataFrame(VALIDATIONS)
anomalies_df = pd.DataFrame(KNOWN_ANOMALIES)
cleaning_df = pd.DataFrame(CLEANING_LOG)
status_counts = validation_df["status"].value_counts().reindex(["PASS", "WARN", "FAIL"]).fillna(0).astype(int)
display(Markdown("**A. Pipeline checks by stage**"))
display(validation_df.pivot_table(index="stage", columns="status", values="check_id", aggfunc="count", fill_value=0)
        .reindex(columns=["PASS", "WARN", "FAIL"], fill_value=0))
if (validation_df["status"] == "WARN").any():
    display(validation_df[validation_df["status"] == "WARN"])
display(Markdown("**B. Known data anomalies**")); display(anomalies_df)
display(Markdown("**C. Cleaning log**")); display(cleaning_df)

**A. Pipeline checks by stage**

status,PASS,WARN,FAIL
stage,,,
cleaned-data validation,10,0,0
extraction,12,0,0
geographic reference,5,0,0
housing cleaning,11,0,0
integrated validation,15,0,0
post-integration cleaning,3,0,0
research checks,8,0,0
setup,2,0,0
sirta cleaning,8,0,0


**B. Known data anomalies**

,anomaly_id,dataset,records,description,handling
0,HOUSING-DATE-ORDER,housing,20,building completion date earlier than project start date,retained with the completion year as reported; listed as a limitation (Section 12)
1,HOUSING-MULTI-PROJECT,housing,247,123 buildings appear in more than one project,retained (HPD counts units per project) and flagged
2,HOUSING-CONFIDENTIAL,housing,1949,"confidential buildings without published location (2,805 counted units in total)",kept in citywide and borough totals; cannot be placed in a ZIP area
3,SIRTA-ID-4597666,sirta,2,"collision id 4597666 recorded on 2023-01-13, 2026-01-13 with fatalities 0, 1","kept T02147 (date consistent with the id sequence); excluded T10709 (its date contradicts the id sequence, see 7.5);..."
4,SIRTA-NONSTANDARD-IDS,sirta,5,collision ids outside the regular NYPD sequence (crash years: 2026: 5),retained; dates are plausible and the id-based check cannot assess them; not treated as date errors
5,SIRTA-SUSPECT-DATES,sirta,0,crash dates that contradict neighbouring collision ids (possible date-entry errors or weak evidence),retained with original dates and flagged; listed as a limitation (Section 12)


**C. Cleaning log**

,dataset,step,action,records_affected,rows_before,rows_after,reason
0,housing,6.1,"standardise text, borough, construction type, yes/no flags",0,9551,9551,consistent categories for grouping and filtering
1,housing,6.2,address placeholders '----' -> missing,3898,9551,9551,placeholder is not an address
2,housing,6.2,"placeholder BINs (e.g. 3000000) -> missing, flagged",482,9551,9551,a borough code followed by zeros is not a real building number
3,housing,6.2,date-like house numbers flagged,7,9551,9551,original address cannot be recovered with certainty
4,housing,6.3,ZIP parsed strictly; invalid ZIPs -> missing,0,9551,9551,only complete 5-digit NYC ZIPs are usable
5,housing,6.3,coordinates outside NYC -> missing,0,9551,9551,a point outside the city cannot be placed in a ZIP area
6,housing,6.4,completion-before-start records flagged,20,9551,9551,cannot tell which date is wrong
7,housing,6.5,negative or fractional unit counts -> missing,0,9551,9551,a building cannot have negative or fractional units
8,housing,6.7,missing all_counted_units derived from agreeing income and tenure sums,0,9551,9551,two independent breakdowns give the same total
9,housing,6.8,structural zeros filled in reconciled breakdowns,105816,9551,9551,"components add up exactly to the reference total, so blanks are zeros"


### 13.2 Saved outputs
All files are saved inside the project folder. The files in `data/processed/` are the ones the website needs: copy that folder into the GitHub repository (see Section 14.5). The website reads these files once when it starts and never downloads data from NYC Open Data.

| File | Contents | Used by |
|---|---|---|
| `data/processed/panel_modzcta_year.csv` | Integrated table: ZIP area × year | website, RQ1–RQ8 |
| `data/processed/crashes_for_app.csv` | Serious crashes placed in an area, with hour and weekday | website RQ4 |
| `data/processed/area_pipeline.csv` | Unfinished affordable units per area, by construction type | website RQ8 |
| `data/processed/borough_year_summary.csv` | Borough totals, including confidential units, crashes not placed in an area and an *Unknown* row for crashes with no borough | website RQ3 coverage |
| `data/processed/buildings_for_app.csv` | Completed buildings with coordinates | website RQ2 map |
| `data/processed/modzcta_simplified.geojson` | ZIP-area boundaries | website RQ2 map |
| `data/processed/metadata.json` | Snapshot date, source mode, row counts | website header |
| `data/processed/housing_clean.csv`, `sirta_clean.csv` | Cleaned record-level datasets with flags | reference |
| `reports/*` | Validation summary, cleaning log, anomalies, record ledger, limitations, data story, results | documentation |

In [79]:
# Save the tables for the website, the cleaned datasets and the reports.
P, R_ = DIRS["processed"], DIRS["reports"]
H_EXPORT_COLS = ["record_id", "project_id", "building_id", "is_confidential", "borough", "construction_type", "completion_status",
                 "project_start_date", "building_completion_date", "start_year", "completion_year", "units_counted", "total_units",
                 *H_INCOME, *H_TENURE, "income_breakdown_status", "zip5", "zip_status", "latitude", "longitude", "has_coords", "modzcta",
                 "area_borough", "assignment_method", "unassigned_reason", "date_order_issue", "building_in_multiple_projects",
                 "units_counted_derived", "units_unknown", "units_size_outlier", "counted_exceeds_total", "status_inconsistent"]
T_EXPORT_COLS = ["record_id", "crash_id", "crash_date", "crash_year", "weekday", "crash_time_hhmm", "crash_hour", "crash_time_status",
                 "time_exact_midnight", "crash_borough", "on_street", "cross_street", *T_COUNTS, "injury_counts_missing",
                 "repeat_location_status", "is_highway", "highway_basis", "zip5", "zip_status", "latitude", "longitude", "has_coords",
                 "modzcta", "area_borough", "summary_borough", "borough_source", "assignment_method", "unassigned_reason",
                 "date_suspect", "id_nonstandard", "id_conflict", "fatality_conflict"]
TABLES = {
    P / "panel_modzcta_year.csv": panel,
    P / "crashes_for_app.csv": crashes_app,
    P / "area_pipeline.csv": area_pipeline,
    P / "borough_year_summary.csv": borough_year,
    P / "buildings_for_app.csv": buildings_app,
    P / "housing_clean.csv": h[H_EXPORT_COLS],
    P / "sirta_clean.csv": t[T_EXPORT_COLS],
    R_ / "sirta_excluded_records.csv": t_excluded[["record_id", "exclusion_reason"] + list(t_raw.columns)],
    R_ / "record_ledger.csv": ledger,
    R_ / "cleaning_log.csv": cleaning_df,
    R_ / "known_anomalies.csv": anomalies_df,
    R_ / "source_data_limitations.csv": limitations,
    R_ / "id_date_audit.csv": id_audit,
    R_ / "repeated_id_resolution.csv": id_resolution,
}
for path, frame in TABLES.items():
    frame.to_csv(path, index=False)
(P / "modzcta_simplified.geojson").write_text(json.dumps(GEOJSON))
METADATA = {"snapshot_date": SNAPSHOT_DATE, "source_mode": SOURCE_MODE, "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "analysis_years": ANALYSIS_YEARS, "areas": int(panel["modzcta"].nunique()), "panel_rows": len(panel),
            "housing_rows_raw": len(h_raw), "sirta_rows_raw": len(t_raw), "sirta_rows_retained": len(t),
            "sources": {name: SOURCES[name]["page"] for name in SOURCES}}
(P / "metadata.json").write_text(json.dumps(METADATA, indent=2))
(R_ / "results_summary.json").write_text(json.dumps(RESULTS, indent=2, default=str))
(R_ / "data_story.md").write_text(STORY_MARKDOWN)
print("Tables written:", len(TABLES))

Tables written: 14


### 13.3 Does the website give the same results as the notebook?
We read the saved files back with the website's own function (`load_app_data`) and calculate the default report again. Every research-question result (and the RQ3 coverage values) must be the same as above. This shows that the website will display the same numbers as this notebook.

In [80]:
# Read the saved files back and compare the results with the ones calculated in Section 10.
reloaded = build_report(load_app_data(P), DEFAULT_FILTERS)
same_area = np.allclose(reloaded["area"].sort_values("modzcta")[["units", "crash_total", "crash_rate"]].astype(float).fillna(-1),
                        AREA.sort_values("modzcta")[["units", "crash_total", "crash_rate"]].astype(float).fillna(-1))
same_rho = (np.isnan(reloaded["rho"]) and np.isnan(REPORT["rho"])) or np.isclose(reloaded["rho"], REPORT["rho"])
same_rq = (reloaded["priority"]["modzcta"].tolist() == REPORT["priority"]["modzcta"].tolist()
           and reloaded["types"]["areas"].fillna(0).tolist() == REPORT["types"]["areas"].fillna(0).tolist()
           and np.allclose(reloaded["time_cells"]["diff_pp"].fillna(0), REPORT["time_cells"]["diff_pp"].fillna(0))
           and reloaded["lookback"].fillna(-1).equals(REPORT["lookback"].fillna(-1))
           and np.allclose(reloaded["coverage"]["assignment_rate_pct"].astype(float), REPORT["coverage"]["assignment_rate_pct"].astype(float)))
check("website_reproduces_notebook", same_area and same_rho and same_rq,
      "the exported files reproduce every research-question result of the notebook",
      "the exported files give different results from the notebook (check the export cell)", stage="outputs")
pd.DataFrame(VALIDATIONS).to_csv(R_ / "validation_summary.csv", index=False)
status_counts = pd.DataFrame(VALIDATIONS)["status"].value_counts().reindex(["PASS", "WARN", "FAIL"]).fillna(0).astype(int)
print("Validation checks:", status_counts.to_dict())
files = sorted(p for p in PROJECT_ROOT.rglob("*") if p.is_file() and "raw" not in p.parts)
pd.DataFrame({"file": [str(p.relative_to(PROJECT_ROOT)) for p in files], "size_kb": [round(p.stat().st_size / 1024, 1) for p in files]})

[PASS] website_reproduces_notebook: the exported files reproduce every research-question result of the notebook
Validation checks: {'PASS': 76, 'WARN': 0, 'FAIL': 0}


,file,size_kb
0,data/processed/area_pipeline.csv,3.2
1,data/processed/borough_year_summary.csv,0.9
2,data/processed/buildings_for_app.csv,126.8
3,data/processed/crashes_for_app.csv,443.2
4,data/processed/housing_clean.csv,2075.9
5,data/processed/metadata.json,0.5
6,data/processed/modzcta_simplified.geojson,386.9
7,data/processed/panel_modzcta_year.csv,108.5
8,data/processed/sirta_clean.csv,2964.8
9,reports/cleaning_log.csv,2.6


In [81]:
# Put data/processed and reports into one zip. The folders inside match the GitHub repository.
bundle_path = PROJECT_ROOT.parent / "processed_outputs.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in ["data/processed", "reports"]:
        for path in sorted((PROJECT_ROOT / folder).glob("*")):
            bundle.write(path, arcname=str(path.relative_to(PROJECT_ROOT)))
print("Bundle:", bundle_path, f"({bundle_path.stat().st_size / 1e6:.1f} MB)")
DOWNLOAD_BUNDLE = False          # set to True in Colab to download the zip to your computer
if DOWNLOAD_BUNDLE:
    from google.colab import files as colab_files
    colab_files.download(str(bundle_path))

Bundle: /content/processed_outputs.zip (1.0 MB)


## 14. Project development, AI assistance and contributions
This section explains how the project was developed, who did what, how the work was tested, and which AI prompts were used.

| Part | Content |
|---|---|
| 14.1 | Development history, stage by stage |
| 14.2 | Roles: the student, ChatGPT and Claude; team contributions |
| 14.3 | AI prompt log (summary) |
| 14.4 | Testing and verification status |
| 14.5 | How to rerun the notebook; data sources |
| 14.6 | Exact prompt texts |

### 14.1 Development history
Each stage lists its objective, the work done, the tools used, the student's own involvement and the result. Dates are given only where they are recorded.

**Stage 1 — Understanding the assignment** · *completed*
* **Objective:** find out exactly what Milestone 1 requires.
* **Work:** the Project Description PDF was reviewed for the main objectives, the dataset rules (one housing and one transportation dataset, at least nine columns, a column that can link them), the data engineering steps (EDA, pre-integration cleaning, integration, post-integration cleaning), the website requirements (dropdown filters, search, Generate Report, charts, story, free hosting), the number of research questions and charts (eight for a four-member team), the grading criteria and the submission rules.
* **Tools:** Project Description PDF; ChatGPT summarised and explained the requirements.
* **Student:** reviewed the requirements and decided to build the simplest complete implementation that meets the PDF.
* **Result:** a requirement list that was used in every later stage.

**Stage 2 — Reviewing the Data Catalog** · *completed*
* **Objective:** find a housing and transportation pair that is reasonably easy to implement, has useful location and time columns, does not need excessive preprocessing, can support eight meaningful research questions and works well in Python and Google Colab.
* **Work:** several housing and transportation datasets from the catalog were compared.
* **Tools:** Data Catalog PDF; ChatGPT compared the candidates and explained their advantages and disadvantages.
* **Student:** compared the options against these goals.
* **Result:** a short list that led to the final pair.

**Stage 3 — Selecting the datasets** · *completed*
* **Objective:** choose the two datasets.
* **Work:** *Affordable Housing Production by Building* (H2) and *SIRTA* serious-crash data (T4) were selected. Both have ZIP codes, coordinates and dates, so they can be combined by area and year. Claude checked the pair against the live NYC Open Data API (row counts, columns and how complete the location fields are) — Claude prompt 1, 14 October 2026.
* **Tools:** ChatGPT (recommendation); Claude (feasibility check).
* **Student:** reviewed and approved the selection.
* **Result:** H2 + T4 approved. MODZCTA is used only as a geographic reference; the TA still has to confirm that this is allowed.

**Stage 4 — Planning the project** · *completed*
* **Objective:** define the main question, the eight research questions and the integration plan.
* **Work:** the main question and eight research questions in four sections (A–D) were defined. Decisions: study 2022–2025 (SIRTA starts in 2022 and 2026 is incomplete); combine the data by ZIP-code area and year; use Python in Google Colab; use Plotly for the charts and Dash for the website.
* **Tools:** ChatGPT (planning and review); Claude (project design and the first notebook, prompt 1).
* **Student:** decided the scope and reviewed and approved the research-question designs.
* **Result:** the project plan used in this notebook.

**Stage 5 — Setting up the environment** · *Colab completed; GitHub in progress*
* **Objective:** prepare the tools for running and sharing the project.
* **Work:** Google Colab was used to run the notebooks; the required libraries were identified (pandas, numpy, requests, Plotly, geopandas, shapely, and Dash for the website); a GitHub repository was created and the project files are being organised in it.
* **Tools:** Google Colab, GitHub.
* **Student:** set up and ran Google Colab; created and prepared the GitHub repository and is organising and uploading the files.
* **Result:** Colab works with all libraries; the GitHub file paths still need to be corrected.

**Stage 6 — Data exploration** · *completed*
* **Objective:** understand both datasets before cleaning them.
* **Work:** Sections 3–5 of this notebook check the number of rows and columns (9,551 × 41 for housing and 13,754 × 23 for SIRTA in the 15 October 2026 download), data types, missing values, duplicates, outliers, dates and location columns, with charts.
* **Tools:** Python (pandas, Plotly) in Colab; code generated by Claude.
* **Student:** ran the notebooks and examined the outputs.
* **Result:** the list of problems in Section 5.5, which guided the cleaning.

**Stage 7 — Data cleaning** · *completed*
* **Objective:** fix or flag the problems found, without inventing values or silently deleting records.
* **Work (numbers from the 15 October 2026 run):** 1,949 confidential housing records have no location and are counted only at borough level; 444 SIRTA rows with no id, date or injury counts were removed (3 of them were exact copies); one collision id (4597666) appeared twice with different dates and fatality counts — the copy whose date fits the id sequence was kept and the fatality difference was flagged; the placeholder ZIP 10000 and other invalid ZIP codes (44 crash records) became missing; 20 buildings completed before their project start were flagged; 227 crash times of exactly 00:00 were flagged as uncertain; 29 buildings with unknown unit counts were kept but left out of unit sums; 1,393 blank look-back values were kept as *unknown*, not zero; text, categories, dates and IDs were put into consistent formats.
* **Tools:** Python; code generated by Claude and reviewed with ChatGPT.
* **Student:** reviewed the cleaning decisions and the outputs.
* **Result:** cleaned housing and crash tables (Sections 6–7).

**Stage 8 — Integrating the datasets** · *completed*
* **Objective:** combine housing and crashes in a way that makes sense.
* **Work:** both datasets were counted per ZIP-code area and year and joined one-to-one (177 areas × 4 years = 708 rows). This avoids pairing individual buildings with unrelated individual crashes. After the join, post-integration cleaning filled the empty counts, removed redundant columns and fixed the data types; validation checks confirmed that no rows were multiplied and that the totals were unchanged (10,777 crashes and 100,057 units placed in areas in the 15 October run).
* **Tools:** Python (pandas, geopandas); code generated by Claude.
* **Student:** reviewed the integration totals and the validation outputs.
* **Result:** the integrated table used by all research questions and the website.

**Stage 9 — Testing and debugging** · *completed*
* **Objective:** make sure the code runs on the real data and the numbers are correct.
* **Work:** the feasibility notebooks were run in Colab and their real outputs were examined. A pandas missing-value error appeared in the area-summary cell (a `pd.NA` value inside a True/False comparison); ChatGPT helped diagnose it and suggest a correction, the corrected cell was applied and rerun, and later versions kept that cell unchanged. Integration totals and validation outputs were reviewed after each run. A review of the real processed outputs then found that the borough summary dropped 36 crashes with no reported borough; this was corrected (Section 9.2) and the notebook was run again.
* **Tools:** Google Colab; ChatGPT (diagnosis and review); Claude (implementation and revisions).
* **Student:** ran the notebooks, reported the errors, reviewed the proposed corrections and checked the results.
* **Result:** both real-data runs passed all checks (Stage 13).

**Stage 10 — Creating the research visualizations** · *completed*
* **Objective:** answer each research question with one suitable chart.
* **Work:** each of the eight questions was linked to one chart: scatter plot (RQ1), map (RQ2), line chart (RQ3), heatmap (RQ4), box plot (RQ5), comparative bar chart (RQ6), stacked bar chart (RQ7) and ranked bar chart (RQ8). The charts were reviewed to check that each one answers its question and links housing with crashes.
* **Tools:** Plotly; chart code generated by Claude; reviews with ChatGPT.
* **Student:** reviewed the charts against the research questions.
* **Result:** the eight charts in Section 10, also used on the website.

**Stage 11 — Reviewing the project against the PDF** · *completed*
* **Objective:** check that everything the PDF asks for is there, and nothing unnecessary.
* **Work:** the implementation was compared with the official requirements. An earlier version had grown more complex than the assignment needs, so advanced statistics were removed: bootstrap confidence intervals, partial correlation, repeated robustness tests, adjusted significance tests and a separate sensitivity section.
* **Tools:** ChatGPT (compliance check and preparing the instructions); Claude (requirement comparison, prompt 4).
* **Student:** decided to simplify the project to what the assignment requires.
* **Result:** a list of what to keep, simplify and remove.

**Stage 12 — Simplifying the final notebook** · *completed*
* **Objective:** a notebook that is complete but easy to follow and explain.
* **Work:** the advanced notebook was turned into this simplified version with 14 sections. All loading, EDA, cleaning, integration and post-integration steps and the eight research charts were kept. The calculations were moved into shared code (`src/analysis.py` and `src/figures.py`) so the website uses the same numbers.
* **Tools:** Claude (prompt 4, 15 October 2026).
* **Student:** requested the simplification and reviewed the result.
* **Result:** the simplified notebook.

**Stage 13 — Real-data execution and validation** · *completed*
* **Objective:** run the simplified notebook on the real NYC Open Data.
* **Work:** the student ran the simplified notebook in Colab on 15 October 2026 with a fresh download: all 80 code cells ran, 74 validation checks passed with no warnings and no failures, the eight research charts were generated, and the exported files reproduced the notebook's research results (check `website_reproduces_notebook`). After the borough correction and the rewritten documentation, the revised notebook was run again on real data: 76 checks passed (including the two new borough checks), with 0 warnings and 0 failures, 708 area-year rows and the eight charts. The borough totals now reconcile with the full data: 11,958 crashes in 2022–2025, 10,777 of them placed in an area, and 101,249 completed affordable units; 28 crashes take the borough of their ZIP area and 8 are counted as Unknown.
* **Tools:** Google Colab.
* **Student:** ran the notebook, checked the outputs and approved the final "What surprised us" paragraph.
* **Result:** verified real-data results. The RQ1–RQ8 results did not change with the borough correction; only the RQ3 coverage bars moved slightly.

**Stage 14 — Website development and GitHub preparation** · *website code completed and tested locally with the final real data; public testing pending*
* **Objective:** build the website the PDF requires and prepare the repository.
* **Work:** Claude generated the Dash website: five dropdown filters, a keyword search, a Generate Report button, eight interactive research charts and a data-story section. Claude tested it with synthetic data on 15 October and locally with the real processed outputs of the 15 October run on 16 October, and again with the outputs of this final notebook's run while packaging the repository. It has not been deployed yet.
* **Tools:** Dash, Plotly; Claude.
* **Student:** created and prepared the GitHub repository and has been organising and uploading the project files.
* **Result:** website source code ready and tested locally with the latest exports; it still has to be deployed.

**Stage 15 — Final review and submission** · *pending*

| Task | Status |
|---|---|
| Review the revised notebook and approve the data-story paragraph | Completed (student) |
| Rerun the revised notebook in Colab (76 checks passed, 0 warnings, 0 failures) | Completed (student) |
| Run this final submission notebook once more in Colab to save its outputs | Completed (student, 16 October 2026: 81 of 81 code cells, 76 checks passed, 0 warnings, 0 failures) |
| Test the website with the latest real-data exports | Completed locally (Claude, while packaging the repository) |
| Correct the GitHub file paths and upload the final files | In progress (student) |
| Deploy the website on Render and test the live site | Pending |
| Document the actual contributors; ask the instructor/TA about the team requirement and about MODZCTA | Pending |
| Send the submission email (member IDs, GitHub link, website link, dataset links) by 9 November 2026 | Pending |

### 14.2 Roles and AI assistance
The project was developed as a series of decisions, reviews and tests by the student, with AI tools helping with the implementation.

**Abdelrahman El-sayed (13004550)**
* reviewed the university assignment;
* compared and approved the dataset choices;
* decided the project scope and reviewed the research-question designs;
* set up and ran Google Colab and examined the execution outputs;
* reported errors, tested the corrections and requested improvements;
* reviewed the project against the assignment;
* created and organised the GitHub repository;
* tests and checks the project's functionality as each part is completed.

He did not write the AI-generated code by hand: he directed it, reviewed it, ran it and checked the results.

**ChatGPT (OpenAI)** — explained the Project Description; compared datasets and recommended suitable options; helped plan the workflow and explained integration decisions; reviewed feasibility results and notebook outputs; helped diagnose errors and suggested and reviewed code corrections; checked compliance with the assignment; helped prepare the implementation and revision instructions sent to Claude.

**Claude (Anthropic)** — generated the Python notebook code, including data cleaning, integration and the research visualizations; revised the code after feedback; simplified the notebook; generated the Dash website code; prepared supporting project files and documentation; ran tests on synthetic data (and, on 16 October, local website and borough checks on the real exported files), as recorded in Section 14.4.

**Team contributions**

| Member | Student ID | Research questions and visualizations | Data cleaning and integration | Website | Other work |
|---|---|---|---|---|---|
| **Abdelrahman El-sayed** | 13004550 | Responsible for all eight research-question sections (RQ1–RQ8, Sections A–D) and their charts | Responsible for data processing: loading, EDA, cleaning, integration and post-integration cleaning | Responsible for the website implementation and the GitHub repository | Coordination; running and reviewing the notebook |
| Team member 2 | Not yet assigned | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided |
| Team member 3 | Not yet assigned | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided |
| Team member 4 | Not yet assigned | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided | Not yet assigned / contribution not provided |

> **Open administrative question.** The Project Description expects teams of 4–6 students in which **each member proposes at least two research questions with one chart each** (eight for a four-member team). Only one contributor is documented here, so this requirement is **not** fulfilled by four members: the eight questions were developed by Abdelrahman El-sayed with AI assistance. If the project is completed individually, this needs **clarification from the instructor/TA** before submission.

### 14.3 AI prompt log (summary)
The Project Description asks for the prompts used in the project. This table summarises the development stages; it does not replace the exact prompts in Section 14.6.

| Prompt / stage | Tool | Purpose | Result | Verification |
|---|---|---|---|---|
| Assignment analysis | ChatGPT | Understand the PDF requirements | Project requirement summary | Reviewed |
| Dataset comparison | ChatGPT | Compare the options | H2 + T4 recommendation | Reviewed |
| Project planning | ChatGPT / Claude | Plan the questions and the integration (Claude prompt 1) | Project design | Reviewed; feasibility checked against the NYC Open Data API |
| Notebook implementation | Claude | Implement the Python pipeline (Claude prompts 1–3) | Notebook files | Tested as recorded (Section 14.4) |
| Debugging | ChatGPT / Claude | Find and correct errors | Code corrections | Verified where tested |
| Simplification | Claude | Remove unnecessary complexity (Claude prompt 4) | Simplified notebook | Real-data run completed (15 October 2026) |
| Website implementation | Claude | Generate the Dash application (Claude prompt 4) | Website source files | Tested locally with synthetic and real processed data; public deployment pending |
| Documentation and borough correction | Claude | Rewrite comments and documentation; correct the borough summary (Claude prompt 5) | Revised notebook, updated documentation | Real-data rerun completed: 76 checks passed |
| Final submission preparation | Claude | Add the approved story paragraph and update the status texts (Claude prompt 6) | This final notebook, updated documentation | Final Colab run completed by the student on 16 October 2026: 76 checks passed, 0 warnings, 0 failures |

**Exact prompts.** The six prompts sent to Claude are reproduced word for word in Section 14.6 and in `docs/AI_PROMPT_LOG.md`. They were sent by the student; ChatGPT helped prepare the instructions. For the ChatGPT conversations (assignment analysis, dataset comparison, planning and review, debugging, compliance checks): *Exact original prompt not available; development-stage summary recorded.*

### 14.4 Testing and verification status
**Completed — runs on real NYC Open Data in Google Colab (by the student):**
* 15 October 2026 (fresh download): 80 of 80 code cells ran; 74 validation checks passed with 0 warnings and 0 failures; the eight research charts were generated; the exported files reproduced the notebook's research results.
* After the borough correction (revised notebook): 76 validation checks passed with 0 warnings and 0 failures; 708 area-year rows; the eight research charts; the borough totals reconcile (11,958 crashes, 10,777 placed in an area, 101,249 completed units; 28 boroughs taken from the ZIP area and 8 Unknown). The RQ1–RQ8 results were the same as on 15 October; only the RQ3 coverage bars moved slightly.
* 16 October 2026, this final submission notebook (fresh download): 81 of 81 code cells ran; 76 validation checks passed with 0 warnings and 0 failures; 708 area-year rows; the eight research charts.

**Checked by Claude (in its own sandbox, not in Colab):** the revised notebook ran end to end on synthetic test files; the corrected borough code gave the totals above on the real records exported on 15 October; the website worked locally with the real processed outputs of the 15 October run; the automated tests in the repository passed. While packaging the repository, the website was tested again locally with the outputs of this final run: all eight charts, the dropdowns, the search and Generate Report worked, and its results matched this notebook.

**This final version** only changes text: the approved story paragraph, a few comments and these status notes. No calculation changed, so the results stay the same; it was run once more in Colab to save its outputs in the notebook.

**Still pending:**
* public deployment of the website and testing of the live site;
* final GitHub check;
* clarification from the instructor/TA about the team requirement and MODZCTA;
* the university submission.

### 14.5 Rerunning the notebook and data sources
1. Open the notebook in Google Colab and choose **Runtime → Run all** (a standard CPU runtime is enough). Missing libraries are installed automatically.
2. The run downloads the data to `dev_project/data/raw/` and saves a manifest (`manifest_<date>.json`) with URLs, times, row counts and checksums. To get exactly the same results later, upload those raw files to `dev_project/data/raw/` and set `RAW_SNAPSHOT_DATE = "<date>"` in Section 3.2.
3. Check Section 13.1: there must be no `FAIL`.
4. In the last cell of Section 13, set `DOWNLOAD_BUNDLE = True` (or download `dev_project/processed_outputs.zip` from the file browser). Unzip it in the root of the GitHub repository: it fills `data/processed/` (the website data) and `reports/`.
5. Save the notebook with its outputs (**File → Download → .ipynb**) into the repository's `notebooks/` folder.

**Data sources**
* Affordable Housing Production by Building (H2): https://data.cityofnewyork.us/d/hg8x-zxpr
* Serious Injury Response, Tracking & Analysis Program — SIRTA (T4): https://data.cityofnewyork.us/d/xeqp-qz8h
* Modified ZIP Code Tabulation Areas — MODZCTA (geographic reference; TA confirmation pending): https://data.cityofnewyork.us/d/pri4-ifjk
* Course documents: *Project Milestone 1* description and *Dataset Catalog*, Data Engineering and Visualization, GIU, Winter Semester 2026.

### 14.6 Exact prompt texts
The prompts sent to Claude, copied word for word from `docs/AI_PROMPT_LOG.md` (long lines are wrapped for display only). Exact ChatGPT prompts are not available (see Section 14.3).

#### Claude prompt 1 — Feasibility verification and project design (2026-10-14)

~~~text
# Data Engineering and Visualization — Project Implementation

You are an experienced Python data engineer, data analyst, and Plotly Dash developer helping me implement my
university project.

I have attached two official university documents:

1. **Project Description.pdf** — the authoritative Milestone 1 requirements.
2. **Data Catalog.pdf** — the recommended datasets and their potential integration fields.

Read both documents thoroughly. Do not ignore, simplify away, or invent assignment requirements.

## Project context

- University: German International University (GIU).
- Course: Data Engineering and Visualization, Winter Semester 2026.
- Milestone: Project Milestone 1.
- Deadline: 9 November 2026.
- Official team size: Four members.
- Required: Exactly one Housing & Development dataset and exactly one Transportation dataset, each having at
least nine columns.
- Required output for our team size: Eight research questions investigating meaningful relationships between
the datasets, each with a corresponding visualization.
- Implementation environment: Google Colab with Python.
- Website framework preference: Python Plotly Dash.
- Final submission: Colab/Jupyter notebook, fully functional interactive website, GitHub repository,
deployment URL, README, source dataset links, documented AI prompts, and honest contribution documentation.

I am coordinating and performing the implementation myself. Do not fabricate work performed by the other three
members. Structure the eight research questions into four labeled sections of two questions each, while
keeping actual contributions accurately documented.

## Proposed datasets — not yet finalized

Housing dataset:

- Name: Affordable Housing Production by Building (H2)
- Dataset ID: `hg8x-zxpr`
- URL: https://data.cityofnewyork.us/d/hg8x-zxpr

Transportation dataset:

- Name: Serious Injury Response, Tracking & Analysis Program (SIRTA) (T4)
- Dataset ID: `xeqp-qz8h`
- URL: https://data.cityofnewyork.us/d/xeqp-qz8h

Proposed research topic:

**Affordable Housing Development and Road Safety in New York City**

Potential integration: Aggregate by ZIP code and calendar year before joining, provided the actual datasets
support this.

Important: SIRTA represents selected serious traffic injury records investigated and reported under the
program, not all traffic collisions. Do not make causal claims.

## PHASE 1 — Feasibility verification

Before implementing the complete project, help me verify whether the proposed dataset pair can meet the
assignment requirements.

Use the actual datasets and Colab output to determine:

1. The number of rows and columns in each dataset.
2. The available geographic, temporal, categorical, and numerical variables.
3. Missing values, duplicate records, date issues, and other data-quality concerns.
4. The availability and consistency of ZIP-code identifiers.
5. The overlapping years and geographical coverage.
6. The feasibility of integrating both datasets at ZIP-code/year granularity.
7. The risk of misleading results due to missing data, partial years, aggregation, or different record
definitions.
8. Whether the pair can support eight genuinely meaningful and sufficiently distinct cross-domain research
questions.

Do not simply join raw records because both datasets contain ZIP codes. Investigate aggregation, potential
duplicate keys, observation grain, and whether an appropriate one-to-one integration can be created at the
aggregated level.

Do not invent column names, row counts, correlations, or findings. If something cannot be verified, state what
is missing.

## PHASE 2 — Proposed project design

After the dataset checks, provide:

- Your recommendation to accept or reject the proposed pair.
- Eight potential research questions that investigate relationships between housing and transportation.
- The exact data fields and metrics needed for each question.
- An appropriate visualization for each question.
- A proposed cleaning and integration strategy.
- A proposed Dash website layout and filter structure.
- Risks, limitations, and recommended mitigation.

Prioritize simplicity, correctness, efficient execution, and a strong grading outcome.

## PHASE 3 — Full implementation, only after my approval

Once I approve the dataset pair and research questions, implement the complete project in clearly separated
stages.

The final project must include:

**A. Google Colab notebook**

Thorough EDA, statistics, appropriate initial visualizations, missing-value handling, outlier detection and
treatment, inconsistent-data checks, standardized geographic and date fields, duplicate handling, justified
aggregation, dataset integration, required post-integration cleaning, and validation.

Include explanatory Markdown before and after the major processing stages. Explain why each cleaning and
integration decision was made and its impact on the data.

**B. Eight research questions and visualizations**

Implement eight meaningful charts corresponding to the eight research questions. Every question must
meaningfully investigate the relationship between the two domains.

Use suitable Plotly visualizations based on the actual data. Avoid repetitive charts and misleading
comparisons.

**C. Interactive Dash website**

Include multiple dropdown filters, a functional text-search mode, and a central Generate Report button.

The Generate Report button must update the visualizations consistently based on the selected filters or parsed
search terms.

Provide meaningful hover information and an informative, responsive, visually consistent interface. Include a
data-story section presenting the research objective, actual findings, limitations, and benefits to NYC.

**D. Complete, reproducible deliverables**

Generate complete source files, a properly structured repository, requirements.txt, README.md, the Colab
notebook, cleaned data or reproducible data-generation process, and deployment configuration.

Prepare deployment to a suitable free hosting platform, preferably Render for the Python Dash application.

Ensure that paths are portable, that the deployed application does not rely on my personal Google Drive, and
that it does not repeatedly download huge datasets at every user interaction.

Include appropriate tests for cleaning, integration, metrics, filters, and report generation.

Document all AI prompts used and provide an accurate contribution log with placeholders where member details
or real contributions are not yet known.

## Quality requirements

- No fabricated results or findings.
- No unnecessary technological complexity.
- No unjustified data deletion or imputation.
- No incorrect many-to-many joins.
- No fake or nonfunctional website controls.
- No claims of causation from correlations.
- No missing mandatory university requirements.
- Clean, modular, documented Python code.
- Clear instructions for executing and verifying each stage.

**For now, focus on PHASE 1 and PHASE 2. Do not begin the complete implementation until I approve the final
dataset selection and proposed research questions.**
~~~

#### Claude prompt 2 — Improve the Colab notebook using real execution results (2026-10-14)

~~~text
# GIU Data Engineering and Visualization — Improve the Google Colab Notebook Using Real Execution Results

You are acting as a senior Python data engineer, data-quality specialist, and data visualization developer.

I want you to **review, correct, improve, and deliver a complete Google Colab notebook** for my university
project.

This is a continuation of our existing project. Do not restart the project, select new datasets, or discard
correct implementation work.

I have now executed the feasibility notebook in Google Colab using real NYC Open Data, and I am providing you
with the execution results.

Your task is to use those results to substantially improve the notebook and prepare a reliable, reproducible
foundation for the final project.

## 1. Project context

University: German International University (GIU)

Course: Data Engineering and Visualization — Winter Semester 2026

Milestone: Project Milestone 1

Submission deadline: 9 November 2026

Team size: Four members

Project title:

**Affordable Housing Development and Road Safety in New York City**

Official requirements:

- Select exactly one Housing & Development dataset and one Transportation dataset.
- Each dataset must contain at least nine columns.
- Explore, clean, and integrate the datasets meaningfully.
- Perform exploratory data analysis.
- Document pre-integration and post-integration cleaning.
- Provide eight meaningful cross-domain research questions and eight supporting visualizations for the
four-member team.
- Ultimately build an interactive website with dropdowns, text search, Generate Report functionality, and data
storytelling.
- Submit the notebook, code, GitHub repository, deployed website, source dataset links, README, contribution
documentation, and AI prompts used.

The implementation is being coordinated and carried out by me. Maintain four labeled research sections, but
never fabricate work performed by other members.

For this request, focus on the **complete, improved Colab notebook**, not the final Dash website.

Read the attached official university PDFs as the authoritative source for grading and deliverable
requirements.

## 2. Approved datasets

Housing & Development:

- Dataset: Affordable Housing Production by Building (H2)
- Dataset ID: `hg8x-zxpr`
- URL: https://data.cityofnewyork.us/d/hg8x-zxpr

Transportation:

- Dataset: Serious Injury Response, Tracking & Analysis Program (SIRTA) (T4)
- Dataset ID: `xeqp-qz8h`
- URL: https://data.cityofnewyork.us/d/xeqp-qz8h

Geographic reference, subject to TA permission:

- MODZCTA
- Dataset ID: `pri4-ifjk`
- URL: https://data.cityofnewyork.us/d/pri4-ifjk

The MODZCTA file is intended only as a supporting geographic reference and population lookup, not as a third
analytical topic dataset.

Keep the analysis window of **2022–2025**. Treat 2026 as a partial year and exclude it from the main
comparisons.

Continue using the existing MODZCTA × year integration strategy, subject to the pending TA permission.

### Explicit exclusion

Do not add a separate geographic-size analysis, polygon-area measurement, land-area comparison, or feature
based on the physical size of geographic regions.

Do not redesign the geographic granularity merely to investigate area size.

Preserve the necessary location matching, geographic data-quality checks, maps, and integration logic, because
these are essential to the project.

## 3. Confirmed Google Colab execution results

I successfully ran the original feasibility notebook on real data.

Snapshot date: 14 October 2026.

### Dataset sizes

Housing:
- 9,551 rows.
- 41 original columns.

SIRTA:
- 13,754 rows.
- 23 original columns.

Geographic reference:
- 178 records, including one excluded placeholder.

These numbers are the observed snapshot, not permanent constants. Revalidate them when loading a new snapshot.

### Housing data findings

- 0 exact duplicate rows.
- 1,949 confidential rows.
- 0 duplicate `(project_id, building_id)` pairs among records with building IDs.
- 123 building IDs appear in multiple projects, representing 247 rows.
- 20 records have a building completion date earlier than the project start date.
- 7,593 records have non-null ZIP codes.
- Coordinate-based and reported-ZIP assignments agree for 95.0% of records where both methods are available,
based on 7,553 comparisons.
- 98.82% of completed affordable units in 2022–2025 are retained when requiring a reported ZIP.
- Some ZIP values are not found in the existing MODZCTA lookup, including 11249 and 10158.

Important: A building appearing in more than one project is not automatically a duplicate. Preserve the
correct record grain.

Confidential buildings have meaningful unit counts even when their individual locations are unpublished. Do
not discard their contribution from appropriately defined citywide or borough-level totals without explaining
the reason.

Missing building-completion dates may indicate unfinished projects. They are not necessarily data errors.

### SIRTA data findings

- 3 exact duplicate rows.
- 444 orphan rows lacking crash ID and date.
- 1 collision ID appears twice.
- 6 records were flagged by the collision-ID-versus-date consistency check.
- 8,072 records have non-null reported ZIP codes.
- Reported ZIP versus coordinate-based assignment agreement: 91.6%, based on 6,692 comparisons.
- 90.5% of dated crash records can be assigned to a MODZCTA.
- 1,336 records were flagged as highway-related by the existing rule.
- Unmapped or placeholder ZIPs include 10000, 11249, 10281, and 11251.

The repeated collision ID is **4597666**.

It appears on 13 January 2023 and 13 January 2026, with matching time and street details but conflicting
fatalities.

Do not automatically merge these records or assume which fatality value is correct. Investigate and provide an
auditable resolution or flag them as unresolved.

The six suspicious crash dates must also be investigated individually. The rolling-median test is a warning
mechanism, not proof that every flagged record is wrong.

### Year-by-year housing and crash records

| Year | Buildings completed | Affordable units completed | SIRTA crashes |
|---|---:|---:|---:|
| 2022 | 654 | 22,007 | 2,879 |
| 2023 | 806 | 30,841 | 3,016 |
| 2024 | 705 | 22,704 | 3,116 |
| 2025 | 792 | 25,697 | 2,947 |

These are observed raw-data counts before the final cleaning pipeline.

### Geographic assignment findings

The real execution showed that assigning crashes using coordinates before falling back to a reported ZIP
substantially improves coverage.

Approximate proportions of dated crash records assigned successfully:

- 2022: 88.8%.
- 2023: 89.4%.
- 2024: 87.5%.
- 2025: 94.8%.

These values are calculated from the notebook's displayed assignment-source counts and should be recomputed
precisely in the improved notebook.

The assignment coverage differs across years and may bias temporal comparisons.

The notebook must explicitly quantify that bias and report annual data coverage.

### Existing integration dry run

The existing notebook produced:

- 177 MODZCTA analytical areas.
- Four full years.
- 708 integrated rows.
- Nine columns in the preliminary integration table.
- Approximately 45.9% of area-year records have zero housing completions.
- Approximately 3.2% have zero assigned serious crashes.
- 130 areas have at least one housing completion during the study window.
- 175 areas have at least one crash in the study window.

The preliminary panel is not yet the final cleaned analytical dataset.

Do not simply copy these numbers into the final notebook. Recompute the integration after cleaning, verify
totals, and document the differences.

## 4. Mandatory notebook improvements

### A. Review and preserve working code

Study my existing notebook cell by cell.

Identify:
- Correct functionality to retain.
- Repeated or unnecessary code.
- Poor variable names.
- Incorrect assumptions.
- Possible silent data-quality errors.
- Inefficient calculations.
- Missing validations.
- Weak explanatory Markdown.
- Charts that are difficult to interpret.

Improve the existing work rather than rebuilding everything unnecessarily.

### B. Robust real-data loading

The notebook must:

1. Run on a fresh Google Colab CPU runtime.
2. Install missing dependencies automatically.
3. Use only necessary Python libraries.
4. Download the real source datasets.
5. Validate downloaded schemas and row counts.
6. Record source URLs, extraction timestamps, and package versions.
7. Save reproducible raw snapshots.
8. Handle network problems with clear errors.
9. Avoid accidentally reading only the default first 1,000 API rows.
10. Never silently replace missing real data with synthetic records.

Use a clear folder structure for raw data, intermediate outputs, final outputs, and validation reports.

### C. Thorough exploratory data analysis

Provide useful descriptive analysis of both datasets:

- Record and column counts.
- Data types.
- Missing-value profiles.
- Numerical summaries.
- Categorical distributions.
- Outliers.
- Duplicate records.
- Date coverage.
- Important variable relationships.
- Geographic-data completeness.
- Examples of problematic records.

Use readable Plotly charts with descriptive titles, axes, legends, and appropriate hover information.

Avoid producing large numbers of repetitive or meaningless charts.

### D. Pre-integration cleaning — Housing

Implement a documented process for:

- Missing values.
- Structural zeros.
- Invalid numeric values.
- ZIP formatting.
- Date standardization.
- Construction-type standardization.
- Duplicate handling.
- Confidential housing records.
- Inconsistent date ordering.
- Implausible unit totals.
- Valid building/project identifiers.
- Projects involving the same building more than once.

Verify whether income-band and bedroom-unit totals agree with the reported overall totals.

Do not assume every null numerical field means zero.

Never automatically delete large but plausible housing projects simply because they are statistical outliers.

Produce before-and-after summaries explaining all changes.

### E. Pre-integration cleaning — SIRTA

Implement and document:

- Removal or exclusion of genuinely unusable orphan records.
- Exact duplicate detection and handling.
- Investigation of repeated crash IDs.
- Investigation of suspicious crash dates.
- Date and crash-time standardization.
- Consistent injury and fatality numeric types.
- Validation of missing and negative values.
- ZIP placeholder handling.
- Highway identification.
- Investigation of inconsistent location assignments.
- Appropriate handling of missing crash times.
- Treatment of missing recurrence/lookback metrics.

In particular:

- Distinguish missing values from true zeros.
- Do not infer missing injury or fatality counts without evidence.
- Do not silently discard records with uncertain values.
- Preserve enough information to explain which records were excluded from a particular analysis and why.

### F. ZIP and location reliability

Retain the coordinate-first location-assignment approach.

Improve the reliability of ZIP parsing so that invalid strings cannot accidentally become valid ZIP codes.

Investigate differences between postal ZIP codes and Census ZCTAs.

Account for the geographic reference's available ZIP-identification fields rather than relying on only one
field without checking it.

Evaluate:
- Coordinate availability.
- Reported ZIP availability.
- Successful assignment.
- Unassigned records.
- ZIP/coordinate disagreement.
- Differences across years.
- Differences across boroughs where relevant.

Avoid arbitrary assignment when reference mappings are ambiguous.

If a coordinate falls exactly on multiple polygon boundaries, use a documented, deterministic strategy and
report the number of affected records.

Do not add geographic-size or polygon-area analysis.

### G. Integration and post-integration cleaning

After preprocessing:

1. Aggregate the housing data to the chosen location/year level.
2. Aggregate crash data to the same level.
3. Confirm that both aggregate keys are unique.
4. Construct the required analytical panel.
5. Use validated joins that prevent many-to-many multiplication.
6. Resolve new missing values caused by the joins.
7. Remove redundant columns.
8. Standardize data types.
9. Calculate well-defined analytical metrics.
10. Verify all source-to-output totals.

Explain which missing aggregate values can legitimately be interpreted as zero and which represent uncertainty
or inadequate source coverage.

Do not equate missing reporting coverage with an observed absence of crashes.

Keep clear records of:
- Original rows.
- Rows cleaned.
- Rows excluded.
- Rows successfully assigned.
- Rows retained in the analysis window.
- Aggregated totals.
- Final panel rows.

### H. Numerical analysis and sensitivity checks

Use sensible descriptive statistical methods for the research questions.

For rates based on population estimates, explicitly state that the available reference population estimate is
from 2018.

A shared and outdated population denominator can distort the relationship between housing and crash rates.

Where appropriate:
- Compare normalized metrics against raw counts.
- Check sensitivity to unusually high-rate observations.
- Examine differences caused by assignment coverage.
- Explain the limitations of area-level analysis.
- Use correlation carefully.
- Report association rather than causation.

Do not add advanced modelling simply to make the project look more complex.

The objective is scientifically defensible analysis that I can understand and explain to my instructor.

## 5. Research-question structure

Preserve the eight proposed research questions as the working design.

### Section A — Spatial Overlap

RQ1: Do areas with more completed affordable housing units per resident also have higher serious-crash rates?

Proposed visualization: Scatter plot.

RQ2: Where do high affordable-housing growth and high serious-crash burden coincide?

Proposed visualization: Interactive map.

### Section B — Timing

RQ3: Did serious crashes change differently across 2022–2025 in areas with high, low, or no affordable-housing
completions?

Proposed visualization: Multi-line chart.

RQ4: Do crashes in high-housing-growth areas occur at different times of day and week?

Proposed visualization: Heatmap.

### Section C — Housing Characteristics

RQ5: Are areas dominated by new affordable-housing construction associated with different crash rates than
preservation-dominant areas?

Proposed visualization: Box plot.

RQ6: Are affordable units serving different income groups being delivered in areas with different
serious-crash rates?

Proposed visualization: Comparative bar chart.

### Section D — Road Safety Priorities

RQ7: Are new affordable units being completed in areas where serious crashes recur at the same locations?

Proposed visualization: Stacked bar chart.

RQ8: Which areas combine unfinished affordable-housing projects with a relatively high serious-crash burden?

Proposed visualization: Ranked bar chart.

These questions may be refined if a metric cannot be supported by the actual cleaned data.

In particular, verify the official definition of SIRTA's historical similar-crash variables before using them
for RQ7.

Do not invent findings, interpretations, or statistical relationships.

For the notebook, prepare the cleaned data and reusable functions required for these questions and include the
appropriate research visualizations where feasible.

## 6. Notebook organization

Produce a professionally organized notebook with the following logical sections:

1. Project introduction, motivation, and research objectives.
2. Official dataset descriptions and selection justification.
3. Environment setup and imports.
4. Data extraction and snapshot information.
5. Initial dataset overview.
6. Exploratory data analysis — Housing.
7. Exploratory data analysis — Transportation.
8. Initial data-quality findings.
9. Pre-integration cleaning — Housing.
10. Pre-integration cleaning — Transportation.
11. Cleaned-data validation.
12. Integration methodology and justification.
13. Dataset aggregation and integration.
14. Required post-integration cleaning.
15. Integrated-data validation.
16. Eight research questions and analytical visualizations.
17. Results and data storytelling.
18. Limitations and sensitivity checks.
19. Final data-quality summary.
20. Saved outputs for the future Dash website.
21. Reproducibility instructions, source links, AI prompt log, and contribution documentation.

Use meaningful Markdown headings and concise explanatory text.

Every major processing step should state:
- What is being done.
- Why it is necessary.
- What changed.
- How the result was validated.

Avoid excessive explanation or repeated paragraphs.

## 7. Mandatory validation and testing

Create explicit checks for:

- Correct source schemas.
- Correct original dataset sizes for the recorded snapshot.
- Unexpected missing fields.
- Duplicate key handling.
- Correct analysis-year selection.
- Valid date parsing.
- Correct treatment of confidential records.
- Correct treatment of unavailable completion dates.
- Successful geographic assignment.
- Unambiguous ZIP lookup behavior.
- Unique aggregated integration keys.
- No accidental many-to-many joins.
- Preserved totals after aggregation.
- Correct post-integration types.
- No negative impossible counts.
- Reproducible calculations.
- Proper handling of missing populations and undefined rates.

For any failed validation, report a meaningful error describing which check failed and the likely reason.

Do not suppress exceptions simply to make the notebook appear successful.

If a known raw-data anomaly is intentionally retained, document it and keep it separate from a genuine
pipeline failure.

## 8. Visual and documentation quality

Use a consistent, professional visual style for notebook figures.

Charts should:
- Be easy to understand.
- Show meaningful labels.
- Use appropriate number formatting.
- Have readable legends.
- Include useful hover information.
- Avoid clutter.
- Explain the relevant insight or validation purpose.

Include tables and summary charts that make the effect of cleaning clear.

The notebook should show actual results rather than only code.

Include a short interpretation after each important finding, but do not invent conclusions before executing
the analysis.

## 9. Outputs and saved files

Deliver the improved notebook as an actual downloadable:

`01_data_engineering_and_visualization.ipynb`

Also generate the appropriate outputs after execution, including:

- Cleaned housing dataset.
- Cleaned SIRTA dataset.
- Integrated analytical dataset.
- Data-quality report.
- Data dictionary.
- Validation summary.
- Relevant output files required by the future Dash application.

Choose portable, clearly named file formats such as CSV, Parquet, or JSON where appropriate.

The notebook must run independently of my personal Google Drive unless I explicitly choose to save files
there.

Avoid unnecessary large binary files.

The final Dash application will be built in a later stage, so provide reusable analytical functions and export
formats that facilitate that implementation.

## 10. Final quality expectations

The improved notebook must:

- Be complete, not a collection of disconnected code snippets.
- Be executable from top to bottom on Google Colab.
- Use the real NYC Open Data.
- Preserve the approved datasets and study period.
- Satisfy the relevant official Milestone 1 notebook requirements.
- Include meaningful pre- and post-integration cleaning.
- Clearly document all important cleaning decisions.
- Preserve data integrity.
- Include tests and validation outputs.
- Be understandable to a university student who needs to explain the implementation.
- Avoid unnecessary dependencies and excessive complexity.
- Be ready to commit to GitHub after validation.

Do not fabricate successful execution logs.

If you cannot run the notebook on real data in your environment, explicitly state that limitation and provide
the complete notebook for execution in Colab.

## 11. What I want you to return

Return the following:

**Deliverable 1 — Improved Colab notebook**

The complete actual `.ipynb` file.

**Deliverable 2 — Improvement report**

A concise explanation of the problems found in the original notebook, the corrections made, and any unresolved
data issues.

**Deliverable 3 — Execution instructions**

Exact Google Colab instructions for running the notebook, checking the results, and saving the files.

**Deliverable 4 — Validation checklist**

A checklist identifying what has been verified, what requires the real Colab run, and whether any official
project requirements remain unimplemented.

**Deliverable 5 — Next-stage handover**

A short technical handover describing the resulting data files, metrics, and reusable functions that will be
needed for the full Plotly Dash website.

## Final instruction

Prioritize correctness, reproducibility, clean code, clear visualizations, understandable explanations, and
compliance with the original university instructions.

Retain the existing successful integration approach, fix the documented data-quality and methodological
issues, and improve the notebook into a polished analytical deliverable.

**Create the improved notebook now. Do not stop at another feasibility report, do not replace implementation
with pseudocode, and do not begin building the complete Dash website during this stage.**
~~~

#### Claude prompt 3 — Final comprehensive notebook review and improvement (2026-10-14)

~~~text
Attached: the executed notebook `01_data_engineering_and_visualization (1).ipynb`.

# FINAL COMPREHENSIVE NOTEBOOK REVIEW AND IMPROVEMENT
## GIU — Data Engineering and Visualization | Milestone 1

### Your role

You are a senior Python data engineer, statistical analyst, data visualization specialist, and university
project quality reviewer.

I am providing my **latest fully executed Google Colab notebook**, containing the Python code, real dataset
outputs, visualizations, research findings, validation results, and data-processing pipeline.

**Notebook:** `01_data_engineering_and_visualization (1).ipynb`

Your objective is to carefully inspect the actual code and saved execution results, identify and correct the
methodological and technical issues listed below, and deliver a complete, improved Google Colab notebook.

**This is a continuation of an existing, functioning project. Do not restart the project, choose different
datasets, or unnecessarily rewrite correct code.**

The notebook already runs successfully. The main objective is to improve analytical accuracy, statistical
validity, efficiency, clarity, reproducibility, and compliance with the university requirements.

---

# 1. Project information and objectives

**University:** German International University (GIU)

**Course:** Data Engineering and Visualization — Winter Semester 2026

**Milestone:** Project Milestone 1

**Submission deadline:** 9 November 2026

**Team size:** Four members

**Project title:** Affordable Housing Development and Road Safety in New York City

**Implementation environment:** Google Colab and Python

**Future website framework:** Plotly Dash

The project investigates relationships between affordable housing production and serious traffic crashes in
NYC.

The central question is:

**Do areas with more affordable housing production also experience higher serious traffic-crash rates, and how
does that relationship vary across locations, years, construction types, and income groups?**

The project must investigate meaningful associations rather than claim causation.

The official university PDFs are authoritative for the assignment requirements.

The project ultimately requires:

- Exactly one Housing & Development dataset and one Transportation dataset.
- Each dataset must have at least nine columns.
- Exploratory data analysis.
- Pre-integration cleaning.
- Meaningful integration of both datasets.
- Required post-integration cleaning.
- Eight research questions and eight visualizations for the four-member team.
- An interactive website with dropdown filters, search mode, and a central Generate Report button.
- Data storytelling, limitations, and relevant findings.
- GitHub repository, source code, README, documented AI prompts, and an actual contribution log.
- A deployed and tested website.

**For this stage, focus only on improving and finalizing the analytical notebook. The interactive website will
be developed afterward.**

---

# 2. Preserve the existing datasets and design

## Housing dataset

**Affordable Housing Production by Building (H2)**

Dataset ID: `hg8x-zxpr`

URL: https://data.cityofnewyork.us/d/hg8x-zxpr

## Transportation dataset

**Serious Injury Response, Tracking & Analysis Program (SIRTA, T4)**

Dataset ID: `xeqp-qz8h`

URL: https://data.cityofnewyork.us/d/xeqp-qz8h

## Supporting geographic reference

**MODZCTA**

Dataset ID: `pri4-ifjk`

URL: https://data.cityofnewyork.us/d/pri4-ifjk

MODZCTA is used as a geographic reference and population lookup, not as a third primary analytical dataset.
Instructor/TA permission remains pending.

## Existing analysis decisions

Preserve:

- Study period: 2022–2025.
- Exclusion of partial 2026 data from the main annual comparisons.
- Coordinate-first geographic assignment.
- MODZCTA × year integration.
- One-to-one validated joins.
- The existing eight research questions.
- Reusable Python analytical functions.
- Real-data extraction and saved snapshots.
- SHA-256 source manifests.
- Pre-integration and post-integration cleaning.
- Data-quality reports.
- Existing Plotly visualizations.
- Final data exports.

**Do not introduce geographic physical-size analysis, polygon-area measurement, or land-area comparisons.**

Geographic boundaries remain necessary for location assignment and mapping.

---

# 3. Actual execution results from the uploaded notebook

The notebook was executed on real NYC Open Data and saved with its outputs.

### Notebook information

- 121 total cells.
- 74 executed Python code cells.
- 69 cells containing saved outputs.
- No saved Python exceptions.
- 64 validation checks passed.
- 0 validation warnings.
- 0 validation failures.
- Snapshot date: 14 October 2026.

### Data information

| Metric | Observed result |
|---|---:|
| Original housing records | 9,551 |
| Original SIRTA records | 13,754 |
| SIRTA records excluded during cleaning | 445 |
| SIRTA retained after cleaning | 13,309 |
| Housing buildings completed in 2022–2025 | 2,957 |
| Housing buildings geographically assigned | 2,241 |
| Affordable units in geographic analytical panel | 100,057 |
| Serious crashes in 2022–2025 | 11,958 |
| Serious crashes geographically assigned | 10,777 |
| Serious crashes excluded from geographic analysis | 1,181 |
| Integrated panel rows | 708 |

These figures are observations from the previously executed snapshot, not values to hardcode into the final
notebook.

### Existing research findings

RQ1: Spearman correlation approximately +0.335.

RQ2: 44 geographic areas classified as relatively high in both housing production and serious-crash rates.

RQ3: Indexed local-street crash-count change from 2022 to 2025:
- No-completion group: +23.0%.
- Lower-half housing-production group: +17.3%.
- Upper-half group: −2.1%.

RQ4: Eight of 56 time/day comparisons exceeded the notebook's approximate two-standard-error threshold.

RQ5: Median crash rates:
- New-construction-dominant areas: 31.4.
- Preservation-dominant areas: 23.2.

RQ6: Unit-weighted area crash-rate averages:
- Extremely-low-income units: 32.6.
- Middle-income units: 42.4.

RQ7: Historical similar-crash metric shares:
- No-completion group: 4.3%.
- Lower-half group: 4.9%.
- Upper-half group: 8.4%.

RQ8: The leading geographic areas in the existing priority ranking include 10035, 11101, 10454, 10474, and
11217.

**These are previous outputs, not necessarily final validated conclusions.**

Inspect their formulas, filters, denominators, interpretation, and limitations. Revised methods may
legitimately change some results.

Never hardcode these research findings.

---

# 4. HIGH PRIORITY — Correct housing terminology and interpretation

### Identified problem

The housing dataset includes two major construction types:

1. New Construction.
2. Preservation.

The notebook sometimes describes both as new housing growth or new residents moving into an area.

That interpretation is potentially incorrect.

Preservation can involve rehabilitating or maintaining existing affordable housing. It does not necessarily
mean additional dwellings or population growth.

### Required improvements

1. Review the notebook's terminology throughout.
2. Replace misleading references to "new homes" or "housing growth" where the measure includes preservation.
3. Use accurate terminology such as "affordable housing production", "completed affordable units", or "new
construction and preservation".
4. Retain separate measures for New Construction and Preservation.
5. Clarify whether each chart uses all completed units or a specific construction type.
6. Update research-question descriptions, titles, legends, captions, findings, and conclusions.
7. Avoid claiming housing completions automatically represent new residents.
8. Preserve the source totals and original records.

Keep the established eight-question design while improving the wording and interpretation.

---

# 5. HIGH PRIORITY — Improve RQ1 and population normalization

### Identified problem

The notebook compares:

- Affordable units per resident.
- Serious crashes per resident.

Both use the same estimated 2018 population denominator.

The analytical period is 2022–2025.

This creates several limitations:

- Population figures may have changed since 2018.
- Both metrics share a denominator, which can introduce or amplify a statistical association.
- Busy commercial locations may have relatively few residents but high commuter traffic.
- Serious crashes per resident do not measure an individual's actual exposure to road traffic.

The existing correlation of approximately +0.335 should not be presented as evidence that housing production
causes higher crash rates.

### Required improvements

1. Explain the potential shared-denominator bias.
2. Preserve the normalized analysis.
3. Compare normalized measures with alternative raw-count comparisons.
4. Add a simple within-borough sensitivity analysis.
5. Check whether a small number of Manhattan or high-rate areas disproportionately influence the correlation.
6. Examine whether results differ when using New Construction alone instead of all housing production.
7. Preserve outliers unless there is a defensible reason for exclusion.
8. Report Spearman correlation accurately and label uncertainty appropriately.
9. Explain the limitations of using 2018 population estimates.
10. Clearly distinguish ecological association from individual exposure or causal effects.

Do not add unnecessarily sophisticated models.

**Goal:** Determine whether the observed association is reasonably stable under alternative descriptive
comparisons.

---

# 6. HIGH PRIORITY — Correct RQ3 temporal interpretation

### Identified problem

The notebook categorizes geographic areas based on housing completions over the entire 2022–2025 period.

It then compares annual crash trends starting in 2022.

This retrospective classification may include housing completed after the crashes occurred.

For example, housing completed in 2025 cannot explain crashes that happened in 2022.

### Required improvements

1. Clearly explain that the housing-production tiers are retrospective.
2. Avoid treating future housing completions as earlier exposure.
3. Preserve the existing annual line chart.
4. Preserve the annual location-assignment coverage information.
5. Do not claim that housing production caused crash increases or decreases.
6. State the limitations of having only four annual observations.
7. If useful and supported by the existing data, introduce a simple supplementary lagged comparison.
8. Keep the main RQ3 descriptive rather than causal.

Any additional comparison should be understandable and genuinely useful.

---

# 7. Improve RQ4 statistical reliability

### Identified problem

RQ4 evaluates 56 weekday/time-period combinations and identifies eight unusual differences using an
approximate two-standard-error rule.

With many simultaneous comparisons, some apparently unusual findings can arise by chance.

### Required improvements

1. Retain the heatmap.
2. Clearly label the analysis as exploratory.
3. Review whether percentage differences are calculated consistently.
4. Ensure the appropriate denominator is used.
5. Check small-sample time/day categories.
6. Avoid describing the eight flagged cells as established statistical discoveries.
7. Either adopt a defensible multiple-comparison approach or remove significance-style annotations.
8. Explain any remaining uncertainty in simple language.

Do not introduce advanced statistical complexity unless necessary.

---

# 8. Improve RQ5 construction-type analysis

Review the comparison between New Construction and Preservation.

Verify:

- Construction-type labels.
- Definition of a construction-type-dominant area.
- Classification thresholds.
- Number of areas in each category.
- Calculation of median crash rates.
- Handling of areas with mixed housing production.
- Outlier effects.
- Population denominators.
- Interpretation of fatality-related metrics.

Clarify that the comparison describes area-level relationships, not necessarily changes in residents, traffic
demand, or crash causation.

Retain a clear and interpretable box plot.

---

# 9. Improve RQ6 income-band analysis

### Identified problem

RQ6 compares unit-weighted crash-rate averages across housing income bands.

However, those figures describe the crash-rate characteristics of geographic areas where different categories
of units were completed.

They do not directly measure the crash exposure or injuries of the occupants of those units.

Also, overlapping confidence intervals are not a sufficient method for establishing whether two income-band
estimates differ.

### Required improvements

1. Verify all income-band unit totals and weighted-average calculations.
2. Correctly handle structural zeros and unknown unit counts.
3. Retain the comparative bar chart.
4. Clarify that the results describe locations of affordable units, not individual residents' risks.
5. If testing differences, use an appropriate direct comparison, such as an area-level paired bootstrap.
6. Preserve dependence when resampling areas.
7. Report uncertainty without overstating significance.
8. Avoid claiming one income group is demonstrably less safe based only on the observed averages.
9. Explain the results in simple, defensible terms.

---

# 10. Improve RQ7 historical crash interpretation

### Identified problem

RQ7 relies on SIRTA historical similar-crash fields, including `similar_fatalities_severe`.

The exact meaning of "same type" and "same location" requires verification against the official SIRTA data
dictionary.

### Required improvements

1. Verify the field's definition using the official source documentation if available.
2. Confirm whether it represents similar injuries, similar crash events, or a more specific DOT-defined
measure.
3. Avoid presenting the field as direct proof of repeated collisions at identical coordinates unless
supported.
4. Preserve missing look-back values as unknown.
5. Do not replace unknown values with zeros.
6. Correct chart titles and labels where appropriate.
7. Document limitations in the three-year historical look-back measure.
8. Preserve the question's cross-domain relationship.

If the definition remains uncertain, use appropriately qualified terminology rather than inventing certainty.

---

# 11. Improve RQ8 priority rankings

### Identified problem

RQ8 combines unfinished housing projects with historical serious-crash rates to create a priority ranking.

However:

- Missing completion dates do not prove a project is close to occupancy.
- Preservation projects do not necessarily add new residents.
- A descriptive ranking does not prove that a particular road intervention is required.
- The priority score is not a validated prediction.

### Required improvements

1. Explain the pipeline definition clearly.
2. Verify pipeline unit calculations and inclusion rules.
3. Clarify how New Construction and Preservation are treated.
4. Explain the percentile-based priority score.
5. Ensure ranking direction is correct.
6. Handle tied scores consistently.
7. Avoid describing the ranking as predictive.
8. Frame top-ranked areas as candidates for further investigation.
9. Retain the ranked visualization and useful hover information.

---

# 12. Improve data-quality handling and transparency

The notebook runs successfully, but unresolved source-data anomalies remain.

### Housing findings

- 1,949 confidential records.
- 20 records with unusual completion-before-project-start ordering.
- 123 building IDs appearing in multiple projects.
- Some missing counted-unit totals.
- Location and ZIP-code discrepancies.

### SIRTA findings

- 444 orphan rows.
- Three exact duplicate rows within the orphan group.
- One repeated collision ID, 4597666, with inconsistent date and fatality information.
- Five additional suspicious crash dates.
- Missing geographic coordinates and ZIP codes.
- Variable geographic completeness by year.
- Some records that cannot be assigned to a geographic area.

### Required improvements

1. Preserve the existing cleaning and exclusion logs.
2. Recheck the mathematical consistency of excluded-record counts.
3. Never automatically delete suspicious but potentially valid records.
4. Do not silently resolve uncertain fatality values.
5. Keep confidential housing contributions in appropriate totals.
6. Distinguish missing from genuine zeros.
7. Document the date anomalies clearly.
8. Review all geographic discrepancies.
9. Ensure excluded records do not disappear from audit reports.
10. Provide clear explanations for unresolved cases.

### Important validation improvement

The previous notebook reported:

- 64 PASS.
- 0 WARN.
- 0 FAIL.

This shows that programmed checks passed, but it does not mean the raw datasets have no unresolved anomalies.

Provide separate summaries for:

**A. Pipeline validation**
- PASS.
- WARN.
- FAIL.

**B. Source-data limitations**
- Unresolved inconsistencies.
- Excluded records.
- Unassigned locations.
- Uncertain observations.
- Potential effects on analysis.

Do not turn a correctly handled source-data limitation into an artificial technical failure.

---

# 13. Improve geographic coverage and reliability

The executed notebook reports different crash-location assignment rates across years:

| Year | Assignment rate |
|---|---:|
| 2022 | 88.82% |
| 2023 | 89.39% |
| 2024 | 87.58% |
| 2025 | 94.84% |

These differences can affect comparisons of crash counts across years.

### Required improvements

1. Retain coordinate-first geographic matching.
2. Preserve ZIP-code parsing and validation.
3. Keep deterministic handling of ambiguous assignments.
4. Quantify how many records remain unassigned.
5. Display annual assignment coverage alongside temporal trends.
6. Explain that unassigned crashes can bias geographic comparisons.
7. Preserve borough/year assignment-coverage checks.
8. Treat coverage-adjusted estimates as sensitivity scenarios, not guaranteed corrections.
9. Document the assumptions behind each coverage adjustment.
10. Keep the location matching reproducible.

Do not introduce geographic physical-size analysis.

---

# 14. Improve the research findings and overall data story

The notebook currently produces research findings from the calculated data.

Preserve this functionality, but review all generated conclusions.

Correct unsupported or overly strong statements about:

- Housing causing traffic crashes.
- Every housing completion representing newly created dwellings.
- Housing completions necessarily bringing new residents.
- Low-income occupants directly experiencing the reported area crash rates.
- Historical similar-crash fields proving exact collision recurrence.
- Priority rankings proving that a safety redesign is necessary.
- Four years of data establishing long-term trends.
- Population normalization removing all biases.

The conclusions must accurately distinguish:

1. What the data actually show.
2. What the analysis suggests.
3. What remains uncertain.
4. What cannot be concluded.
5. How NYC planners could use the information responsibly.

Do not create stronger findings simply to make the project appear more impressive.

A weak association or a null result is scientifically acceptable.

---

# 15. Code quality and efficiency

The current notebook runs successfully. Do not unnecessarily rewrite it.

Review for:

- Repeated operations.
- Unnecessary DataFrame copies.
- Duplicate grouping calculations.
- Inefficient geographic joins.
- Repeated loading of the same files.
- Unused imports.
- Unnecessary dependencies.
- Expensive calculations that provide little analytical benefit.
- Repeated plotting logic.
- Fragile Pandas nullable-value comparisons.
- Missing deterministic random seeds.
- Hardcoded assumptions that should be documented.
- Functions that are difficult to reuse in the upcoming Dash website.

### Required improvements

- Keep descriptive variable names.
- Use clear modular functions.
- Maintain readable and concise Markdown.
- Minimize dependencies.
- Preserve compatibility with a standard Google Colab CPU runtime.
- Avoid unnecessary statistical models.
- Make calculations reproducible.
- Retain the existing project folder and export structure where possible.

Prioritize maintainability and correctness over adding more code.

---

# 16. Strengthen mathematical and analytical validation

The existing validations mainly establish structural consistency and successful processing.

Add focused tests that verify analytical calculations.

Check:

1. Housing counts and construction-type aggregation.
2. Correct handling of preservation.
3. Rate denominators.
4. Number of years in annual-average calculations.
5. Unique geographic/year integration keys.
6. No accidental many-to-many joins.
7. RQ1 correlation calculations.
8. RQ2 classification thresholds.
9. RQ3 indexed trend calculations.
10. RQ4 percentage differences.
11. RQ5 construction-type classifications.
12. RQ6 unit-weighted averages.
13. RQ7 handling of unknown historical similarity values.
14. RQ8 percentile ranking.
15. Correct filtering by borough, year, construction type, and crash scope.

Use small, controlled test datasets for known-answer calculations where useful.

Do not hardcode the previously observed real-data outputs as the expected answers to every test.

---

# 17. Improve visual quality

Review all Plotly visualizations.

The notebook includes exploratory charts and eight research visualizations.

Keep the figures professional, readable, and consistent.

Improve where needed:

- Titles.
- Axis labels.
- Legends.
- Hover information.
- Number formatting.
- Percentage formatting.
- Readable text.
- Appropriate chart dimensions.
- Explanation of normalization.
- Display of uncertainty.
- Empty or sparse categories.
- Color consistency.

Avoid unnecessary charts that duplicate information.

Make each research figure clearly answer its corresponding question.

Do not change working charts purely for decoration.

---

# 18. Submission requirements and documentation

Preserve or improve:

- Dataset overview.
- Dataset-selection rationale.
- Exploratory analysis.
- Pre-integration cleaning.
- Data integration.
- Post-integration cleaning.
- Validation tables.
- Research visualizations.
- Data story.
- Methodological limitations.
- Source links.
- AI prompt documentation.
- Contribution tracking.
- Reproducibility instructions.
- Export files.
- Data dictionary.

### AI prompt log

The university requires documentation of prompts used in the project.

Include the actual prompt text wherever available.

If a complete previous prompt is unavailable, mark it as missing rather than inventing it.

### Four-member contribution documentation

Retain four member sections.

Do not falsely assign work performed by one person to other members.

Allow actual member names, IDs, research questions, code contributions, and reviews to be documented
accurately.

---

# 19. Deliver the complete final notebook

Create an actual downloadable file:

`01_data_engineering_and_visualization_FINAL.ipynb`

It must include:

1. Complete Python code.
2. Required package installation.
3. Real NYC dataset extraction.
4. Dataset overview and EDA.
5. Documented preprocessing.
6. Data cleaning.
7. Geographic assignment.
8. Data integration.
9. Required post-integration cleaning.
10. Analytical validation.
11. Eight research visualizations.
12. Data storytelling and interpretations.
13. Sensitivity analysis.
14. Data-quality summary.
15. Reproducible outputs.
16. Export functionality.
17. README/website handover information.
18. AI prompt and contribution documentation.

The code must execute from top to bottom on Google Colab.

Do not supply only pseudocode or fragmented code snippets.

---

# 20. Required additional deliverables

Alongside the notebook, provide:

### A. Detailed but concise improvement report

Include a table containing:

- Existing problem.
- Why it matters.
- Exact correction made.
- Functions or sections modified.
- Whether numerical findings may change.
- Remaining limitations.

Separate technical corrections from methodological and presentational improvements.

### B. Google Colab execution instructions

Explain exactly how I should:

1. Upload the notebook.
2. Use a fresh CPU runtime.
3. Run all cells.
4. Confirm real-data mode.
5. Review validation results.
6. Inspect the eight research charts.
7. Save the executed notebook with outputs.
8. Export the processed datasets and project ZIP archive.

### C. Validation report

Explain which tests you have actually performed and which still require execution in Google Colab.

### D. Website implementation handover

List:

- Processed datasets.
- Required CSV, Parquet, or JSON files.
- Map data.
- Analytical functions.
- Research-chart functions.
- Relevant filter parameters.
- Metric definitions.
- Data-quality limitations to display on the website.

The handover must make it straightforward to reuse the notebook's calculations in the future Dash application.

---

# 21. Critical rule — Do not fabricate execution results

The notebook I provide contains real execution outputs.

If you modify calculations, previous outputs may become stale.

Do not reuse old outputs as if they were generated by the revised code.

If you cannot execute the revised notebook on real NYC Open Data:

1. Clearly disclose that limitation.
2. Deliver the complete revised notebook.
3. Clear the execution outputs affected by changed calculations.
4. Retain the original executed notebook as the baseline.
5. Identify which numerical results need to be recomputed.
6. Do not manufacture successful tests, charts, or validation counts.
7. Instruct me to run the final notebook on Google Colab.

The notebook must never silently substitute synthetic data for real NYC records.

---

# 22. Final implementation requirements

I want a final analytical notebook that is:

- Technically correct.
- Scientifically defensible.
- Efficient.
- Reproducible.
- Easy to understand.
- Well documented.
- Clear in its visualizations.
- Transparent about limitations.
- Consistent with the original GIU assignment.
- Suitable for a four-member project.
- Ready to support the upcoming interactive Dash website.

**Preserve everything that works correctly and modify only what materially improves the project.**

My goal is not to make the notebook unnecessarily complicated. My goal is to produce a reliable and
understandable university project with strong research quality.

### What I want you to do now

1. Read the original project instructions.
2. Inspect the entire supplied executed notebook, including saved outputs.
3. Review every identified limitation above.
4. Correct the genuine problems.
5. Improve methodological interpretation and efficiency.
6. Add focused analytical validation.
7. Create the complete improved `.ipynb` file.
8. Provide the improvement report and Colab execution instructions.
9. Prepare the data-processing and analytical handover for the future Dash website.

**Do not stop at a feasibility review or theoretical recommendations. Perform the notebook improvements and
deliver the final file.**

**Do not develop the complete Dash website during this stage.**

**Do not restart dataset selection.**

**Do not invent results or member contributions.**

**Return the complete final notebook ready for real-data testing in Google Colab.**
~~~

#### Claude prompt 4 — Final project implementation: simplified notebook, Dash website and repository (2026-10-15)

~~~text
# FINAL PROJECT IMPLEMENTATION — STRICT COMPLIANCE WITH GIU PROJECT DESCRIPTION

## 1. Your role and main objective

You are an experienced Python developer and data visualization engineer helping me complete my university Data
Engineering and Visualization project.

I am providing three files:

1. **Project Description.pdf** — The official university assignment and grading requirements.
2. **Data Catalog.pdf** — The official recommended datasets and integration information.
3. **01_data_engineering_and_visualization_FINAL(1).ipynb** — My latest existing Python/Google Colab project
notebook.

**Your most important instruction is to implement exactly what the Project Description PDF requires, without
unnecessary additional functionality, advanced statistical techniques, or research complexity.**

The Project Description PDF is the authoritative source. If anything in the existing notebook is more
complicated than necessary, simplify it.

Do not sacrifice correctness, meaningful analysis, data quality, or any explicitly required deliverable merely
to reduce code length.

My goal is a complete, correct, understandable project that satisfies the assignment and that I can
confidently explain during evaluation.

## 2. Project information

University: German International University (GIU)

Course: Data Engineering and Visualization — Winter Semester 2026

Milestone: Project Milestone 1

Deadline: 9 November 2026

Project title: Affordable Housing Development and Road Safety in New York City

Implementation: Python, Google Colab, Plotly and Dash

Version control: GitHub

Deployment: A suitable free hosting platform, preferably Render.

### Contributor information

Name: **Abdelrahman El-sayed**

Student ID: **13004550**

I am currently performing and coordinating the project work myself.

For contribution documentation:

- Record Abdelrahman El-sayed (13004550) as the responsible contributor for the project work I undertake,
including the eight research-question sections, data processing, notebook execution/review, and website
implementation.
- Clearly document AI assistance in code generation and analysis.
- Do not invent other contributors, names, IDs, or work.
- Keep the three other team-member entries marked "Not yet assigned / contribution not provided."
- Do not falsely claim that the assignment's requirement for each of four members to propose two research
questions has been fulfilled.
- Clearly identify this as an administrative requirement that needs instructor clarification if the project is
being completed individually.

Do not fabricate a team contribution history.

## 3. First task — Review the official PDF against the notebook

Before modifying the existing project, examine the university's Project Description PDF carefully.

Create a requirement-by-requirement comparison with three classifications:

**A. Explicitly required by the PDF**

Must be implemented and preserved.

**B. Necessary supporting implementation**

Not necessarily mentioned by name in the PDF, but needed for correct data cleaning, integration,
reproducibility, or accurate research conclusions.

**C. Unnecessary additional complexity**

Advanced features that do not meaningfully help satisfy the assignment and can be removed or simplified.

For each requirement, identify:

- Relevant PDF requirement.
- Relevant existing notebook section or function.
- Whether it is already satisfied.
- Whether modification is needed.
- What you propose to keep, simplify, remove, or implement.

Do not introduce your own extra university requirements.

## 4. Preserve the selected datasets

The datasets are already selected and should not be changed.

### Housing

Affordable Housing Production by Building (H2)

Dataset ID: `hg8x-zxpr`

URL: https://data.cityofnewyork.us/d/hg8x-zxpr

### Transportation

Serious Injury Response, Tracking & Analysis Program (SIRTA, T4)

Dataset ID: `xeqp-qz8h`

URL: https://data.cityofnewyork.us/d/xeqp-qz8h

### Geographic reference

MODZCTA

Dataset ID: `pri4-ifjk`

URL: https://data.cityofnewyork.us/d/pri4-ifjk

MODZCTA is used only for geographic matching, mapping and population reference, not as a third primary
analytical domain.

Its acceptability under the exactly-two-datasets rule has not yet been confirmed by the TA. Preserve that
limitation in the documentation.

### Existing analysis scope

- Study period: 2022–2025.
- Integration: Geographic area × year.
- Housing data: New Construction and Preservation.
- Transportation data: Serious crashes reported through SIRTA.

Do not include 2026 in full-year annual comparisons.

Do not add physical geographic-area measurement or polygon-size analysis.

## 5. Remove unnecessary advanced statistics

The existing notebook became more statistically complicated than the official project requires.

Review the code and simplify or remove these features where they are not necessary to answer a research
question correctly:

- Repeated bootstrap confidence intervals.
- Partial correlation modelling.
- Multiple alternative correlation scenarios.
- Leave-one-out influence analysis.
- Excessive borough-by-borough statistical testing.
- Holm-adjusted significance testing.
- Paired bootstrap inference between housing income bands.
- Wilson confidence intervals.
- Repeated robustness charts.
- Extra exploratory models not required by the assignment.
- Statistical calculations that add little interpretive value.

Do not remove fundamental descriptive statistics or comparisons necessary for valid analysis.

For example, a clear scatter plot with a descriptive correlation and appropriate limitations is usually
preferable to several advanced statistical procedures that are difficult to understand and unnecessary for the
project.

Prioritize:

- Counts.
- Percentages.
- Means and medians.
- Descriptive statistics.
- Appropriate correlations when useful.
- Time trends.
- Categorical comparisons.
- Simple normalization when justified.
- Clear, appropriately labelled visualizations.

Use additional statistical complexity only when a simpler approach would lead to a materially misleading
answer.

Do not introduce machine learning or predictive modelling.

## 6. Preserve all required data engineering work

The university PDF explicitly requires data exploration, cleaning, integration, and post-integration cleaning.

These must remain complete.

### A. Data loading

- Load the two NYC Open Data datasets correctly.
- Record dataset IDs and source links.
- Display row and column counts.
- Show sample records and column information.
- Avoid incomplete downloads or accidental API row limits.
- Run in Google Colab.

### B. Exploratory Data Analysis

For both datasets:

- Inspect shape and structure.
- Identify numerical and categorical columns.
- Use descriptive statistics.
- Examine missing values.
- Investigate outliers.
- Identify duplicates.
- Examine relevant dates and geographic fields.
- Create appropriate initial visualizations.

Keep the EDA thorough enough to meet the rubric without filling the notebook with redundant charts.

### C. Pre-integration cleaning

Implement the required cleaning:

- Missing-value handling.
- Justified deletion or imputation.
- Duplicate detection and removal where appropriate.
- Outlier detection and treatment.
- Inconsistent-data checks.
- Date formatting.
- Geographic-field formatting.
- Data-type standardization.
- Category and string standardization.

Explain important cleaning decisions in Markdown.

Preserve valid extreme observations instead of automatically deleting them.

### D. Dataset integration

Keep the existing justified area/year integration strategy.

- Aggregate the two datasets to compatible levels.
- Prevent incorrect many-to-many joins.
- Preserve correct housing and crash totals.
- Document join keys and aggregation.
- Explain why the integration is meaningful.

Avoid unnecessary redesign of the existing integration pipeline.

### E. Mandatory post-integration cleaning

This is explicitly required by the project PDF.

After integration:

- Handle missing values created by joins.
- Resolve redundant or inconsistent columns.
- Correct data-type mismatches.
- Check integration integrity.
- Verify that records or totals have not been incorrectly multiplied.

Do not remove this section.

## 7. Preserve essential data-quality protections

The existing notebook successfully handles several actual data-quality problems.

Keep the necessary logic for:

- Confidential housing records.
- Genuine duplicates.
- Missing crash locations.
- Invalid ZIP values.
- Inconsistent or suspicious dates.
- Missing numeric values.
- Geographic assignment reliability.
- Uncertain collision records.
- Inconsistent construction categories.
- Missing building completion dates.

Do not fabricate values or silently delete uncertain records.

Keep concise audit tables showing what changed during cleaning.

Avoid excessively detailed technical reporting that is not necessary to explain the cleaning.

Retain important validation checks, particularly those preventing incorrect totals, invalid joins, and broken
research calculations.

Simplify repetitive tests and reports where appropriate.

## 8. Keep exactly eight main research questions

The assignment requires eight research questions and eight corresponding visualizations for a four-member
team.

Keep the following eight themes, provided the actual data support them.

### Section A — Spatial relationship

**RQ1:** Is affordable housing production associated with serious traffic-crash rates across NYC areas?

Visualization: Scatter plot.

**RQ2:** Which areas have both relatively high affordable housing production and relatively high serious-crash
rates?

Visualization: Map.

### Section B — Time patterns

**RQ3:** How did serious crashes change between 2022 and 2025 across areas with different levels of affordable
housing production?

Visualization: Line chart.

**RQ4:** How do crash times and weekdays differ between areas with different housing-production levels?

Visualization: Heatmap.

### Section C — Housing characteristics

**RQ5:** Do areas dominated by new construction and preservation show different serious-crash patterns?

Visualization: Box plot.

**RQ6:** Are affordable housing units in different income categories located in areas with different
serious-crash rates?

Visualization: Comparative bar chart.

### Section D — Road safety priorities

**RQ7:** Do areas with greater affordable housing production also have higher shares of crashes with reported
historical similar serious-injury or fatality events?

Visualization: Stacked bar chart.

Use the actual SIRTA field definition. Do not claim precise collision recurrence unless documented.

**RQ8:** Which areas combine unfinished affordable housing projects with relatively high serious-crash rates?

Visualization: Ranked bar chart.

Frame the result as a descriptive prioritization for further investigation, not a prediction.

### Requirements for the eight visualizations

Each chart must:

- Directly answer its research question.
- Include a meaningful connection between housing and transportation.
- Use appropriate labels and units.
- Have readable titles and legends.
- Display accurate numerical values.
- Include useful hover information.
- Be understandable to a student explaining it during evaluation.

The eight research visualizations must be individually identifiable.

Additional charts may be retained only where needed for EDA or cleaning validation, as the PDF explicitly
requires those.

Do not add extra research questions.

## 9. Simplify the interpretation of results

The university expects honest findings, limitations, and a meaningful data story.

Review all generated conclusions and ensure:

- Housing production is not described as causing crashes.
- Correlation is not presented as causation.
- Preservation is distinguished from newly built housing.
- Population-based rates are correctly labelled.
- Missing geographic data are acknowledged.
- The limited 2022–2025 period is explained.
- Historical population estimates are recognized as a limitation.
- Descriptive differences are not exaggerated as statistically proven discoveries.
- The income-band analysis describes geographic patterns, not personal injury risk.
- Priority rankings are presented as screening tools rather than predictions.

Keep the explanations clear and concise.

Do not add unnecessary statistical theory to the notebook.

## 10. Simplify the notebook structure

The existing notebook has approximately 128 cells and contains a substantial amount of Python code.

Do not target a particular number of cells or lines. Instead, remove unnecessary complexity while ensuring
complete functionality.

I recommend a clear structure:

1. Project introduction and objectives.
2. Dataset descriptions and selection.
3. Python environment and data loading.
4. Exploratory Data Analysis — Housing.
5. Exploratory Data Analysis — Transportation.
6. Pre-integration cleaning — Housing.
7. Pre-integration cleaning — Transportation.
8. Data integration.
9. Post-integration cleaning.
10. Research questions and eight visualizations.
11. Findings and data story.
12. Limitations.
13. Validation and saved project outputs.
14. AI prompt documentation and team contributions.

Sections may be reorganized if necessary, but the final notebook should be straightforward to read from top to
bottom.

Each important stage should explain:

- What is happening.
- Why it is necessary.
- What the outcome means.

Use concise Markdown, descriptive names and relevant code comments.

Avoid repeating the same explanations or calculations.

## 11. Build the website exactly as the PDF requires

The website is a mandatory part of Milestone 1, not an optional enhancement.

Use **Python Plotly Dash** to avoid an unnecessarily complex frontend/backend architecture.

Implement the following requirements.

### A. Multiple dropdown filters

Provide meaningful dropdown filters, such as:

- Borough.
- Year.
- Housing construction type.
- Relevant transportation category or crash measure.

All filters must work on the real integrated data.

### B. Text search

Users must be able to type queries such as:

"Brooklyn 2023 new construction"

The website should interpret supported keywords and apply the appropriate filters.

A simple rule-based parser is sufficient.

Do not add an AI chatbot, natural-language model, or complicated search engine.

### C. Generate Report

Include a prominent **Generate Report** button.

Clicking it must update the research visualizations consistently using the selected filters or search terms.

The button must perform a real action, not merely exist visually.

### D. Eight interactive charts

Reuse the eight research visualizations and analytical calculations from the notebook.

Ensure charts support useful interactivity such as hover and filter updates.

### E. Data story

Include a short, readable section explaining:

- The main research question.
- Which datasets were selected.
- How they were integrated.
- The main findings.
- The limitations.
- Why the findings may matter to NYC.

### F. Website layout

Build a clean, professional and responsive dashboard.

Avoid unnecessary:

- Animations.
- Login systems.
- User accounts.
- External databases.
- AI chatbots.
- Complex multi-page navigation.
- Decorative components with no analytical purpose.

A simple single-page Dash application is acceptable if it fully satisfies the PDF.

### G. Hosting and deployment

Prepare the website for deployment to a suitable free hosting service, preferably Render.

Ensure the application:

- Runs independently of Google Colab.
- Uses available processed data.
- Does not depend on my personal Google Drive.
- Does not download the original datasets every time a user opens the website.
- Includes all necessary dependencies and configuration.

Provide exact deployment instructions.

Do not claim the website has been deployed or tested publicly unless that has actually happened.

## 12. GitHub and deliverables

Prepare a complete GitHub-ready project.

Include the minimum appropriate structure:

- `notebooks/` — Final Google Colab notebook.
- `data/` — Required cleaned and integrated data files.
- `src/` — Reusable functions where appropriate.
- `app.py` — Dash application.
- `requirements.txt` — Dependencies.
- `README.md` — Setup, usage and deployment.
- `docs/AI_PROMPT_LOG.md` — Actual AI prompts used.
- Contribution documentation.
- Any deployment configuration required by the chosen hosting platform.

Avoid unnecessary folders and files.

The README must explain how to:

1. Run the Colab notebook.
2. Understand the selected datasets.
3. Execute the website locally.
4. Deploy the website.
5. Access the data sources.
6. Understand the actual contributions to the project.

The final submission must be consistent with the original PDF.

## 13. Contributor documentation

Use the following identity:

**Abdelrahman El-sayed**

**Student ID: 13004550**

I am the person currently coordinating and carrying out the implementation.

Document my actual work accurately, including the research questions and visualizations, data preparation,
integration, testing, and website work as each part is undertaken and verified.

Record AI assistance transparently.

For the other three expected team members:

- Do not invent names.
- Do not invent IDs.
- Do not assign them work they did not perform.
- Keep placeholders until genuine contributor information is available.

The official assignment expects each team member to propose two research questions and visualizations. Clearly
flag the outstanding team-structure requirement rather than concealing it.

Do not claim that four students contributed when only one person's contribution has been documented.

## 14. AI prompt log

Preserve the existing AI prompt log and append this implementation prompt to it.

The official PDF requires prompts used in the project.

Record:

- Date.
- Tool used.
- Purpose.
- Exact prompt text where available.
- Files or components produced.
- Whether the outputs were reviewed and tested.

Do not fabricate missing prompts or testing records.

## 15. Validation expectations

Use straightforward tests to confirm that:

- Both datasets load correctly.
- Cleaning executes successfully.
- Required data fields exist.
- Invalid joins do not occur.
- Aggregated totals remain consistent.
- Eight research charts can be generated.
- The website starts successfully.
- Dropdown filters work.
- Text search applies the correct filters.
- Generate Report updates the charts.
- Empty filter results are handled appropriately.
- The website does not display misleading numerical conclusions.

The purpose is to confirm correct behavior, not to create a large testing framework beyond the assignment.

## 16. How to implement the modifications

Work directly from the existing final notebook.

Follow this order:

### Stage A — Audit and simplify

Compare the notebook with the official project requirements.

Remove unnecessary statistical complexity and redundant features.

Preserve all correct required data engineering work.

### Stage B — Produce the simplified final notebook

Deliver the revised `.ipynb` with the necessary EDA, cleaning, integration, eight research visualizations,
findings and limitations.

### Stage C — Build the required Dash application

Reuse the analytical outputs and functions rather than implementing contradictory calculations.

Provide complete source code and configuration.

### Stage D — Prepare the final repository

Organize the notebook, application, processed data, documentation and deployment instructions.

### Stage E — Verify compliance

Produce a short checklist referencing each official project requirement and the file or component satisfying
it.

Mark anything not actually tested or completed as pending.

Do not stop after writing another theoretical project plan. Produce the requested files and code.

## 17. Required output from you

Deliver:

**1. Simplified final Colab notebook**

`01_data_engineering_and_visualization_SIMPLIFIED.ipynb`

**2. Complete Dash website source code**

Including the required filters, search functionality, Generate Report button, eight charts and story section.

**3. GitHub-ready project files**

Include only files necessary for the assignment, execution, reproducibility and deployment.

**4. README and AI prompt log**

With correct project information and truthful contribution details.

**5. Change report**

A concise table showing:

- What was removed.
- What was simplified.
- What was preserved.
- Why each decision was made.
- Whether numerical results or research conclusions may change.

**6. PDF compliance checklist**

For every mandatory requirement, identify:

- Requirement.
- Implementing file or section.
- Completion status.
- Testing status.

**7. Step-by-step execution and deployment instructions**

Explain how I can run the notebook in Google Colab, save its outputs, test the website locally, publish the
repository and deploy the website.

## 18. Important rules

- Do not change the selected datasets.
- Do not invent findings.
- Do not invent team contributions.
- Do not add unnecessary advanced statistical techniques.
- Do not introduce machine learning or prediction.
- Do not add extra research questions.
- Do not remove required EDA or cleaning visualizations.
- Do not remove post-integration cleaning.
- Do not remove the eight required charts.
- Do not omit the website or Generate Report functionality.
- Do not create decorative website features unrelated to the assignment.
- Do not claim to have executed tests that you did not run.
- Do not reuse stale numerical results as if they were produced by revised code.

If real-data execution is unavailable in your environment, deliver complete runnable code and explicitly
identify which results require my Google Colab execution.

Keep the project understandable, maintainable and academically defensible.

## FINAL INSTRUCTION

**Implement the simplest complete version of the project that satisfies every mandatory requirement in the GIU
Project Description PDF.**

Reuse the working notebook, remove unnecessary additional research complexity, keep the required data
engineering and eight visualizations, implement the required interactive Dash website, and prepare the
complete submission files.

Prioritize correctness, clarity and full assignment compliance over sophistication.

Do not make the project more advanced merely for the sake of adding features.

**Deliver the complete implementation and a concise explanation of how every requirement in the PDF has been
satisfied.**
~~~

#### Claude prompt 5 — Final revision: comments, documentation and borough correction (2026-10-16)

~~~text
# FINAL REVISION — SIMPLIFIED GIU DATA ENGINEERING PROJECT

## 1. Your role and objective

You are a Python developer and data engineering project reviewer helping me finalize my university project.

I am providing my latest **SIMPLIFIED Google Colab notebook**, which you previously generated and which I have
now executed successfully on real NYC Open Data.

**Important: Work directly from the simplified notebook I am attaching. This is your authoritative starting
point.**

Do not work from the older FINAL notebook, earlier feasibility notebooks, or previous packages.

The current simplified notebook is already functioning and largely satisfies my university's Project
Description PDF.

I do not want a new implementation, a redesign, additional research questions, or more advanced functionality.

Instead, I want you to make a focused set of revisions involving:

1. Humanizing Python comments throughout the notebook.
2. Making Markdown explanations sound natural and understandable.
3. Reorganizing and improving the project development and contribution documentation.
4. Documenting the actual roles of ChatGPT, Claude, and myself.
5. Correcting one verified data-reconciliation issue.
6. Updating the notebook's documentation to reflect its successful real-data execution.
7. Completing the required data-story explanation using verified results.
8. Preserving the existing project requirements and functionality.

**The highest priority is the writing style, particularly the Python code comments.**

---

# 2. Project information

University: German International University (GIU)

Course: Data Engineering and Visualization — Winter Semester 2026

Project: Milestone 1

Deadline: 9 November 2026

Project title: Affordable Housing Development and Road Safety in New York City

Student responsible for the current implementation:

**Abdelrahman El-sayed**

**Student ID: 13004550**

Environment: Google Colab, Python, Plotly, Dash and GitHub.

Selected datasets:

- H2 — Affordable Housing Production by Building.
- T4 — Serious Injury Response, Tracking & Analysis Program (SIRTA).

Analysis period: 2022–2025.

The project has eight research questions and eight corresponding visualizations.

The official Project Description PDF remains authoritative.

Do not introduce functionality that is not required or necessary for a correct implementation.

---

# 3. HIGHEST PRIORITY — Humanize all Python code comments

I want you to review **every Python code cell** in the simplified notebook and improve the code comments.

The existing comments sometimes sound overly technical, formal, or automatically generated.

I want the comments to read like clear, normal explanations written for a university data engineering project.

They should be understandable to me so I can explain the code myself during evaluation.

## Required writing style

Use:

- Simple English.
- Short, natural sentences.
- Direct explanations.
- Familiar Python terminology.
- Concise comments describing important decisions.
- A consistent student-level writing style.

Avoid:

- Overly formal academic language.
- Unnecessary advanced terminology.
- Long paragraphs inside code.
- Repeating what an obvious Python statement already does.
- Comments that sound like AI-generated documentation.
- Unnecessary decorative separators.
- Excessive capitalization.
- Repetitive explanations.
- Comments describing the project as unusually sophisticated.
- Claims that I personally authored AI-generated code.

## Examples

### Example 1 — Loading data

Before:

```python
# Initialize the comprehensive data acquisition procedure
# to retrieve and validate authoritative source datasets.
```

After:

```python
# Download the two datasets from NYC Open Data.
# We will check the columns and number of rows before cleaning.
```

### Example 2 — Missing values

Before:

```python
# Apply conditional null-value remediation based on
# semantic interpretation and domain-specific constraints.
```

After:

```python
# Check the missing values.
# Only fill a value when we know what it should be.
```

### Example 3 — Integration

Before:

```python
# Construct the unified analytical panel through
# validated geographic-temporal aggregation.
```

After:

```python
# Combine the housing and crash data by area and year.
# This lets us compare both datasets at the same level.
```

### Example 4 — Duplicates

Before:

```python
# Execute duplicate-record identification and
# systematic exclusion while preserving data integrity.
```

After:

```python
# Find duplicate records and remove the ones
# that represent the same observation.
```

### Example 5 — Validation

Before:

```python
# Ensure preservation of aggregate metrics throughout
# the transformation and reconciliation pipeline.
```

After:

```python
# Check that the totals did not change by mistake
# when we combined the datasets.
```

## Important comment rules

Do not remove comments that explain genuinely complicated or important logic.

For example, we still need clear comments explaining:

- Why some missing values are not replaced with zero.
- Why confidential housing locations cannot be assigned.
- Why the datasets are aggregated before joining.
- Why uncertain crash records are flagged.
- How the geographic assignment works.
- Why some observations are excluded from a calculation.
- Why post-integration cleaning is necessary.

However, explain these decisions in the simplest accurate language possible.

**The goal is not to disguise AI involvement. The goal is to make the code readable, consistent, and
understandable to the student responsible for reviewing and presenting it.**

Do not alter working Python logic merely to make its comments shorter.

Apply the same comment style to any Python code embedded in the notebook for shared website calculations.

---

# 4. Humanize the notebook's Markdown explanations

After improving the comments, review the explanatory Markdown throughout the notebook.

The Markdown should sound like a clear university project report, not a highly technical research publication.

## What I want

Keep the existing structure and required content, but improve readability.

Explain:

- What each section does.
- Why the step is necessary.
- What the outputs mean.
- What problems were discovered.
- How those problems were handled.

Use concise paragraphs and natural language.

Avoid unnecessarily elaborate descriptions.

For example:

Instead of:

"Geospatial normalization facilitates the establishment of a consistent spatial analytical framework for the
cross-domain investigation."

Use:

"We use the same geographic areas for both datasets so we can compare housing and crash records correctly."

Keep terms such as EDA, missing values, outliers, integration, correlation and data types when they are
relevant to the assignment.

Do not oversimplify explanations to the point that they become inaccurate.

Retain a professional academic tone suitable for a university project.

---

# 5. Reorganize Section 14 — Project development and contributions

This is another major requested change.

Currently, Section 14 focuses heavily on AI prompts and contribution summaries.

I want it to include a clear, detailed, chronological explanation of how the project was developed.

The development history should show the sequence of activities, decisions, testing, and improvements.

It should be easy for my professor to understand how the project progressed.

## Required development history

Organize the development process into the following stages.

### Stage 1 — Understanding the assignment

Document that the Project Description PDF was reviewed to identify:

- Main objectives.
- Dataset requirements.
- Data engineering requirements.
- Website requirements.
- Number of research questions and visualizations.
- Grading criteria.
- Submission requirements.

ChatGPT assisted with summarizing and explaining the assignment.

My contribution was reviewing the requirements and deciding to follow the simplest complete implementation
that met the PDF.

### Stage 2 — Reviewing the Data Catalog

Document that different housing and transportation datasets were considered.

The objective was to identify a combination that:

- Was reasonably easy to implement.
- Had useful geographic or temporal information.
- Would not require excessive preprocessing.
- Could support eight meaningful research questions.
- Would work efficiently with Python and Google Colab.

ChatGPT assisted with comparing candidate datasets and explaining their advantages and disadvantages.

### Stage 3 — Selecting the datasets

Explain the selection of:

- Affordable Housing Production by Building (H2).
- SIRTA Serious Traffic Injury Data (T4).

Document why the selected pair was considered suitable.

Explain that both datasets support geographic and temporal analysis, making meaningful integration possible.

My contribution included reviewing and approving the dataset selection.

### Stage 4 — Planning the project

Document how the project objective and eight research questions were developed.

Explain the decision to:

- Analyze 2022–2025.
- Integrate data by geographic area and year.
- Use Python and Google Colab.
- Use Plotly and Dash for the visualizations and website.

Document ChatGPT's planning and review assistance and Claude's subsequent implementation assistance.

### Stage 5 — Setting up the environment

Document the actual setup activities:

- Using Google Colab.
- Preparing the Python notebook.
- Identifying the required libraries.
- Creating a GitHub repository.
- Organizing the project files.

Record only activities that actually occurred.

### Stage 6 — Data exploration

Describe the initial inspection of the datasets:

- Number of rows and columns.
- Missing values.
- Data types.
- Duplicate records.
- Outliers.
- Dates.
- Geographic information.

Explain that the notebook was used to examine the datasets before integration.

### Stage 7 — Data cleaning

Describe the cleaning work carried out for both datasets.

Include examples of actual problems found:

- Missing or confidential locations.
- Invalid ZIP codes.
- Duplicates.
- Suspicious dates.
- Missing numeric values.
- Inconsistent data formats.

Explain the main cleaning decisions without presenting an unnecessarily long technical report.

### Stage 8 — Integrating the datasets

Document the decision to aggregate both datasets by geographic area and year.

Explain that this avoids incorrectly combining individual buildings with individual crash records.

Describe the post-integration cleaning and validation performed.

### Stage 9 — Testing and debugging

This stage must include actual activities from our development process.

For example:

- Running feasibility notebooks in Google Colab.
- Examining the real dataset outputs.
- Identifying a Pandas missing-value error.
- Correcting the `pd.NA` Boolean comparison.
- Rerunning the relevant code.
- Reviewing integration totals.
- Checking validation outputs.

Clarify that ChatGPT assisted with diagnosing and reviewing issues, while Claude assisted with implementation
and revisions.

My contribution included executing notebooks, reporting errors, reviewing proposed corrections and verifying
results.

### Stage 10 — Creating the research visualizations

Explain how eight research questions were connected to eight charts.

Include the use of scatter plots, maps, line charts, heatmaps, box plots and bar charts.

Explain that the charts were reviewed to ensure they answered their questions.

### Stage 11 — Reviewing the project against the PDF

Document that the implementation was compared with the official requirements.

Explain the decision to simplify the project by removing unnecessary advanced statistics, such as bootstrap
inference and excessive robustness testing.

The goal was to satisfy the assignment without introducing unnecessary complexity.

### Stage 12 — Simplifying the final notebook

Explain the transition from the advanced notebook to the current simplified version.

Document that the main cleaning, integration and research visualizations were preserved.

### Stage 13 — Real-data execution and validation

Include the results from the successfully executed simplified notebook.

Confirmed previous execution:

- Source mode: `fresh-download`.
- Real NYC Open Data.
- Execution date: 15 October 2026.
- 80 of 80 Python code cells executed.
- 74 validation checks passed.
- Zero validation warnings.
- Zero validation failures.
- Eight research visualizations generated.
- Exported research results reconciled with the notebook outputs.

This information describes the previous verified execution.

If you modify any data-processing calculation, require a fresh execution before claiming that the revised
notebook has passed the same checks.

### Stage 14 — Website development and GitHub preparation

Explain that Claude generated the Dash website implementation.

Document the website features:

- Multiple dropdown filters.
- Search functionality.
- Generate Report button.
- Eight interactive research charts.
- Data-story section.

Clarify that the website was previously tested with synthetic data and still requires full independent
verification on the real processed data and public deployment.

Document that I created and prepared the GitHub repository and have been organizing and uploading project
files.

Do not falsely claim that deployment has already happened.

### Stage 15 — Final review and submission

Explain the remaining work:

- Review the final notebook.
- Verify the real-data website.
- Correct GitHub file paths.
- Deploy the website.
- Document actual contributors.
- Prepare the university submission.

Use the correct completed/pending status for each task.

## Formatting of the development history

Use a readable table or structured sections.

For each stage, show:

- Stage name.
- Main objective.
- Work performed.
- Tools used.
- My involvement.
- Outcome or current status.

Do not invent exact dates if they are not supported by the available records.

Do not invent meetings, commits, manual code changes, or testing activities.

---

# 6. Explain the roles of ChatGPT, Claude and myself accurately

The current notebook focuses primarily on Claude.

I want the documentation to reflect the full development process.

Use these roles as the basis, provided they match the available history.

## Abdelrahman El-sayed — 13004550

Document my actual involvement in:

- Reviewing the university assignment.
- Comparing and approving dataset choices.
- Deciding the project scope.
- Reviewing research-question designs.
- Setting up and running Google Colab.
- Examining execution outputs.
- Reporting errors and testing corrections.
- Requesting improvements.
- Reviewing the project against the assignment.
- Preparing and organizing GitHub.
- Testing and validating project functionality as completed.

Do not claim I manually authored code that was generated by Claude or ChatGPT.

## ChatGPT — OpenAI

Document assistance with:

- Explaining the Project Description PDF.
- Comparing datasets and recommending suitable options.
- Planning the workflow.
- Explaining integration decisions.
- Reviewing feasibility results.
- Diagnosing errors.
- Suggesting and reviewing code corrections.
- Reviewing notebook outputs.
- Checking compliance with the assignment.
- Preparing implementation and revision instructions.

## Claude — Anthropic

Document assistance with:

- Generating Python notebook code.
- Implementing data cleaning and integration.
- Building research visualizations.
- Revising code after feedback.
- Simplifying the notebook.
- Generating Dash website code.
- Preparing supporting project files and documentation.
- Running synthetic tests where documented.

## Presentation requirements

Do not portray AI as the project owner.

Do not portray me as the author of code I did not personally write.

Explain the development process as a sequence of human decisions, review, testing and AI-assisted
implementation.

Avoid unnecessary repetition of AI attribution throughout the notebook.

A short factual attribution is sufficient in the introduction; detailed documentation belongs in Section 14
and the AI prompt log.

Keep everything truthful and consistent with the university requirements.

---

# 7. Improve the AI Prompt Log

The Project Description PDF requires that AI prompts used during the project be documented.

Keep that requirement.

However, I want the AI Prompt Log to be organized more clearly.

Use the following structure:

| Prompt / Stage | Tool | Purpose | Result | Verification |
|---|---|---|---|---|
| Assignment analysis | ChatGPT | Understand PDF requirements | Project requirement summary | Reviewed |
| Dataset comparison | ChatGPT | Compare options | H2 + T4 recommendation | Reviewed |
| Project planning | ChatGPT / Claude | Plan questions and integration | Project design | Reviewed |
| Notebook implementation | Claude | Implement Python pipeline | Notebook files | Tested as recorded |
| Debugging | ChatGPT / Claude | Identify and correct errors | Code corrections | Verified where tested |
| Simplification | Claude | Remove unnecessary complexity | Simplified notebook | Real-data run completed |
| Website implementation | Claude | Generate Dash application | Website source files | Real-data website
testing pending |

These are development-stage summaries, not substitutes for the actual prompts.

### Exact prompt text

Preserve the original complete prompt text wherever available.

Include actual ChatGPT prompts only if their wording is available from the supplied conversation or documents.

Do not reconstruct fictional verbatim conversations.

If a previous prompt is unavailable, write:

"Exact original prompt not available; development-stage summary recorded."

Separate:

1. Development history.
2. AI assistance disclosure.
3. Exact prompts used.
4. Testing and verification status.

Avoid repeating the full prompts unnecessarily in multiple locations unless required by the university or
necessary for submission.

If repository documentation includes `docs/AI_PROMPT_LOG.md`, keep it consistent with Section 14.

---

# 8. Correct the missing-borough data issue

A review of the actual processed outputs identified one data-reconciliation issue.

The main integrated geographic panel is correct, but the borough-level summary omits crash records when their
original borough field is missing.

Confirmed previous results:

- 36 crash records from 2022–2025 have no reported borough.
- 28 of these have valid geographic assignments.
- Eight remain geographically unassigned.

The previous borough summary totals were:

- 11,922 crashes, compared with 11,958 in the full analysis.
- 10,749 geographically assigned crashes, compared with 10,777 in the integrated panel.

## Required correction

1. Inspect the relevant cleaning and borough-summary code.
2. For records with reliable MODZCTA assignments, determine whether borough information can be derived from
the geographic reference.
3. Clearly identify derived borough values.
4. Keep records without a reliable borough under an Unknown category or in a separate reconciliation category.
5. Do not invent geographic assignments.
6. Ensure the original crash records are not duplicated.
7. Verify that borough totals reconcile with the overall dataset.
8. Add a concise validation check confirming the reconciliation.
9. Update affected exports and documentation.

Use the provided real output data to verify this discrepancy.

Do not modify the main geographic integration methodology unnecessarily.

Do not add advanced analysis to fix this issue.

---

# 9. Complete the "What surprised us" data-story section

The executed simplified notebook includes a placeholder in the data story.

The assignment requires an explanation of what the analysis found and what was interesting or surprising.

Use the actual verified data to prepare a short, understandable paragraph highlighting a meaningful
observation.

For example, the previous analysis found that New Construction and Preservation had different relationships
with serious-crash rates.

However:

- Verify the current numbers.
- Do not invent personal reactions.
- Do not make unsupported causal claims.
- Do not state that a result personally surprised me unless I have confirmed that reaction.

You may write a clear proposed paragraph for my review and mark it as requiring student confirmation.

The paragraph should be concise and consistent with the project findings.

Do not add another research question or additional visualization.

---

# 10. Update the notebook's execution and verification status

The simplified notebook was previously executed successfully on real data.

Some of its documentation still describes real-data testing as pending.

Correct outdated statements to distinguish:

**Previously verified:**

- Real-data Google Colab execution.
- 80 executed code cells.
- 74 passed validations.
- Zero validation warnings.
- Zero validation failures.
- Eight research charts.
- Exported results that match notebook calculations.

**Still pending or requiring reconfirmation:**

- Rerunning the revised notebook after the borough correction.
- Testing the website using updated real-data exports.
- Public website deployment.
- Final GitHub verification.
- Final university submission.

Do not make false claims about revised code having passed tests that have not actually been rerun.

If your environment cannot execute the real NYC datasets, clearly distinguish the prior verified version from
the revised version awaiting Colab validation.

---

# 11. Preserve the existing project scope

This instruction is critical.

The simplified notebook already satisfies almost all technical notebook requirements.

Do not:

- Change datasets.
- Change the analysis period.
- Add research questions.
- Add additional charts beyond those necessary for required EDA.
- Add machine learning.
- Reintroduce advanced statistical analysis.
- Add complicated testing frameworks.
- Change the geographic integration method without a demonstrated need.
- Redesign the dashboard.
- Reorganize the entire repository unnecessarily.
- Remove required EDA or cleaning stages.

Preserve:

- Data loading.
- Exploratory Data Analysis.
- Pre-integration cleaning.
- Missing-value handling.
- Outlier detection.
- Duplicate handling.
- Dataset integration.
- Post-integration cleaning.
- Eight research questions and charts.
- Data story and limitations.
- Required exports.
- Existing website compatibility.

The highest-priority changes are writing improvements, documentation, and the one verified data-reconciliation
correction.

---

# 12. Keep the notebook and website code consistent

The simplified notebook includes or generates shared Python code used by the Dash website.

If a comment or documentation change affects shared code, update its source consistently.

If the borough-summary correction changes exported data or functions, ensure the website continues to use the
corrected values.

If you are only given the notebook and not the full repository, identify any corresponding external files that
must also be updated rather than pretending to have modified files you cannot access.

Do not rewrite the website's appearance or controls.

Preserve the existing:

- Five dropdown filters.
- Search mode.
- Generate Report button.
- Eight charts.
- Data story.
- Deployment configuration.

Only modify website-related logic if necessary for consistency with the corrected notebook.

---

# 13. Code quality and readability

Besides the comments, review the notebook for small readability improvements.

You may improve:

- Unclear variable names.
- Repetitive explanations.
- Unnecessary comments.
- Long Markdown paragraphs.
- Inconsistent terminology.
- Unnecessarily formal chart subtitles.
- Confusing output labels.

Do not perform broad refactoring.

Do not replace clear, working code simply because another implementation looks more sophisticated.

Use the simplest maintainable solution.

A student familiar with basic Python, pandas and Plotly should be able to follow the important steps.

---

# 14. Deliverables

Return the following.

## Deliverable A — Updated Google Colab notebook

An actual downloadable `.ipynb` file:

`01_data_engineering_and_visualization_SIMPLIFIED_REVISED.ipynb`

It must contain:

- Humanized comments throughout.
- Clear Markdown explanations.
- Corrected Section 14.
- Development history.
- Accurate ChatGPT and Claude documentation.
- Corrected borough summary.
- Updated data-story section.
- Updated testing status.
- All existing required project functionality.

## Deliverable B — Change report

Provide a concise table explaining:

- What was changed.
- Why it was changed.
- Which section was affected.
- Whether the change affects numerical results.
- Whether additional Colab testing is required.

## Deliverable C — Updated documentation

Provide corrected versions of applicable files, particularly:

- `CONTRIBUTIONS.md`
- `docs/AI_PROMPT_LOG.md`

Preserve actual AI prompts and verified contribution records.

## Deliverable D — Required shared-code updates

If the notebook's revised borough logic requires changes to `src/analysis.py`, `src/figures.py`, or other
shared website modules, provide the corresponding updated files.

Do not modify unrelated website functionality.

## Deliverable E — Google Colab testing instructions

Explain how to:

1. Upload the revised notebook.
2. Run it from a fresh CPU runtime.
3. Check validation results.
4. Verify the corrected borough totals.
5. Inspect the eight charts.
6. Download the notebook with outputs.
7. Download the revised processed-output ZIP.

---

# 15. Final verification requirements

Before delivering, review the entire revised notebook.

Check:

- All existing mandatory sections are present.
- Python code compiles.
- The notebook remains runnable in Google Colab.
- Code comments are consistent and understandable.
- Markdown explanations are natural and concise.
- The borough-summary totals reconcile.
- No additional statistical complexity has been introduced.
- The eight visualizations are preserved.
- Research results are not fabricated.
- Old outputs affected by revised calculations are cleared if not recomputed.
- AI contributions and student activities are documented truthfully.
- Prompt texts are not invented.
- The revised notebook remains compatible with the existing Dash project.

If the real data cannot be executed in your environment, state that clearly.

Do not manufacture successful execution counts or new research findings.

---

# FINAL INSTRUCTION

**Work from the simplified notebook I am providing.**

Do not restart the project.

Do not create an advanced version.

Do not introduce features beyond the university requirements.

Focus especially on making the Python comments natural, simple, consistent and easy to explain.

Improve the Markdown and development history, document the actual roles of ChatGPT, Claude and myself, correct
the verified borough-summary issue, and update the project's execution status.

Preserve everything that already works.

**Deliver the complete revised notebook and the necessary supporting documentation, not merely a list of
suggested changes.**
~~~

#### Claude prompt 6 — Final submission preparation (2026-10-16)

~~~text
# FINAL SUBMISSION PREPARATION — GIU DATA ENGINEERING AND VISUALIZATION

## Objective

I want you to prepare the final submission-ready version of my Google Colab notebook for my German
International University (GIU) Data Engineering and Visualization project.

**Project:** Affordable Housing Development and Road Safety in New York City

**Student:** Abdelrahman El-sayed

**Student ID:** 13004550

**Deadline:** 9 November 2026

I will provide the latest revised simplified notebook:

`01_data_engineering_and_visualization_SIMPLIFIED_REVISED.ipynb`

I may also provide the latest `processed_outputs.zip` and supporting documentation.

**You MUST work from this latest revised simplified notebook. Do not use older notebook versions.**

The current notebook has already been improved and successfully run with real NYC data in Google Colab.

Its latest processed outputs report:

- 76 passed validation checks.
- 0 warnings.
- 0 failures.
- 708 integrated area-year records.
- Eight research-question visualizations.
- Corrected borough summaries.
- Consistent numerical research results.

The purpose of this request is FINALIZATION, not further development.

Do not rebuild the notebook, change the research questions, introduce new analysis, or modify working
calculations unnecessarily.

---

## 1. Finalize the "What surprised us" section

I have now reviewed and approved the interpretation based on the actual results.

Replace the provisional paragraph with this final wording:

**What surprised us**

One finding that stood out was the difference between new construction and preservation. Areas with more
new-construction units per resident tended to have higher serious-crash rates (ρ = 0.42), while preservation
showed a much weaker relationship (ρ = 0.10). We also found that new-construction-dominant areas had a higher
median crash rate (31.4) than preservation-dominant areas (23.2). This is an interesting difference, but it
does not mean that new construction causes more crashes.

Verify that the latest real-data results still support these values before retaining them.

Remove this provisional message everywhere it appears:

"Proposed text based on the results — to be confirmed by Abdelrahman El-sayed before submission."

Also remove any equivalent wording saying that the paragraph still requires my approval.

**Important:** If the notebook generates the data-story paragraph automatically, update the Python code or
Markdown generation responsible for it. Do not simply remove the message from an existing cell output if it
will reappear when the notebook runs again.

Ensure that `reports/data_story.md` and any website data-story text remain consistent with the final approved
version where those files are available.

---

## 2. Final review of Python comments — VERY IMPORTANT

This is still one of my highest priorities.

Review the comments in every Python code cell one last time.

The comments should:

- Sound natural and straightforward.
- Be written in simple English.
- Explain why important steps are performed.
- Be easy for a university student to understand and explain.
- Avoid unnecessarily advanced terminology.
- Avoid repetitive or excessively formal language.
- Be short unless a complicated decision genuinely requires more explanation.

Examples of the writing style I want:

```python
# Check for missing values before cleaning the data.
```

```python
# Combine both datasets by ZIP area and year.
# This avoids joining individual buildings to individual crashes.
```

```python
# Some locations are confidential, so we cannot place
# those buildings on the map.
```

```python
# Make sure the totals still match after the join.
```

Keep comments explaining important cleaning and integration decisions.

Do not add comments to every single line.

Do not add decorative separators or overly elaborate section descriptions.

Do not falsely claim that I manually wrote AI-generated code.

**If the comments are already clear and natural, leave them unchanged.**

Avoid unnecessary modifications to working code.

---

## 3. Final review of Markdown explanations

Review the entire notebook from Section 1 through Section 14.

Make sure the explanations are:

- Clear.
- Concise.
- Natural.
- Understandable.
- Appropriate for a university project.
- Consistent with the actual Python code and computed outputs.

Avoid paragraphs that repeat information already explained elsewhere.

Make sure every important stage clearly explains:

1. What was done.
2. Why it was needed.
3. What the results show.
4. Any important limitations.

Preserve required technical terms such as EDA, missing values, outliers, data integration, post-integration
cleaning and correlation.

Do not remove any explanation required by the Project Description PDF.

Do not turn the notebook into a complex academic research paper.

---

## 4. Finalize Section 14 — Development history and contributions

Preserve the detailed 15-stage development history from the latest revised notebook.

The history must accurately document the progression from:

1. Understanding the assignment PDF.
2. Comparing the recommended datasets.
3. Selecting H2 and SIRTA.
4. Planning the research questions.
5. Setting up Google Colab.
6. Exploring the datasets.
7. Cleaning the datasets.
8. Integrating the datasets.
9. Debugging and testing.
10. Creating the visualizations.
11. Reviewing the project against the PDF.
12. Simplifying the original implementation.
13. Running and validating the notebook on real data.
14. Developing the website and preparing GitHub.
15. Preparing the final submission.

Ensure the explanations remain natural, concise and factually correct.

### Contribution documentation

Preserve the accurate distinction between:

**My involvement:** Reviewing requirements, evaluating and approving datasets, choosing the project scope,
reviewing research questions, running Colab, reporting errors, verifying outputs, requesting corrections,
checking compliance and preparing GitHub.

**ChatGPT:** Assignment interpretation, dataset comparison, planning, debugging assistance, reviewing outputs,
compliance checks and preparing prompts.

**Claude:** Code generation, notebook implementation, cleaning and integration code, visualizations,
revisions, website implementation and documented testing.

Do not remove AI assistance information required by the university.

Do not exaggerate my manual coding contributions.

Do not invent team-member contributions.

Keep the unresolved four-member requirement visible until the instructor clarifies whether individual
submission is allowed.

---

## 5. Finalize the AI prompt documentation

Preserve the original Claude prompts currently recorded in Section 14 and `docs/AI_PROMPT_LOG.md`.

Do not invent exact ChatGPT prompts that are unavailable.

Keep the documentation organized into:

- Project development stages.
- AI tools and their roles.
- Actual student contributions.
- Original AI prompts where available.
- Testing and verification.

Avoid excessively repeating the same information throughout the notebook.

The notebook should mention AI assistance briefly in its introduction, with detailed documentation in Section
14.

Do not remove required AI disclosures just to make the notebook appear manually written.

---

## 6. Verify the final real-data results

Review the latest executed notebook and processed outputs.

The previous real-data execution reported:

- 76 PASS.
- 0 WARN.
- 0 FAIL.
- 11,958 SIRTA crashes during 2022–2025.
- 10,777 geographically assigned crashes.
- 101,249 completed affordable housing units.
- 708 integrated geographic area-year rows.

The borough correction should remain in place:

- 28 crash records with missing original boroughs assigned through their known geographic area.
- Eight records retained under Unknown.
- Full borough totals reconcile correctly.

Preserve the borough validation checks.

Confirm that the eight research questions and visualizations remain consistent with the actual results.

If you make any change that affects calculations, rerun the relevant tests.

Never insert historical results as fabricated current notebook outputs.

If the notebook cannot be executed on real NYC data in your environment, preserve the code and clearly state
that a new Google Colab run is required.

---

## 7. Check the official PDF requirements one last time

Use the Project Description PDF, if attached, as the authoritative source.

Verify that the notebook includes:

- Dataset descriptions and justification.
- Exploratory Data Analysis.
- Descriptive statistics.
- Initial visualizations.
- Missing-value handling.
- Outlier detection and handling.
- Duplicate and inconsistent-data handling.
- Data cleaning before integration.
- Meaningful integration of the selected datasets.
- Cleaning after integration.
- Eight research questions.
- Eight research visualizations.
- Explanations and interpretations.
- A meaningful data story.
- Data limitations.
- Clearly documented contributions.
- AI prompt documentation.
- Reproducibility information.

Report any genuinely missing requirement.

Do not add optional advanced statistics, more research questions or unnecessary features.

The website, deployment and complete submission requirements may have separate pending tasks; do not claim
they are completed merely because the notebook is ready.

---

## 8. Remove unnecessary provisional or outdated wording

Search the entire notebook for phrases such as:

- "To be confirmed."
- "Proposed text."
- "Placeholder."
- "Draft."
- "Testing pending."
- "Real-data run pending."
- "Replace before submission."
- "Student to review."

Review every occurrence in context.

If it refers to an activity that is now genuinely completed, update or remove the outdated wording.

If it refers to something that is still actually pending, keep an accurate status rather than hiding it.

For example:

- The real-data notebook validation is completed.
- The data-story paragraph is now approved.
- Public website deployment is still pending unless independently verified.
- The university's team-contribution requirement still needs clarification.

The final notebook must not contain unnecessary authoring reminders in its submitted analytical content.

---

## 9. Preserve everything that already works

This is critical.

Do not change:

- The two selected datasets.
- The 2022–2025 analysis period.
- The existing geographic integration methodology.
- The main cleaning decisions.
- The eight research questions.
- The eight visualizations.
- The statistical calculations.
- The verified borough correction.
- The existing website design.
- The overall notebook organization.

Only make targeted changes necessary to finalize presentation, documentation, and factual consistency.

Prefer minimal changes over broad refactoring.

---

## 10. Deliverables

### A. Final submission notebook

Provide a downloadable, complete Google Colab file named:

`01_data_engineering_and_visualization_FINAL_SUBMISSION.ipynb`

The notebook must preserve its executable code, 14-section structure, and required charts.

If you have executed the final code successfully, include its genuine saved outputs.

Otherwise, clearly indicate that I need to perform the final execution in Colab.

### B. Final change report

Provide a short report identifying:

- What was modified.
- Which section was modified.
- Whether numerical calculations changed.
- Whether another Colab execution is necessary.

### C. Updated documentation, if needed

Only provide updated supporting files where there are actual changes, such as:

- `CONTRIBUTIONS.md`
- `docs/AI_PROMPT_LOG.md`
- `README.md`
- `reports/data_story.md`

Keep these documents consistent with the final notebook.

### D. Final compliance checklist

Provide a concise checklist mapping the official PDF requirements to the final notebook sections.

Distinguish between completed notebook requirements and pending website, deployment or administrative tasks.

### E. Final Google Colab instructions

Give me the exact steps to:

1. Open the final notebook.
2. Run all cells if required.
3. Confirm zero validation failures.
4. Verify all eight charts.
5. Download the executed notebook.
6. Download the latest processed output ZIP.
7. Identify the final files that should replace the previous GitHub versions.

---

## FINAL INSTRUCTION

This is the final preparation of an already working project.

**Do not rebuild the project. Do not add complexity. Do not create unnecessary new functionality.**

Prioritize:

1. Clear, natural, understandable Python comments.
2. Concise and professional Markdown.
3. Finalizing the approved "What surprised us" section.
4. Accurate contribution and AI documentation.
5. Preserving the successful real-data processing and validations.
6. Ensuring compliance with the university's project requirements.

Deliver the complete final notebook and a concise report of any remaining genuine blockers.

The finished notebook should be clean, understandable, consistent, professional and ready for my final Google
Colab verification and university submission.
~~~